# 06F — PCA Species Association

## Objective

This notebook tests whether the multivariate acoustic structure identified
without ecological labels is systematically associated with species identity.

The scientific question is:

> Do the dominant acoustic dimensions identified from unsupervised acoustic
> features show systematic association with species identity?

Species identity is introduced only after PCA construction. It is therefore
used for post-hoc ecological interpretation rather than as an input to the
unsupervised representation.

## Primary analysis unit

The recording is the primary unit of analysis.

The focal species-association subset contains:

- 56 species
- 172 recordings

This restriction is applied before statistical testing to ensure that species
with sufficient recording representation contribute to the association analysis.

## Statistical principle

Observed species–PCA relationships will be compared against a permutation null
distribution in which species labels are shuffled across recordings while the
acoustic PCA structure is held fixed.

This tests whether the observed association is stronger than expected from
the marginal species frequencies alone.

## Important distinction

A significant species association does not mean that PCA provides a direct
measure of biodiversity or species richness.

It indicates that acoustic variation contains reproducible structure associated
with species identity.

In [1]:
# ============================================================
# 06F.1 — Setup and locate ecological-analysis inputs
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

DATA_PROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

PCA_ANALYSIS_DIR = (
    DATA_PROCESSED_DIR / "pca_analysis"
)

PCA_REPRESENTATION_DIR = (
    DATA_PROCESSED_DIR / "pca_representations"
)

BOOTSTRAP_DIR = (
    PCA_ANALYSIS_DIR / "bootstrap_stability"
)


# ------------------------------------------------------------
# Analysis constants
# ------------------------------------------------------------

N_RECORDINGS_FULL = 275
N_SPECIES_TARGET = 56
N_RECORDINGS_TARGET = 172

RANDOM_SEED = 42


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("06F — PCA SPECIES ASSOCIATION")
print("=" * 70)

print(f"Project root:")
print(PROJECT_ROOT)

print(f"\nProcessed-data directory:")
print(DATA_PROCESSED_DIR)

print(f"\nPCA analysis directory:")
print(PCA_ANALYSIS_DIR)

print(f"\nPCA representation directory:")
print(PCA_REPRESENTATION_DIR)

print(f"\nBootstrap directory:")
print(BOOTSTRAP_DIR)

print("\n" + "=" * 70)
print("TARGET ECOLOGICAL SUBSET")
print("=" * 70)

print(f"Target species:    {N_SPECIES_TARGET}")
print(f"Target recordings: {N_RECORDINGS_TARGET}")
print(f"Random seed:       {RANDOM_SEED}")

06F — PCA SPECIES ASSOCIATION
Project root:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity

Processed-data directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed

PCA analysis directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis

PCA representation directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_representations

Bootstrap directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\bootstrap_stability

TARGET ECOLOGICAL SUBSET
Target species:    56
Target recordings: 172
Random seed:       42


In [2]:
# ============================================================
# 06F.1 — Cell 2: Inventory relevant analysis artifacts
# ============================================================

print("=" * 70)
print("PCA ANALYSIS ARTIFACTS")
print("=" * 70)

for path in sorted(PCA_ANALYSIS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))


print("\n" + "=" * 70)
print("PCA REPRESENTATION ARTIFACTS")
print("=" * 70)

for path in sorted(PCA_REPRESENTATION_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))


print("\n" + "=" * 70)
print("LIKELY METADATA / FEATURE TABLES")
print("=" * 70)

# Search processed data for CSV/Parquet files whose names
# suggest recording-level metadata or feature tables.
candidate_files = []

for path in sorted(DATA_PROCESSED_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in {".csv", ".parquet"}:
        name = path.name.lower()

        keywords = [
            "feature",
            "metadata",
            "recording",
            "master",
            "acoustic",
            "species",
            "pca",
        ]

        if any(keyword in name for keyword in keywords):
            candidate_files.append(path)

for path in candidate_files:
    print(path.relative_to(PROJECT_ROOT))

print(f"\nCandidate metadata/feature files found: {len(candidate_files)}")

PCA ANALYSIS ARTIFACTS
data\processed\pca_analysis\baseline_components.npy
data\processed\pca_analysis\baseline_explained_variance.npy
data\processed\pca_analysis\baseline_explained_variance_ratio.npy
data\processed\pca_analysis\baseline_feature_names.json
data\processed\pca_analysis\baseline_pa_null_95pct.npy
data\processed\pca_analysis\baseline_pa_null_mean.npy
data\processed\pca_analysis\baseline_pa_observed_eigenvalues.npy
data\processed\pca_analysis\baseline_pa_retained.npy
data\processed\pca_analysis\baseline_scores.npy
data\processed\pca_analysis\baseline_singular_values.npy
data\processed\pca_analysis\block_balanced_components.npy
data\processed\pca_analysis\block_balanced_explained_variance.npy
data\processed\pca_analysis\block_balanced_explained_variance_ratio.npy
data\processed\pca_analysis\block_balanced_feature_names.json
data\processed\pca_analysis\block_balanced_pa_null_95pct.npy
data\processed\pca_analysis\block_balanced_pa_null_mean.npy
data\processed\pca_analysis\bloc

In [3]:
# ============================================================
# 06F.1 — Cell 3: Inspect candidate recording-level tables
# ============================================================

print("=" * 70)
print("CANDIDATE RECORDING / FEATURE TABLES")
print("=" * 70)

for path in candidate_files:
    print(f"\n{'-' * 70}")
    print(f"FILE: {path.relative_to(PROJECT_ROOT)}")

    try:
        if path.suffix.lower() == ".csv":
            df_tmp = pd.read_csv(path, nrows=5)
        elif path.suffix.lower() == ".parquet":
            df_tmp = pd.read_parquet(path).head(5)
        else:
            continue

        print(f"Shape preview: {df_tmp.shape}")
        print("Columns:")
        print(list(df_tmp.columns))

    except Exception as e:
        print(f"Could not inspect file: {e}")

CANDIDATE RECORDING / FEATURE TABLES

----------------------------------------------------------------------
FILE: data\processed\acoustic_features_recording.csv
Shape preview: (5, 114)
Columns:
['recording_id', 'state', 'species_id', 'common_name', 'scientific_name', 'original_sample_rate', 'original_channels', 'flag_short_duration', 'flag_low_amplitude', 'flag_clipping', 'flag_low_quality', 'flag_any', 'flag_bandlimit_incompatible', 'n_windows', 'aci_mean', 'adi_mean', 'bioacoustic_index_mean', 'ndsi_mean', 'ndsi_ratio_ba_mean', 'anthropophony_mean', 'biophony_mean', 'spectral_entropy_eas_mean', 'spectral_entropy_ecu_mean', 'spectral_entropy_ecv_mean', 'spectral_entropy_eps_mean', 'spectral_entropy_eps_kurt_mean', 'spectral_entropy_eps_skew_mean', 'temporal_entropy_mean', 'rms_mean_mean', 'rms_std_mean', 'zcr_mean_mean', 'zcr_std_mean', 'spectral_centroid_mean_mean', 'spectral_centroid_std_mean', 'spectral_bandwidth_mean_mean', 'spectral_bandwidth_std_mean', 'spectral_rolloff_mean_me

In [4]:
# ============================================================
# 06F.1 — Cell 4: Audit species representation
# ============================================================

FEATURES_PATH = DATA_PROCESSED_DIR / "acoustic_features_recording.csv"

recording_features = pd.read_csv(FEATURES_PATH)

print("=" * 70)
print("RECORDING-LEVEL FEATURE TABLE")
print("=" * 70)

print(f"Shape: {recording_features.shape}")
print(f"Unique recordings: {recording_features['recording_id'].nunique()}")
print(f"Unique species: {recording_features['species_id'].nunique()}")

print("\n" + "=" * 70)
print("SPECIES RECORDING COUNTS")
print("=" * 70)

species_counts = (
    recording_features
    .groupby(
        ["species_id", "common_name", "scientific_name"],
        dropna=False
    )
    .size()
    .reset_index(name="n_recordings")
    .sort_values(
        ["n_recordings", "scientific_name"],
        ascending=[False, True]
    )
)

print(species_counts.to_string(index=False))

print("\n" + "=" * 70)
print("SPECIES-COUNT SUMMARY")
print("=" * 70)

print(species_counts["n_recordings"].describe())

print("\nSpecies by minimum recording threshold:")

for threshold in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    eligible = species_counts[
        species_counts["n_recordings"] >= threshold
    ]

    n_species = len(eligible)
    n_recordings = eligible["n_recordings"].sum()

    print(
        f"  >= {threshold:2d} recordings/species: "
        f"{n_species:3d} species, "
        f"{n_recordings:3d} recordings"
    )

RECORDING-LEVEL FEATURE TABLE
Shape: (275, 114)
Unique recordings: 275
Unique species: 159

SPECIES RECORDING COUNTS
                species_id                  common_name            scientific_name  n_recordings
         Anas carolinensis            Green-winged Teal          Anas carolinensis             5
     Archilochus alexandri    Black-chinned Hummingbird      Archilochus alexandri             5
          Chordeiles minor             Common Nighthawk           Chordeiles minor             5
   Geococcyx californianus           Greater Roadrunner    Geococcyx californianus             5
           Mareca strepera                      Gadwall            Mareca strepera             5
  Phalaenoptilus nuttallii              Common Poorwill   Phalaenoptilus nuttallii             5
   Selasphorus platycercus     Broad-tailed Hummingbird    Selasphorus platycercus             5
        Anas platyrhynchos                      Mallard         Anas platyrhynchos             4
          

In [5]:
# ============================================================
# 06F.1 — Cell 5: Audit QC and state structure
# ============================================================

print("=" * 70)
print("QC FLAG DISTRIBUTION")
print("=" * 70)

qc_cols = [
    "flag_short_duration",
    "flag_low_amplitude",
    "flag_clipping",
    "flag_low_quality",
    "flag_any",
    "flag_bandlimit_incompatible",
]

for col in qc_cols:
    if col in recording_features.columns:
        print(f"\n{col}:")
        print(recording_features[col].value_counts(dropna=False).to_string())


print("\n" + "=" * 70)
print("STATE × QC")
print("=" * 70)

print(
    recording_features
    .groupby("state")
    .agg(
        n_recordings=("recording_id", "nunique"),
        n_species=("species_id", "nunique"),
        n_flagged=("flag_any", "sum"),
        n_bandlimit_incompatible=("flag_bandlimit_incompatible", "sum"),
    )
    .to_string()
)


print("\n" + "=" * 70)
print("CLEAN / UNFLAGGED DATASET")
print("=" * 70)

clean = recording_features[
    ~recording_features["flag_any"].astype(bool)
].copy()

print(f"Clean recordings: {len(clean)}")
print(f"Clean species:    {clean['species_id'].nunique()}")

clean_species_counts = (
    clean.groupby("species_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nNumber of species by recording count:")
print(clean_species_counts.to_string())


print("\n" + "=" * 70)
print("BANDLIMIT-COMPATIBLE DATASET")
print("=" * 70)

bandlimit_compatible = recording_features[
    ~recording_features["flag_bandlimit_incompatible"].astype(bool)
].copy()

print(f"Bandlimit-compatible recordings: {len(bandlimit_compatible)}")
print(f"Bandlimit-compatible species:    {bandlimit_compatible['species_id'].nunique()}")

QC FLAG DISTRIBUTION

flag_short_duration:
flag_short_duration
False    267
True       8

flag_low_amplitude:
flag_low_amplitude
False    222
True      53

flag_clipping:
flag_clipping
False    267
True       8

flag_low_quality:
flag_low_quality
False    275

flag_any:
flag_any
False    208
True      67

flag_bandlimit_incompatible:
flag_bandlimit_incompatible
False    274
True       1

STATE × QC
       n_recordings  n_species  n_flagged  n_bandlimit_incompatible
state                                                              
AZ               75         44         30                         0
CA              100         66         15                         0
TX              100        100         22                         1

CLEAN / UNFLAGGED DATASET
Clean recordings: 208
Clean species:    134

Number of species by recording count:
1    92
2    20
3    13
4     8
5     1

BANDLIMIT-COMPATIBLE DATASET
Bandlimit-compatible recordings: 274
Bandlimit-compatible species:    158


In [6]:
# ============================================================
# 06F.1 — Cell 6: Trace the origin of the 56-species /
#             172-recording subset
# ============================================================

import re

SEARCH_TERMS = [
    "172",
    "56 species",
    "56_species",
    "species association",
    "species_association",
    "species association subset",
    "eligible species",
]

SEARCH_EXTENSIONS = {
    ".ipynb",
    ".py",
    ".md",
    ".txt",
    ".json",
    ".csv",
}

matches = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    # Skip large/raw/generated data directories where possible
    relative = path.relative_to(PROJECT_ROOT)

    if any(
        part.lower() in {
            ".git",
            "audio",
            "raw",
            "__pycache__",
        }
        for part in relative.parts
    ):
        continue

    if path.suffix.lower() not in SEARCH_EXTENSIONS:
        continue

    try:
        text_content = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    for term in SEARCH_TERMS:
        if term.lower() in text_content.lower():
            matches.append(
                (
                    str(relative),
                    term
                )
            )

print("=" * 70)
print("SUBSET-PROVENANCE SEARCH")
print("=" * 70)

if matches:
    # Remove duplicate file/term pairs while preserving order
    seen = set()

    for filename, term in matches:
        key = (filename, term)

        if key not in seen:
            seen.add(key)
            print(f"{filename}  <--  {term}")

    print(f"\nTotal matches: {len(seen)}")

else:
    print("No matches found.")

SUBSET-PROVENANCE SEARCH
data\interim\xenocanto_selected\xc_a1_selected_AZ.json  <--  172
data\interim\xenocanto_selected\xc_a1_selected_CA.json  <--  172
data\manifests\a1_xenocanto_selection_manifest.csv  <--  172
data\manifests\cleaning_log.csv  <--  172
data\processed\acoustic_features_recording.csv  <--  172
data\processed\acoustic_features_window.csv  <--  172
data\processed\ebird_observations.csv  <--  172
data\processed\master_metadata.csv  <--  172
data\processed\pca_analysis\parallel_analysis_detail.csv  <--  172
data\processed\pca_analysis\pca_first10.csv  <--  172
data\processed\pca_analysis\pca_variance_summary.csv  <--  172
data\processed\pca_analysis\recording_ids.json  <--  172
data\processed\pca_analysis\bootstrap_stability\bootstrap_explained_variance_summary.csv  <--  172
data\processed\pca_analysis\bootstrap_stability\principal_angle_summary.csv  <--  172
data\processed\pca_analysis\bootstrap_stability\subspace_stability_comparison.csv  <--  172
data\processed\pca_a

In [7]:
# ============================================================
# 06F.1 — Cell 7: Validate PCA recording-ID lineage
# ============================================================

PCA_IDS_PATH = PCA_ANALYSIS_DIR / "recording_ids.json"
REP_IDS_PATH = PCA_REPRESENTATION_DIR / "recording_ids.json"

print("=" * 70)
print("PCA RECORDING-ID FILES")
print("=" * 70)

print(f"PCA analysis IDs exist:       {PCA_IDS_PATH.exists()}")
print(f"PCA representation IDs exist: {REP_IDS_PATH.exists()}")


# ------------------------------------------------------------
# Load IDs
# ------------------------------------------------------------

with open(PCA_IDS_PATH, "r", encoding="utf-8") as f:
    pca_recording_ids = json.load(f)

with open(REP_IDS_PATH, "r", encoding="utf-8") as f:
    representation_recording_ids = json.load(f)


# ------------------------------------------------------------
# Normalize to simple lists
# ------------------------------------------------------------

def extract_id_list(obj):
    """
    Handle either:
      - a plain JSON list
      - a dictionary containing recording IDs
    """
    if isinstance(obj, list):
        return obj

    if isinstance(obj, dict):
        for key in [
            "recording_ids",
            "ids",
            "recording_id",
        ]:
            if key in obj:
                return obj[key]

    raise ValueError(
        f"Could not identify recording IDs. "
        f"JSON object type: {type(obj)}"
    )


pca_ids = extract_id_list(pca_recording_ids)
representation_ids = extract_id_list(
    representation_recording_ids
)

pca_ids = [str(x) for x in pca_ids]
representation_ids = [str(x) for x in representation_ids]


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ID COUNTS")
print("=" * 70)

print(f"PCA IDs:            {len(pca_ids)}")
print(f"Unique PCA IDs:     {len(set(pca_ids))}")

print(f"Representation IDs: {len(representation_ids)}")
print(f"Unique repr IDs:    {len(set(representation_ids))}")

print(f"Feature-table IDs:  "
      f"{recording_features['recording_id'].nunique()}")


# ------------------------------------------------------------
# Compare PCA and representation ordering
# ------------------------------------------------------------

same_length = len(pca_ids) == len(representation_ids)
same_order = pca_ids == representation_ids

print("\n" + "=" * 70)
print("PCA ↔ REPRESENTATION ALIGNMENT")
print("=" * 70)

print(f"Same length: {same_length}")
print(f"Same order:  {same_order}")


# ------------------------------------------------------------
# Compare against recording-level feature table
# ------------------------------------------------------------

feature_ids = recording_features["recording_id"].astype(str).tolist()
feature_id_set = set(feature_ids)

pca_id_set = set(pca_ids)

print("\n" + "=" * 70)
print("PCA ↔ FEATURE-TABLE COVERAGE")
print("=" * 70)

print(
    f"PCA IDs found in feature table: "
    f"{len(pca_id_set & feature_id_set)} / {len(pca_id_set)}"
)

print(
    f"Feature-table IDs represented in PCA: "
    f"{len(feature_id_set & pca_id_set)} / {len(feature_id_set)}"
)

missing_from_features = sorted(
    pca_id_set - feature_id_set
)

missing_from_pca = sorted(
    feature_id_set - pca_id_set
)

print(f"\nPCA IDs missing from feature table: {len(missing_from_features)}")
print(f"Feature IDs missing from PCA:        {len(missing_from_pca)}")


# ------------------------------------------------------------
# Show first few IDs for manual sanity check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 10 PCA IDS")
print("=" * 70)

print(pca_ids[:10])

PCA RECORDING-ID FILES
PCA analysis IDs exist:       True
PCA representation IDs exist: True

ID COUNTS
PCA IDs:            275
Unique PCA IDs:     275
Representation IDs: 275
Unique repr IDs:    275
Feature-table IDs:  275

PCA ↔ REPRESENTATION ALIGNMENT
Same length: True
Same order:  True

PCA ↔ FEATURE-TABLE COVERAGE
PCA IDs found in feature table: 275 / 275
Feature-table IDs represented in PCA: 275 / 275

PCA IDs missing from feature table: 0
Feature IDs missing from PCA:        0

FIRST 10 PCA IDS
['1035206', '1080401', '1080405', '1089596', '109164', '109171', '1101338', '1106496', '110689', '114789']


In [8]:
# ============================================================
# 06F.1 — Cell 8: Search existing PCA artifacts for
#             an explicit ecological subset
# ============================================================

print("=" * 70)
print("SEARCHING PCA ARTIFACTS FOR RECORDING SUBSETS")
print("=" * 70)

# ------------------------------------------------------------
# Inspect CSV artifacts that may contain recording-level rows
# ------------------------------------------------------------

pca_csvs = sorted(PCA_ANALYSIS_DIR.glob("*.csv"))

for path in pca_csvs:
    print(f"\n{'-' * 70}")
    print(path.name)

    try:
        df_tmp = pd.read_csv(path)

        print(f"Shape: {df_tmp.shape}")
        print("Columns:", list(df_tmp.columns))

        # If recording IDs are present, report their count.
        id_columns = [
            c for c in df_tmp.columns
            if "recording" in c.lower() and "id" in c.lower()
        ]

        for col in id_columns:
            ids = df_tmp[col].dropna().astype(str)

            print(
                f"  {col}: "
                f"{len(ids)} rows, "
                f"{ids.nunique()} unique IDs"
            )

    except Exception as e:
        print(f"Could not read: {e}")


# ------------------------------------------------------------
# Inspect JSON artifacts for recording-level ID collections
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("JSON ARTIFACTS")
print("=" * 70)

for path in sorted(PCA_ANALYSIS_DIR.glob("*.json")):
    print(f"\n{path.name}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, list):
            print(f"  JSON list length: {len(obj)}")

        elif isinstance(obj, dict):
            print(f"  JSON keys: {list(obj.keys())}")

            for key, value in obj.items():
                if isinstance(value, list):
                    print(
                        f"  {key}: list length = {len(value)}"
                    )

    except Exception as e:
        print(f"  Could not read: {e}")

SEARCHING PCA ARTIFACTS FOR RECORDING SUBSETS

----------------------------------------------------------------------
parallel_analysis_detail.csv
Shape: (294, 6)
Columns: ['representation', 'PC', 'observed_eigenvalue', 'null_mean', 'null_95pct', 'retained']

----------------------------------------------------------------------
parallel_analysis_summary.csv
Shape: (6, 10)
Columns: ['representation', 'n_features', 'retained_PCs', 'first_PC_below_null', 'PC1_observed', 'PC1_null_95', 'PC2_observed', 'PC2_null_95', 'PC5_observed', 'PC5_null_95']

----------------------------------------------------------------------
pca_first10.csv
Shape: (60, 4)
Columns: ['representation', 'PC', 'explained_variance_ratio', 'cumulative_variance']

----------------------------------------------------------------------
pca_variance_summary.csv
Shape: (6, 16)
Columns: ['representation', 'n_features', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC10', 'cumulative_5', 'cumulative_10', 'cumulative_20', 'PCs_to_50pct',

In [9]:
# ============================================================
# 06F.1 — Cell 9: Trace the 56-species ecological subset
# ============================================================

NOTEBOOK_DIR = PROJECT_ROOT / "research" / "notebooks"

TARGET_TERMS = [
    "56 species",
    "56_species",
    "species_count",
    "species threshold",
    "minimum species",
    "species subset",
    "eBird",
    "ebird",
    "172 recordings",
]

print("=" * 70)
print("TARGETED ECOLOGICAL-SUBSET SEARCH")
print("=" * 70)

results = []

for path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):

    try:
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
    except Exception:
        continue

    # Combine markdown + code cells
    cell_texts = []

    for cell in notebook.get("cells", []):
        source = "".join(cell.get("source", []))
        cell_texts.append(source)

    notebook_text = "\n".join(cell_texts).lower()

    found = [
        term for term in TARGET_TERMS
        if term.lower() in notebook_text
    ]

    if found:
        results.append((path.name, found))

for filename, found in results:
    print(f"\n{filename}")
    print("  " + ", ".join(found))

print(f"\nNotebooks containing relevant terms: {len(results)}")

TARGETED ECOLOGICAL-SUBSET SEARCH

02_data_acquisition.ipynb
  species_count, eBird, ebird

03_data_audit.ipynb
  eBird, ebird

04_cleaning.ipynb
  eBird, ebird

06_eda.ipynb
  eBird, ebird

06_F_species_association.ipynb
  56 species, 56_species, species_count, species threshold, minimum species, species subset, eBird, ebird, 172 recordings

Notebooks containing relevant terms: 5


In [10]:
# Find the existing 06F notebook
for p in sorted((PROJECT_ROOT / "research" / "notebooks").glob("*06*F*species*")):
    print(p)

c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\notebooks\06_F_species_association.ipynb


In [11]:
# ============================================================
# 06F.1 — Cell 10: Find code defining 56 / 172 together
# ============================================================

import json

NOTEBOOK_DIR = PROJECT_ROOT / "research" / "notebooks"

print("=" * 70)
print("SEARCHING FOR THE ORIGINAL 56 / 172 SELECTION LOGIC")
print("=" * 70)

for path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):

    try:
        with open(path, "r", encoding="utf-8") as f:
            nb_tmp = json.load(f)
    except Exception:
        continue

    for cell_num, cell in enumerate(nb_tmp.get("cells", [])):

        source = "".join(cell.get("source", []))

        # Look for cells that mention both target quantities.
        lower = source.lower()

        has_56 = (
            "56" in lower
            or "56_species" in lower
            or "56 species" in lower
        )

        has_172 = (
            "172" in lower
            or "172 recordings" in lower
        )

        if has_56 and has_172:
            print("\n" + "-" * 70)
            print(f"FILE: {path.name}")
            print(f"CELL: {cell_num}")
            print("-" * 70)
            print(source[:8000])

SEARCHING FOR THE ORIGINAL 56 / 172 SELECTION LOGIC

----------------------------------------------------------------------
FILE: 06_F_species_association.ipynb
CELL: 1
----------------------------------------------------------------------
# 06F — PCA Species Association

## Objective

This notebook tests whether the multivariate acoustic structure identified
without ecological labels is systematically associated with species identity.

The scientific question is:

> Do the dominant acoustic dimensions identified from unsupervised acoustic
> features show systematic association with species identity?

Species identity is introduced only after PCA construction. It is therefore
used for post-hoc ecological interpretation rather than as an input to the
unsupervised representation.

## Primary analysis unit

The recording is the primary unit of analysis.

The focal species-association subset contains:

- 56 species
- 172 recordings

This restriction is applied before statistical testing to

In [12]:
# ============================================================
# 06F.1 — Cell 10: Search EDA notebook for species-selection
#             logic involving eBird
# ============================================================

EDA_PATH = PROJECT_ROOT / "research" / "notebooks" / "06_eda.ipynb"

with open(EDA_PATH, "r", encoding="utf-8") as f:
    eda_nb = json.load(f)

print("=" * 70)
print("06_EDA — eBird / SPECIES SELECTION CELLS")
print("=" * 70)

terms = [
    "ebird",
    "species",
    "threshold",
    "merge",
    "common_species",
    "overlap",
    "shared",
]

for i, cell in enumerate(eda_nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    lower = source.lower()

    found = [term for term in terms if term.lower() in lower]

    if found:
        print("\n" + "-" * 70)
        print(f"CELL {i} | matches: {found}")
        print("-" * 70)
        print(source[:6000])

06_EDA — eBird / SPECIES SELECTION CELLS

----------------------------------------------------------------------
CELL 0 | matches: ['ebird']
----------------------------------------------------------------------
# Notebook 06 — Exploratory Data Analysis

## From Soundscapes to Biodiversity

This notebook performs exploratory data analysis on the cleaned acoustic and independent eBird observation datasets produced by the preceding data preparation and audio feature extraction workflows.

The analysis examines sampling coverage, geographic and temporal structure, recording characteristics, acoustic feature distributions, data-quality patterns, and relationships among acoustic descriptors. The resulting figures and structured JSON outputs are used both for the project website and as the exploratory foundation for subsequent statistical learning and modeling modules.

The analysis distinguishes between **dataset representation** and **ecological inference**. Xeno-canto recordings represent

In [13]:
# ============================================================
# 06F.1 — Cell 11: Find the original acoustic species-selection
#             logic in acquisition / cleaning notebooks
# ============================================================

SEARCH_NOTEBOOKS = [
    "02_data_acquisition.ipynb",
    "03_data_audit.ipynb",
    "04_cleaning.ipynb",
    "05_feature_extraction.ipynb",
]

SEARCH_TERMS = [
    "species_stratified",
    "round_robin",
    "species selection",
    "species_count",
    "recordings_per_species",
    "minimum",
    "56",
    "172",
    "eBird",
]

for notebook_name in SEARCH_NOTEBOOKS:

    path = PROJECT_ROOT / "research" / "notebooks" / notebook_name

    if not path.exists():
        print(f"\nNOT FOUND: {notebook_name}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    print("\n" + "=" * 80)
    print(f"{notebook_name}")
    print("=" * 80)

    for cell_num, cell in enumerate(nb.get("cells", [])):

        source = "".join(cell.get("source", []))
        lower = source.lower()

        matches = [
            term for term in SEARCH_TERMS
            if term.lower() in lower
        ]

        if matches:
            print("\n" + "-" * 80)
            print(f"CELL {cell_num} | matches: {matches}")
            print("-" * 80)
            print(source[:10000])


02_data_acquisition.ipynb

--------------------------------------------------------------------------------
CELL 0 | matches: ['eBird']
--------------------------------------------------------------------------------
import ebird

print("Loaded from:", ebird.__file__)
print("fetch_region_observations:", hasattr(ebird, "fetch_region_observations"))
print("save_raw_observations:", hasattr(ebird, "save_raw_observations"))
print("REGIONS:", hasattr(ebird, "REGIONS"))

--------------------------------------------------------------------------------
CELL 1 | matches: ['eBird']
--------------------------------------------------------------------------------
import sys
from pathlib import Path

ACQUISITION_DIR = Path.cwd()

if str(ACQUISITION_DIR) not in sys.path:
    sys.path.insert(0, str(ACQUISITION_DIR))

from xenocanto import (
    fetch_metadata,
    save_raw_metadata,
    download_audio_sample,
    A1_TARGETS,
)

from ebird import (
    fetch_region_observations,
    save_raw_observati

In [14]:
# ============================================================
# 06F.1 — Cell 12: Inspect species-stratified acquisition logic
# ============================================================

ACQ_PATH = PROJECT_ROOT / "research" / "notebooks" / "02_data_acquisition.ipynb"

with open(ACQ_PATH, "r", encoding="utf-8") as f:
    acq_nb = json.load(f)

KEYWORDS = [
    "species_stratified",
    "round_robin",
    "species",
    "selection",
    "recordings_per_species",
    "selected",
    "manifest",
    "sampling",
]

for i, cell in enumerate(acq_nb.get("cells", [])):

    source = "".join(cell.get("source", []))
    lower = source.lower()

    # Prioritize cells containing actual code rather than markdown-only
    # descriptions.
    if any(keyword in lower for keyword in KEYWORDS):

        print("\n" + "=" * 90)
        print(f"CELL {i}")
        print("=" * 90)
        print(source[:15000])


CELL 5
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
RAW_XC_DIR = PROJECT_ROOT / "data" / "raw" / "xenocanto"

for state in A1_TARGETS:

    metadata_path = RAW_XC_DIR / f"xc_metadata_raw_{state}.json"

    with open(metadata_path, encoding="utf-8") as f:
        recs = json.load(f)[:A1_TARGETS[state]]

    species = {
        r["en"]
        for r in recs
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(recs)} recordings, "
        f"{len(species)} unique species"
    )

CELL 6
print("A1 Xeno-canto sample")
print("=" * 40)

sample_species = {
    "CA": 6,
    "AZ": 16,
    "TX": 15,
}

for state, target in A1_TARGETS.items():
    print(
        f"{state}: "
        f"{target} recordings | "
        f"{sample_species[state]} unique species"
    )

print(f"\nTotal recordings: {sum(A1_TARGETS.values())}")
print("Audio download failures: 0")
print("Restricted/skipped recordings: 0")

CELL 7
for state in A1_TARGETS:
    with open(RA

In [15]:
# ============================================================
# 06F.2 — Full species representation audit
# ============================================================

print("=" * 80)
print("FULL SPECIES REPRESENTATION AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# Load recording-level acoustic feature table
# ------------------------------------------------------------

FEATURE_TABLE_PATH = (
    DATA_PROCESSED_DIR
    / "acoustic_features_recording.csv"
)

recording_df = pd.read_csv(FEATURE_TABLE_PATH)

required_cols = [
    "recording_id",
    "species_id",
]

missing_cols = [
    col for col in required_cols
    if col not in recording_df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

n_recordings = recording_df["recording_id"].nunique()
n_species = recording_df["species_id"].nunique()

print(f"Rows:                         {len(recording_df):,}")
print(f"Unique recordings:            {n_recordings:,}")
print(f"Unique species:               {n_species:,}")

assert len(recording_df) == 275
assert n_recordings == 275
assert n_species == 159

# ------------------------------------------------------------
# Species recording counts
# ------------------------------------------------------------

species_counts = (
    recording_df
    .groupby("species_id", dropna=False)
    .agg(
        n_recordings=("recording_id", "nunique")
    )
    .sort_values(
        "n_recordings",
        ascending=False
    )
)

species_counts["n_species_with_this_count"] = (
    species_counts["n_recordings"]
    .map(
        species_counts["n_recordings"]
        .value_counts()
    )
)

print("\nSpecies-level recording counts:")
display(species_counts)

# ------------------------------------------------------------
# Frequency distribution
# ------------------------------------------------------------

count_distribution = (
    species_counts["n_recordings"]
    .value_counts()
    .sort_index()
    .rename_axis("recordings_per_species")
    .reset_index(name="n_species")
)

count_distribution["n_recordings"] = (
    count_distribution["recordings_per_species"]
    * count_distribution["n_species"]
)

count_distribution["pct_species"] = (
    100
    * count_distribution["n_species"]
    / n_species
)

count_distribution["pct_recordings"] = (
    100
    * count_distribution["n_recordings"]
    / n_recordings
)

print("\n" + "-" * 80)
print("SPECIES-COUNT DISTRIBUTION")
print("-" * 80)

display(count_distribution)

# ------------------------------------------------------------
# Cumulative thresholds
# ------------------------------------------------------------

threshold_rows = []

for minimum_recordings in range(
    1,
    int(species_counts["n_recordings"].max()) + 1
):

    eligible = (
        species_counts["n_recordings"]
        >= minimum_recordings
    )

    threshold_rows.append({
        "minimum_recordings_per_species":
            minimum_recordings,

        "n_species":
            int(eligible.sum()),

        "n_recordings":
            int(
                species_counts.loc[
                    eligible,
                    "n_recordings"
                ].sum()
            ),

        "pct_species":
            100 * eligible.sum() / n_species,

        "pct_recordings":
            100
            * species_counts.loc[
                eligible,
                "n_recordings"
            ].sum()
            / n_recordings,
    })

threshold_summary = pd.DataFrame(threshold_rows)

print("\n" + "-" * 80)
print("THRESHOLD SUMMARY")
print("-" * 80)

display(threshold_summary)

# ------------------------------------------------------------
# State × species representation
# ------------------------------------------------------------

state_species = (
    recording_df
    .groupby(
        ["state", "species_id"],
        dropna=False
    )
    .agg(
        n_recordings=("recording_id", "nunique")
    )
    .reset_index()
)

state_species_summary = (
    state_species
    .groupby("state")
    .agg(
        n_species=("species_id", "nunique"),
        n_recordings=("n_recordings", "sum"),
        median_recordings_per_species=(
            "n_recordings",
            "median"
        ),
        max_recordings_per_species=(
            "n_recordings",
            "max"
        ),
    )
    .reset_index()
)

print("\n" + "-" * 80)
print("STATE-LEVEL SPECIES REPRESENTATION")
print("-" * 80)

display(state_species_summary)

# ------------------------------------------------------------
# Identify sparse species explicitly
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SPARSE SPECIES")
print("-" * 80)

for threshold in [1, 2, 3, 4, 5]:

    n_species_threshold = int(
        (species_counts["n_recordings"] >= threshold).sum()
    )

    n_recordings_threshold = int(
        species_counts.loc[
            species_counts["n_recordings"] >= threshold,
            "n_recordings"
        ].sum()
    )

    print(
        f">= {threshold} recordings/species: "
        f"{n_species_threshold} species, "
        f"{n_recordings_threshold} recordings"
    )

print("\nAudit complete.")

FULL SPECIES REPRESENTATION AUDIT
Rows:                         275
Unique recordings:            275
Unique species:               159

Species-level recording counts:


,n_recordings,n_species_with_this_count
species_id,,
Anas carolinensis,5,7
Archilochus alexandri,5,7
Selasphorus platycercus,5,7
Mareca strepera,5,7
Phalaenoptilus nuttallii,5,7
...,...,...
Tyrannus tyrannus,1,103
Tyrannus vociferans,1,103
Tyto furcata,1,103



--------------------------------------------------------------------------------
SPECIES-COUNT DISTRIBUTION
--------------------------------------------------------------------------------


,recordings_per_species,n_species,n_recordings,pct_species,pct_recordings
0,1,103,103,64.779874,37.454545
1,2,23,46,14.465409,16.727273
2,3,13,39,8.176101,14.181818
3,4,13,52,8.176101,18.909091
4,5,7,35,4.402516,12.727273



--------------------------------------------------------------------------------
THRESHOLD SUMMARY
--------------------------------------------------------------------------------


,minimum_recordings_per_species,n_species,n_recordings,pct_species,pct_recordings
0,1,159,275,100.000000,100.000000
1,2,56,172,35.220126,62.545455
2,3,33,126,20.754717,45.818182
3,4,20,87,12.578616,31.636364
4,5,7,35,4.402516,12.727273



--------------------------------------------------------------------------------
STATE-LEVEL SPECIES REPRESENTATION
--------------------------------------------------------------------------------


,state,n_species,n_recordings,median_recordings_per_species,max_recordings_per_species
0,AZ,44,75,2.0,2
1,CA,66,100,2.0,2
2,TX,100,100,1.0,1



--------------------------------------------------------------------------------
SPARSE SPECIES
--------------------------------------------------------------------------------
>= 1 recordings/species: 159 species, 275 recordings
>= 2 recordings/species: 56 species, 172 recordings
>= 3 recordings/species: 33 species, 126 recordings
>= 4 recordings/species: 20 species, 87 recordings
>= 5 recordings/species: 7 species, 35 recordings

Audit complete.


In [16]:
# ============================================================
# 06F.2 — Full species representation audit
#              corrected distribution calculation
# ============================================================

print("=" * 80)
print("FULL SPECIES REPRESENTATION AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# Load recording-level acoustic feature table
# ------------------------------------------------------------

FEATURE_TABLE_PATH = (
    DATA_PROCESSED_DIR
    / "acoustic_features_recording.csv"
)

recording_df = pd.read_csv(FEATURE_TABLE_PATH)

required_cols = [
    "recording_id",
    "species_id",
    "state",
]

missing_cols = [
    col for col in required_cols
    if col not in recording_df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

n_recordings = recording_df["recording_id"].nunique()
n_species = recording_df["species_id"].nunique()

print(f"Rows:                         {len(recording_df):,}")
print(f"Unique recordings:            {n_recordings:,}")
print(f"Unique species:               {n_species:,}")

assert len(recording_df) == 275
assert n_recordings == 275
assert n_species == 159

# ------------------------------------------------------------
# Species-level recording counts
# ------------------------------------------------------------

species_counts = (
    recording_df
    .groupby("species_id", dropna=False)
    .agg(
        n_recordings=("recording_id", "nunique")
    )
    .sort_values(
        "n_recordings",
        ascending=False
    )
)

print("\nSpecies-level recording counts:")
display(species_counts)

# ------------------------------------------------------------
# Correct frequency distribution
# ------------------------------------------------------------

count_distribution = (
    species_counts["n_recordings"]
    .value_counts()
    .sort_index()
    .rename_axis("recordings_per_species")
    .reset_index(name="n_species")
)

count_distribution["n_recordings"] = (
    count_distribution["recordings_per_species"]
    * count_distribution["n_species"]
)

count_distribution["pct_species"] = (
    100
    * count_distribution["n_species"]
    / n_species
)

count_distribution["pct_recordings"] = (
    100
    * count_distribution["n_recordings"]
    / n_recordings
)

print("\n" + "-" * 80)
print("CORRECT SPECIES-COUNT DISTRIBUTION")
print("-" * 80)

display(count_distribution)

# ------------------------------------------------------------
# Threshold summary
# ------------------------------------------------------------

threshold_rows = []

max_reps = int(
    species_counts["n_recordings"].max()
)

for minimum_recordings in range(1, max_reps + 1):

    eligible = (
        species_counts["n_recordings"]
        >= minimum_recordings
    )

    n_species_threshold = int(eligible.sum())

    n_recordings_threshold = int(
        species_counts.loc[
            eligible,
            "n_recordings"
        ].sum()
    )

    threshold_rows.append({
        "minimum_recordings_per_species":
            minimum_recordings,

        "n_species":
            n_species_threshold,

        "n_recordings":
            n_recordings_threshold,

        "pct_species":
            100 * n_species_threshold / n_species,

        "pct_recordings":
            100 * n_recordings_threshold / n_recordings,
    })

threshold_summary = pd.DataFrame(threshold_rows)

print("\n" + "-" * 80)
print("THRESHOLD SUMMARY")
print("-" * 80)

display(threshold_summary)

# ------------------------------------------------------------
# State-level species representation
# ------------------------------------------------------------

state_species = (
    recording_df
    .groupby(
        ["state", "species_id"],
        dropna=False
    )
    .agg(
        n_recordings=("recording_id", "nunique")
    )
    .reset_index()
)

state_species_summary = (
    state_species
    .groupby("state")
    .agg(
        n_species=("species_id", "nunique"),
        n_recordings=("n_recordings", "sum"),
        median_recordings_per_species=(
            "n_recordings",
            "median"
        ),
        max_recordings_per_species=(
            "n_recordings",
            "max"
        ),
    )
    .reset_index()
)

print("\n" + "-" * 80)
print("STATE-LEVEL SPECIES REPRESENTATION")
print("-" * 80)

display(state_species_summary)

# ------------------------------------------------------------
# State composition of each threshold cohort
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("STATE COMPOSITION OF THRESHOLD COHORTS")
print("-" * 80)

for minimum_recordings in range(1, max_reps + 1):

    eligible_species = set(
        species_counts.index[
            species_counts["n_recordings"]
            >= minimum_recordings
        ]
    )

    cohort = recording_df[
        recording_df["species_id"]
        .isin(eligible_species)
    ]

    print(
        f"\n>= {minimum_recordings} recordings/species"
    )

    print(
        cohort["state"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print(
        f"Total recordings: {len(cohort)}"
    )

    print(
        f"Total species: "
        f"{cohort['species_id'].nunique()}"
    )

print("\nAudit complete.")

FULL SPECIES REPRESENTATION AUDIT
Rows:                         275
Unique recordings:            275
Unique species:               159

Species-level recording counts:


,n_recordings
species_id,
Anas carolinensis,5
Archilochus alexandri,5
Selasphorus platycercus,5
Mareca strepera,5
Phalaenoptilus nuttallii,5
...,...
Tyrannus tyrannus,1
Tyrannus vociferans,1
Tyto furcata,1



--------------------------------------------------------------------------------
CORRECT SPECIES-COUNT DISTRIBUTION
--------------------------------------------------------------------------------


,recordings_per_species,n_species,n_recordings,pct_species,pct_recordings
0,1,103,103,64.779874,37.454545
1,2,23,46,14.465409,16.727273
2,3,13,39,8.176101,14.181818
3,4,13,52,8.176101,18.909091
4,5,7,35,4.402516,12.727273



--------------------------------------------------------------------------------
THRESHOLD SUMMARY
--------------------------------------------------------------------------------


,minimum_recordings_per_species,n_species,n_recordings,pct_species,pct_recordings
0,1,159,275,100.000000,100.000000
1,2,56,172,35.220126,62.545455
2,3,33,126,20.754717,45.818182
3,4,20,87,12.578616,31.636364
4,5,7,35,4.402516,12.727273



--------------------------------------------------------------------------------
STATE-LEVEL SPECIES REPRESENTATION
--------------------------------------------------------------------------------


,state,n_species,n_recordings,median_recordings_per_species,max_recordings_per_species
0,AZ,44,75,2.0,2
1,CA,66,100,2.0,2
2,TX,100,100,1.0,1



--------------------------------------------------------------------------------
STATE COMPOSITION OF THRESHOLD COHORTS
--------------------------------------------------------------------------------

>= 1 recordings/species
state
AZ     75
CA    100
TX    100
Total recordings: 275
Total species: 159

>= 2 recordings/species
state
AZ    72
CA    79
TX    21
Total recordings: 172
Total species: 56

>= 3 recordings/species
state
AZ    48
CA    59
TX    19
Total recordings: 126
Total species: 33

>= 4 recordings/species
state
AZ    40
CA    37
TX    10
Total recordings: 87
Total species: 20

>= 5 recordings/species
state
AZ    14
CA    14
TX     7
Total recordings: 35
Total species: 7

Audit complete.


In [17]:
# 06F.3 — Species × State Structure
#
# Purpose:
# Before running the PCA–species association test, quantify how species
# representation is distributed across states.
#
# This determines whether an unrestricted species-label permutation would
# inadvertently confound species association with geographic/state structure.

# -------------------------------------------------------------------
# 1. Build species × state recording-count table
# -------------------------------------------------------------------

species_state_counts = pd.crosstab(
    recording_df["species_id"],
    recording_df["state"]
).sort_index()

print("=" * 80)
print("SPECIES × STATE RECORDING STRUCTURE")
print("=" * 80)

print(f"Species: {species_state_counts.shape[0]}")
print(f"States:  {species_state_counts.shape[1]}")
print(f"Total recordings: {species_state_counts.to_numpy().sum()}")

display(species_state_counts)

# -------------------------------------------------------------------
# 2. Number of states represented by each species
# -------------------------------------------------------------------

species_n_states = (
    (species_state_counts > 0)
    .sum(axis=1)
    .rename("n_states")
)

state_span_summary = (
    species_n_states
    .value_counts()
    .sort_index()
    .rename_axis("n_states")
    .reset_index(name="n_species")
)

print("\n" + "=" * 80)
print("SPECIES GEOGRAPHIC SPAN")
print("=" * 80)

display(state_span_summary)

# -------------------------------------------------------------------
# 3. How many species are state-specific vs shared?
# -------------------------------------------------------------------

n_species_total = len(species_n_states)
n_species_one_state = int((species_n_states == 1).sum())
n_species_two_states = int((species_n_states == 2).sum())
n_species_three_states = int((species_n_states == 3).sum())

print("\nSpecies represented in exactly one state:",
      f"{n_species_one_state} / {n_species_total}"
      f" ({100 * n_species_one_state / n_species_total:.1f}%)")

print("Species represented in exactly two states:",
      f"{n_species_two_states} / {n_species_total}"
      f" ({100 * n_species_two_states / n_species_total:.1f}%)")

print("Species represented in all three states:",
      f"{n_species_three_states} / {n_species_total}"
      f" ({100 * n_species_three_states / n_species_total:.1f}%)")

# -------------------------------------------------------------------
# 4. State-specific species in the ≥2-recording cohort
# -------------------------------------------------------------------

species_state_subset = species_state_counts.loc[
    species_state_counts.index.isin(species_counts.loc[
        species_counts["n_recordings"] >= 2
    ].index)
].copy()

print("\n" + "=" * 80)
print("≥2-RECORDING SPECIES COHORT: SPECIES × STATE")
print("=" * 80)

print(f"Species: {species_state_subset.shape[0]}")
print(f"Recordings: {int(species_state_subset.to_numpy().sum())}")

display(species_state_subset)

# -------------------------------------------------------------------
# 5. State distribution of the ≥2 cohort
# -------------------------------------------------------------------

cohort_state_summary = (
    species_state_subset
    .sum(axis=0)
    .rename("n_recordings")
    .to_frame()
)

cohort_state_summary["pct_recordings"] = (
    100
    * cohort_state_summary["n_recordings"]
    / cohort_state_summary["n_recordings"].sum()
)

print("\n" + "=" * 80)
print("≥2-RECORDING COHORT: STATE COMPOSITION")
print("=" * 80)

display(cohort_state_summary)

# -------------------------------------------------------------------
# 6. Identify species occurring in multiple states
# -------------------------------------------------------------------

shared_species = species_state_subset.loc[
    (species_state_subset > 0).sum(axis=1) >= 2
]

print("\n" + "=" * 80)
print("SPECIES SHARED ACROSS ≥2 STATES")
print("=" * 80)

print(f"Shared species: {len(shared_species)}")

if len(shared_species) > 0:
    display(shared_species)

# -------------------------------------------------------------------
# 7. Check whether each state contains repeated species
# -------------------------------------------------------------------

state_species_repeat_summary = []

for state in sorted(recording_df["state"].dropna().unique()):
    state_counts = (
        recording_df.loc[
            recording_df["state"] == state
        ]
        .groupby("species_id")["recording_id"]
        .nunique()
    )

    state_species_repeat_summary.append({
        "state": state,
        "n_species": int(state_counts.shape[0]),
        "n_species_with_2plus": int((state_counts >= 2).sum()),
        "n_recordings": int(state_counts.sum()),
        "max_recordings_per_species": int(state_counts.max()),
    })

state_species_repeat_summary = pd.DataFrame(
    state_species_repeat_summary
)

print("\n" + "=" * 80)
print("WITHIN-STATE SPECIES REPLICATION")
print("=" * 80)

display(state_species_repeat_summary)

# -------------------------------------------------------------------
# 8. Preliminary interpretation
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("PRELIMINARY PERMUTATION IMPLICATION")
print("=" * 80)

print(
    "Species and state are not independent grouping variables in this "
    "sampling design. Therefore, an unrestricted permutation of species "
    "labels could partially convert geographic structure into apparent "
    "species association."
)

print(
    "The next analysis should therefore compare an unrestricted permutation "
    "scheme with a state-restricted permutation scheme, provided that the "
    "within-state replication structure supports the latter."
)

SPECIES × STATE RECORDING STRUCTURE
Species: 159
States:  3
Total recordings: 275


state,AZ,CA,TX
species_id,,,
Accipiter cooperii,0,0,1
Aechmophorus clarkii,0,0,1
Aeronautes saxatalis,1,1,0
Aix sponsa,0,1,0
Alectoris chukar,0,2,1
...,...,...,...
Tyrannus tyrannus,0,0,1
Tyrannus vociferans,0,0,1
Tyto furcata,0,0,1



SPECIES GEOGRAPHIC SPAN


,n_states,n_species
0,1,118
1,2,31
2,3,10



Species represented in exactly one state: 118 / 159 (74.2%)
Species represented in exactly two states: 31 / 159 (19.5%)
Species represented in all three states: 10 / 159 (6.3%)

≥2-RECORDING SPECIES COHORT: SPECIES × STATE
Species: 56
Recordings: 172


...


SPECIES SHARED ACROSS ≥2 STATES
Shared species: 41


state,AZ,CA,TX
species_id,,,
Aeronautes saxatalis,1,1,0
Alectoris chukar,0,2,1
Anas acuta,1,1,0
Anas carolinensis,2,2,1
Anas platyrhynchos,2,1,1
Anser albifrons,2,2,0
Anser caerulescens,1,1,0
Antrostomus arizonae,2,1,1
Antrostomus vociferus,0,2,1



WITHIN-STATE SPECIES REPLICATION


,state,n_species,n_species_with_2plus,n_recordings,max_recordings_per_species
0,AZ,44,31,75,2
1,CA,66,34,100,2
2,TX,100,0,100,1



PRELIMINARY PERMUTATION IMPLICATION
Species and state are not independent grouping variables in this sampling design. Therefore, an unrestricted permutation of species labels could partially convert geographic structure into apparent species association.
The next analysis should therefore compare an unrestricted permutation scheme with a state-restricted permutation scheme, provided that the within-state replication structure supports the latter.


In [19]:
# 06F.4a — Locate Existing PCA Score/Result Artifacts

print("=" * 80)
print("PCA ANALYSIS DIRECTORY")
print("=" * 80)

print("PCA_ANALYSIS_DIR:")
print(PCA_ANALYSIS_DIR)
print()

if not PCA_ANALYSIS_DIR.exists():
    raise FileNotFoundError(
        f"PCA analysis directory does not exist:\n{PCA_ANALYSIS_DIR}"
    )

for path in sorted(PCA_ANALYSIS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PCA_ANALYSIS_DIR))

PCA ANALYSIS DIRECTORY
PCA_ANALYSIS_DIR:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis

baseline_components.npy
baseline_explained_variance.npy
baseline_explained_variance_ratio.npy
baseline_feature_names.json
baseline_pa_null_95pct.npy
baseline_pa_null_mean.npy
baseline_pa_observed_eigenvalues.npy
baseline_pa_retained.npy
baseline_scores.npy
baseline_singular_values.npy
block_balanced_components.npy
block_balanced_explained_variance.npy
block_balanced_explained_variance_ratio.npy
block_balanced_feature_names.json
block_balanced_pa_null_95pct.npy
block_balanced_pa_null_mean.npy
block_balanced_pa_observed_eigenvalues.npy
block_balanced_pa_retained.npy
block_balanced_scores.npy
block_balanced_singular_values.npy
bootstrap_stability\baseline_bootstrap_explained_variance_ratio.npy
bootstrap_stability\baseline_principal_angles_deg.npy
bootstrap_stability\block_balanced_bootstrap_explained_variance_ratio.npy
bootstrap_stability\block_balanced_princ

In [23]:
# 06F.4 — Species vs. State Structure in PCA Space
#
# Purpose:
# Quantify whether the retained PCA structure is associated with state
# as well as species, before selecting the permutation design.
#
# PCA was constructed without species or state labels. These variables are
# introduced only here for post-hoc ecological interpretation.

# -------------------------------------------------------------------
# 1. Load baseline PCA scores from 06C
# -------------------------------------------------------------------

baseline_scores_array = np.load(
    PCA_ANALYSIS_DIR / "baseline_scores.npy"
)

baseline_feature_names = json.loads(
    (PCA_ANALYSIS_DIR / "baseline_feature_names.json").read_text()
)

# 06C stores parallel-analysis retention as a Boolean/indicator vector,
# one value for each PCA component.
baseline_pa_retained = np.asarray(
    np.load(PCA_ANALYSIS_DIR / "baseline_pa_retained.npy")
)

if baseline_pa_retained.ndim != 1:
    raise ValueError(
        "Expected baseline_pa_retained.npy to be a 1-D retention mask, "
        f"but found shape {baseline_pa_retained.shape}."
    )

if baseline_pa_retained.size != baseline_scores_array.shape[1]:
    raise ValueError(
        "Retention mask length does not match the number of PCA scores: "
        f"mask={baseline_pa_retained.size}, "
        f"scores={baseline_scores_array.shape[1]}."
    )

baseline_retained = int(
    baseline_pa_retained.astype(bool).sum()
)

print("=" * 80)
print("BASELINE PCA SCORE AUDIT")
print("=" * 80)

print("Score matrix shape:", baseline_scores_array.shape)
print("Feature count:", len(baseline_feature_names))

print(
    "Parallel-analysis retention mask:",
    baseline_pa_retained.astype(int)
)

print(
    "Parallel-analysis retained PCs:",
    baseline_retained
)

assert baseline_scores_array.shape[0] == N_RECORDINGS_FULL
assert baseline_scores_array.shape[1] == len(baseline_feature_names)
assert np.isfinite(baseline_scores_array).all()

# -------------------------------------------------------------------
# 2. Load the exact recording-level feature table used for PCA
# -------------------------------------------------------------------

recording_feature_table = pd.read_csv(
    DATA_PROCESSED_DIR / "acoustic_features_recording.csv"
)

recording_feature_table["recording_id"] = (
    recording_feature_table["recording_id"].astype(str)
)

recording_metadata = recording_feature_table[
    ["recording_id", "species_id", "state"]
].copy()

assert len(recording_metadata) == baseline_scores_array.shape[0]
assert recording_metadata["recording_id"].is_unique

# -------------------------------------------------------------------
# 3. Reconstruct PCA score table
# -------------------------------------------------------------------

pc_columns = [
    f"PC{i}"
    for i in range(1, baseline_scores_array.shape[1] + 1)
]

baseline_scores = pd.DataFrame(
    baseline_scores_array,
    columns=pc_columns
)

baseline_scores.insert(
    0,
    "recording_id",
    recording_metadata["recording_id"].values
)

pca_species_state = baseline_scores.merge(
    recording_metadata,
    on="recording_id",
    how="inner",
    validate="one_to_one"
)

assert len(pca_species_state) == N_RECORDINGS_FULL
assert pca_species_state["recording_id"].is_unique

print("\nMerged PCA + metadata shape:", pca_species_state.shape)

# -------------------------------------------------------------------
# 4. Define retained PCA subspace
# -------------------------------------------------------------------

N_RETAINED_PCS = baseline_retained

retained_pc_columns = [
    f"PC{i}"
    for i in range(1, N_RETAINED_PCS + 1)
]

X_pca = pca_species_state[
    retained_pc_columns
].to_numpy(dtype=float)

print("\nRetained PCA dimensions:", X_pca.shape)
print("Retained PCs:", retained_pc_columns)

assert X_pca.shape == (
    N_RECORDINGS_FULL,
    N_RETAINED_PCS
)

# -------------------------------------------------------------------
# 5. State-level PCA centroids and dispersion
# -------------------------------------------------------------------

state_centroids = (
    pca_species_state
    .groupby("state")[retained_pc_columns]
    .mean()
)

state_distances = []

for _, row in pca_species_state.iterrows():

    point = row[
        retained_pc_columns
    ].to_numpy(dtype=float)

    centroid = (
        state_centroids
        .loc[row["state"]]
        .to_numpy(dtype=float)
    )

    state_distances.append(
        np.linalg.norm(point - centroid)
    )

pca_species_state[
    "distance_to_state_centroid"
] = state_distances

state_dispersion = (
    pca_species_state
    .groupby("state")[
        "distance_to_state_centroid"
    ]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std"
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("PCA DISPERSION BY STATE")
print("=" * 80)

display(state_dispersion)

# -------------------------------------------------------------------
# 6. Focal ≥2-recording species cohort
# -------------------------------------------------------------------

focal_species = (
    species_counts.loc[
        species_counts["n_recordings"] >= 2
    ]
    .index
)

focal = pca_species_state[
    pca_species_state["species_id"].isin(
        focal_species
    )
].copy()

print("\n" + "=" * 80)
print("FOCAL SPECIES COHORT")
print("=" * 80)

print(
    "Species:",
    focal["species_id"].nunique()
)

print(
    "Recordings:",
    len(focal)
)

assert focal["species_id"].nunique() == N_SPECIES_TARGET
assert len(focal) == N_RECORDINGS_TARGET

# -------------------------------------------------------------------
# 7. Species-level PCA centroids and dispersion
# -------------------------------------------------------------------

species_centroids = (
    focal
    .groupby("species_id")[retained_pc_columns]
    .mean()
)

species_distances = []

for _, row in focal.iterrows():

    point = row[
        retained_pc_columns
    ].to_numpy(dtype=float)

    centroid = (
        species_centroids
        .loc[row["species_id"]]
        .to_numpy(dtype=float)
    )

    species_distances.append(
        np.linalg.norm(point - centroid)
    )

focal[
    "distance_to_species_centroid"
] = species_distances

species_dispersion = (
    focal
    .groupby("species_id")[
        "distance_to_species_centroid"
    ]
    .agg(
        n="count",
        mean="mean",
        median="median"
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("PCA DISPERSION BY SPECIES")
print("=" * 80)

display(species_dispersion)

# -------------------------------------------------------------------
# 8. Compare descriptive within-group dispersion
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("DISPERSION COMPARISON")
print("=" * 80)

print(
    "Mean distance to state centroid:",
    f"{pca_species_state['distance_to_state_centroid'].mean():.4f}"
)

print(
    "Mean distance to species centroid "
    "(≥2-recording cohort):",
    f"{focal['distance_to_species_centroid'].mean():.4f}"
)

# -------------------------------------------------------------------
# 9. State sample sizes
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("STATE SAMPLE SIZES")
print("=" * 80)

display(
    pca_species_state["state"]
    .value_counts()
    .sort_index()
    .rename("n_recordings")
    .to_frame()
)

# -------------------------------------------------------------------
# 10. Species replication in focal cohort
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("SPECIES REPLICATION IN FOCAL COHORT")
print("=" * 80)

display(
    focal["species_id"]
    .value_counts()
    .value_counts()
    .sort_index()
    .rename_axis("recordings_per_species")
    .rename("n_species")
    .to_frame()
)

# -------------------------------------------------------------------
# 11. Design conclusion
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("DESIGN CONCLUSION")
print("=" * 80)

print(
    "The ≥2-recording cohort provides species-level replication, "
    "but species and state remain partially confounded by the acquisition design."
)

print(
    "Because TX has no repeated species within state, a permutation scheme "
    "restricted entirely within state cannot serve as the primary test "
    "for the full three-state cohort."
)

print(
    "The next step is to quantify species association in PCA space using "
    "a multivariate statistic and an explicit permutation design that "
    "preserves the relevant sampling structure."
)

BASELINE PCA SCORE AUDIT
Score matrix shape: (275, 50)
Feature count: 50
Parallel-analysis retention mask: [1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]
Parallel-analysis retained PCs: 9

Merged PCA + metadata shape: (275, 53)

Retained PCA dimensions: (275, 9)
Retained PCs: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9']

PCA DISPERSION BY STATE


,state,n,mean,median,std
0,AZ,75,4.930086,4.415891,2.093862
1,CA,100,5.369777,4.942237,2.145882
2,TX,100,5.543131,5.151649,2.296787



FOCAL SPECIES COHORT
Species: 56
Recordings: 172

PCA DISPERSION BY SPECIES


,species_id,n,mean,median
0,Aeronautes saxatalis,2,2.732808,2.732808
1,Alectoris chukar,3,5.933856,5.588189
2,Alopochen aegyptiaca,2,6.560954,6.560954
3,Anas acuta,2,1.604229,1.604229
4,Anas carolinensis,5,2.615072,2.698360
5,Anas diazi,2,2.057193,2.057193
6,Anas platyrhynchos,4,2.913316,2.480718
7,Anser albifrons,4,4.194226,4.156373
8,Anser caerulescens,2,3.493584,3.493584
9,Antrostomus arizonae,4,3.207344,3.334251



DISPERSION COMPARISON
Mean distance to state centroid: 5.3129
Mean distance to species centroid (≥2-recording cohort): 3.8141

STATE SAMPLE SIZES


,n_recordings
state,
AZ,75
CA,100
TX,100



SPECIES REPLICATION IN FOCAL COHORT


,n_species
recordings_per_species,
2,23
3,13
4,13
5,7



DESIGN CONCLUSION
The ≥2-recording cohort provides species-level replication, but species and state remain partially confounded by the acquisition design.
Because TX has no repeated species within state, a permutation scheme restricted entirely within state cannot serve as the primary test for the full three-state cohort.
The next step is to quantify species association in PCA space using a multivariate statistic and an explicit permutation design that preserves the relevant sampling structure.


In [24]:
# 06F.5 — Observed Multivariate Association in PCA Space
#
# Scientific question:
#
# Do recordings from the same species occupy more similar positions
# in the unsupervised PCA space than expected from the overall
# multivariate variation?
#
# This cell computes the observed pseudo-F statistic only.
# NO permutation testing is performed here.
#
# Species and state are evaluated separately because geographic structure
# may confound species-level acoustic association.

# -------------------------------------------------------------------
# 1. Prepare the focal PCA matrix
# -------------------------------------------------------------------

analysis_df = focal.copy()

analysis_df = analysis_df[
    [
        "recording_id",
        "species_id",
        "state",
        *retained_pc_columns
    ]
].copy()

analysis_df = analysis_df.reset_index(drop=True)

X = analysis_df[
    retained_pc_columns
].to_numpy(dtype=float)

species_labels = (
    analysis_df["species_id"]
    .astype(str)
    .to_numpy()
)

state_labels = (
    analysis_df["state"]
    .astype(str)
    .to_numpy()
)

assert X.shape == (
    N_RECORDINGS_TARGET,
    N_RETAINED_PCS
)

assert np.isfinite(X).all()

# -------------------------------------------------------------------
# 2. Helper function: pseudo-F statistic
# -------------------------------------------------------------------
#
# PERMANOVA-style decomposition:
#
# Total SS = sum squared distances from the overall centroid
#
# Between-group SS = sum over groups of
#                    n_g * squared distance(group centroid,
#                                            overall centroid)
#
# Within-group SS = Total SS - Between-group SS
#
# pseudo-F = (SS_between / df_between) /
#            (SS_within / df_within)

def permanova_pseudo_f(X, groups):
    """
    Calculate a PERMANOVA-style pseudo-F statistic.

    Parameters
    ----------
    X : ndarray, shape (n_samples, n_features)
        Multivariate observations.

    groups : array-like, shape (n_samples,)
        Group labels.

    Returns
    -------
    result : dict
        Total, between, and within sums of squares,
        degrees of freedom, pseudo-F, and group count.
    """

    X = np.asarray(X, dtype=float)
    groups = np.asarray(groups)

    if X.ndim != 2:
        raise ValueError(
            f"X must be 2-D; got shape {X.shape}."
        )

    if len(groups) != X.shape[0]:
        raise ValueError(
            "Number of group labels must equal number of observations."
        )

    if not np.isfinite(X).all():
        raise ValueError(
            "X contains non-finite values."
        )

    unique_groups = np.unique(groups)
    n = X.shape[0]
    k = len(unique_groups)

    if k < 2:
        raise ValueError(
            "At least two groups are required."
        )

    # Overall centroid
    grand_centroid = X.mean(axis=0)

    # Total sum of squares
    centered_total = X - grand_centroid
    ss_total = np.sum(centered_total ** 2)

    # Between-group sum of squares
    ss_between = 0.0

    group_sizes = {}

    for group in unique_groups:

        mask = groups == group

        X_group = X[mask]

        n_group = X_group.shape[0]

        group_sizes[group] = n_group

        group_centroid = X_group.mean(axis=0)

        centroid_difference = (
            group_centroid - grand_centroid
        )

        ss_between += (
            n_group
            * np.sum(centroid_difference ** 2)
        )

    # Within-group sum of squares
    ss_within = ss_total - ss_between

    # Numerical precision can create tiny negative values.
    if ss_within < 0 and np.isclose(
        ss_within,
        0.0,
        atol=1e-10
    ):
        ss_within = 0.0

    df_between = k - 1
    df_within = n - k

    if df_within <= 0:
        raise ValueError(
            "Insufficient residual degrees of freedom."
        )

    if ss_within <= 0:
        pseudo_f = np.inf
    else:
        pseudo_f = (
            (ss_between / df_between)
            /
            (ss_within / df_within)
        )

    return {
        "n": n,
        "n_groups": k,
        "ss_total": ss_total,
        "ss_between": ss_between,
        "ss_within": ss_within,
        "df_between": df_between,
        "df_within": df_within,
        "pseudo_F": pseudo_f,
        "group_sizes": group_sizes
    }


# -------------------------------------------------------------------
# 3. Calculate observed species effect
# -------------------------------------------------------------------

species_result = permanova_pseudo_f(
    X,
    species_labels
)

# -------------------------------------------------------------------
# 4. Calculate observed state effect
# -------------------------------------------------------------------

state_result = permanova_pseudo_f(
    X,
    state_labels
)

# -------------------------------------------------------------------
# 5. Display species result
# -------------------------------------------------------------------

print("=" * 80)
print("OBSERVED SPECIES ASSOCIATION")
print("=" * 80)

print(
    "Observations:",
    species_result["n"]
)

print(
    "Species groups:",
    species_result["n_groups"]
)

print(
    "Between-species SS:",
    f"{species_result['ss_between']:.6f}"
)

print(
    "Within-species SS:",
    f"{species_result['ss_within']:.6f}"
)

print(
    "Pseudo-F:",
    f"{species_result['pseudo_F']:.6f}"
)

# -------------------------------------------------------------------
# 6. Display state result
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVED STATE ASSOCIATION")
print("=" * 80)

print(
    "Observations:",
    state_result["n"]
)

print(
    "State groups:",
    state_result["n_groups"]
)

print(
    "Between-state SS:",
    f"{state_result['ss_between']:.6f}"
)

print(
    "Within-state SS:",
    f"{state_result['ss_within']:.6f}"
)

print(
    "Pseudo-F:",
    f"{state_result['pseudo_F']:.6f}"
)

# -------------------------------------------------------------------
# 7. Compare variance partitioning descriptively
# -------------------------------------------------------------------

species_between_pct = (
    100
    * species_result["ss_between"]
    / species_result["ss_total"]
)

state_between_pct = (
    100
    * state_result["ss_between"]
    / state_result["ss_total"]
)

comparison = pd.DataFrame({
    "grouping": [
        "species",
        "state"
    ],
    "n_groups": [
        species_result["n_groups"],
        state_result["n_groups"]
    ],
    "SS_between": [
        species_result["ss_between"],
        state_result["ss_between"]
    ],
    "SS_within": [
        species_result["ss_within"],
        state_result["ss_within"]
    ],
    "pseudo_F": [
        species_result["pseudo_F"],
        state_result["pseudo_F"]
    ],
    "pct_total_SS_between": [
        species_between_pct,
        state_between_pct
    ]
})

print("\n" + "=" * 80)
print("SPECIES VS. STATE — OBSERVED STRUCTURE")
print("=" * 80)

display(comparison)

# -------------------------------------------------------------------
# 8. Validate sums of squares
# -------------------------------------------------------------------

assert np.isclose(
    species_result["ss_total"],
    species_result["ss_between"]
    + species_result["ss_within"],
    rtol=1e-10,
    atol=1e-10
)

assert np.isclose(
    state_result["ss_total"],
    state_result["ss_between"]
    + state_result["ss_within"],
    rtol=1e-10,
    atol=1e-10
)

print("\n" + "=" * 80)
print("VALIDATION")
print("=" * 80)

print("Species SS decomposition: PASSED")
print("State SS decomposition:   PASSED")

# -------------------------------------------------------------------
# 9. Interpretation guardrail
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("INTERPRETATION GUARDRAIL")
print("=" * 80)

print(
    "These pseudo-F values are observed statistics only."
)

print(
    "They do NOT establish statistical significance until compared "
    "with an appropriate permutation null distribution."
)

print(
    "A larger species pseudo-F than state pseudo-F would be consistent "
    "with stronger species grouping in the observed PCA space, but this "
    "comparison is descriptive and should not be interpreted as evidence "
    "that species effects are independent of geographic sampling."
)

OBSERVED SPECIES ASSOCIATION
Observations: 172
Species groups: 56
Between-species SS: 2884.182682
Within-species SS: 3120.935610
Pseudo-F: 1.949096

OBSERVED STATE ASSOCIATION
Observations: 172
State groups: 3
Between-state SS: 157.015740
Within-state SS: 5848.102552
Pseudo-F: 2.268741

SPECIES VS. STATE — OBSERVED STRUCTURE


,grouping,n_groups,SS_between,SS_within,pseudo_F,pct_total_SS_between
0,species,56,2884.182682,3120.935610,1.949096,48.028741
1,state,3,157.015740,5848.102552,2.268741,2.614699



VALIDATION
Species SS decomposition: PASSED
State SS decomposition:   PASSED

INTERPRETATION GUARDRAIL
These pseudo-F values are observed statistics only.
They do NOT establish statistical significance until compared with an appropriate permutation null distribution.
A larger species pseudo-F than state pseudo-F would be consistent with stronger species grouping in the observed PCA space, but this comparison is descriptive and should not be interpreted as evidence that species effects are independent of geographic sampling.


In [26]:
# 06F.6 — State-Adjusted Species Association
#
# Scientific question:
#
# Does species identity explain multivariate acoustic structure in PCA space
# AFTER accounting for geographic/state structure?
#
# Reduced model:
#     Y ~ State
#
# Full model:
#     Y ~ State + Species
#
# The species effect is evaluated as the additional multivariate variation
# explained after state is accounted for.
#
# Significance is assessed using a Freedman–Lane residual permutation:
#   1. Fit the reduced State-only model.
#   2. Permute its residuals.
#   3. Reconstruct the response under the reduced model.
#   4. Fit the full State + Species model.
#   5. Calculate the species pseudo-F.
#
# IMPORTANT:
# This tests acoustic association with species identity.
# It does NOT test whether PCA directly measures biodiversity or richness.

# -------------------------------------------------------------------
# 1. Prepare analysis matrix
# -------------------------------------------------------------------

analysis_df = focal.copy()

analysis_df = analysis_df[
    [
        "recording_id",
        "species_id",
        "state",
        *retained_pc_columns
    ]
].copy()

analysis_df = analysis_df.reset_index(drop=True)

Y = analysis_df[
    retained_pc_columns
].to_numpy(dtype=float)

species_labels = (
    analysis_df["species_id"]
    .astype(str)
    .to_numpy()
)

state_labels = (
    analysis_df["state"]
    .astype(str)
    .to_numpy()
)

n = Y.shape[0]

assert n == N_RECORDINGS_TARGET
assert Y.shape[1] == N_RETAINED_PCS
assert np.isfinite(Y).all()

# -------------------------------------------------------------------
# 2. Construct design matrices
# -------------------------------------------------------------------

state_categories = np.sort(
    np.unique(state_labels)
)

species_categories = np.sort(
    np.unique(species_labels)
)

# Intercept + state indicators
X_state = np.column_stack([
    np.ones(n),
    *[
        (state_labels == state).astype(float)
        for state in state_categories[1:]
    ]
])

# Species indicators with first species as reference
X_species = np.column_stack([
    (species_labels == species).astype(float)
    for species in species_categories[1:]
])

# Full model: intercept + state + species
X_full = np.column_stack([
    X_state,
    X_species
])

# -------------------------------------------------------------------
# 3. Check model ranks
# -------------------------------------------------------------------

rank_state = np.linalg.matrix_rank(X_state)
rank_full = np.linalg.matrix_rank(X_full)

df_species = rank_full - rank_state
df_residual = n - rank_full

print("=" * 80)
print("MODEL DESIGN")
print("=" * 80)

print("Observations:", n)
print("PCA dimensions:", Y.shape[1])
print("Species groups:", len(species_categories))
print("State groups:", len(state_categories))

print("\nState-only design shape:", X_state.shape)
print("State-only rank:", rank_state)

print("\nFull design shape:", X_full.shape)
print("Full model rank:", rank_full)

print("\nSpecies degrees of freedom:", df_species)
print("Residual degrees of freedom:", df_residual)

assert rank_state == len(state_categories)
assert rank_full > rank_state
assert df_residual > 0

# -------------------------------------------------------------------
# 4. Helper function
# -------------------------------------------------------------------

def fitted_values(X, Y):
    """
    Multivariate least-squares fitted values.

    X: n observations × p parameters
    Y: n observations × q response dimensions

    Returns:
        n observations × q fitted response matrix
    """

    return X @ np.linalg.pinv(X) @ Y


# -------------------------------------------------------------------
# 5. Fit reduced and full observed models
# -------------------------------------------------------------------

Yhat_state = fitted_values(
    X_state,
    Y
)

Yhat_full = fitted_values(
    X_full,
    Y
)

residual_state = (
    Y - Yhat_state
)

residual_full = (
    Y - Yhat_full
)

SS_resid_state = np.sum(
    residual_state ** 2
)

SS_resid_full = np.sum(
    residual_full ** 2
)

# Additional SS explained by species after state
SS_species_adjusted = (
    SS_resid_state
    - SS_resid_full
)

# Numerical precision guard
if (
    SS_species_adjusted < 0
    and np.isclose(
        SS_species_adjusted,
        0.0,
        atol=1e-10
    )
):
    SS_species_adjusted = 0.0

# -------------------------------------------------------------------
# 6. Observed state-adjusted pseudo-F
# -------------------------------------------------------------------

MS_species_adjusted = (
    SS_species_adjusted
    / df_species
)

MS_residual = (
    SS_resid_full
    / df_residual
)

pseudo_F_adjusted = (
    MS_species_adjusted
    / MS_residual
)

# -------------------------------------------------------------------
# 7. Partial R²
# -------------------------------------------------------------------

partial_R2_species = (
    SS_species_adjusted
    /
    (
        SS_species_adjusted
        + SS_resid_full
    )
)

# -------------------------------------------------------------------
# 8. Display observed result
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVED STATE-ADJUSTED SPECIES ASSOCIATION")
print("=" * 80)

print(
    "Species SS after accounting for state:",
    f"{SS_species_adjusted:.6f}"
)

print(
    "Residual SS after state + species:",
    f"{SS_resid_full:.6f}"
)

print(
    "Species degrees of freedom:",
    df_species
)

print(
    "Residual degrees of freedom:",
    df_residual
)

print(
    "State-adjusted pseudo-F:",
    f"{pseudo_F_adjusted:.6f}"
)

print(
    "Partial R² for species:",
    f"{partial_R2_species:.6f}"
)

print(
    "Partial R² (%):",
    f"{100 * partial_R2_species:.3f}%"
)

# -------------------------------------------------------------------
# 9. Validate observed dimensions
# -------------------------------------------------------------------

assert Yhat_state.shape == Y.shape
assert Yhat_full.shape == Y.shape
assert residual_state.shape == Y.shape
assert residual_full.shape == Y.shape

print("\nObserved model dimension checks: PASSED")

# -------------------------------------------------------------------
# 10. Freedman–Lane permutation test
# -------------------------------------------------------------------

N_PERMUTATIONS = 4999
PERMUTATION_SEED = 42

rng = np.random.default_rng(
    PERMUTATION_SEED
)

# The reduced-model fitted values remain fixed.
Yhat_reduced = Yhat_state

# Precompute the projection matrix for the full model:
#
# H_full = X_full (X_full' X_full)^-1 X_full'
#
# Using pinv makes this robust to the categorical design matrix.

H_full = (
    X_full
    @ np.linalg.pinv(X_full)
)

# Check that the projection matrix has the correct dimensions.
assert H_full.shape == (n, n)

permutation_F = np.empty(
    N_PERMUTATIONS,
    dtype=float
)

for i in range(N_PERMUTATIONS):

    # ---------------------------------------------------------------
    # Permute residual rows
    # ---------------------------------------------------------------

    permutation_indices = rng.permutation(n)

    permuted_residuals = (
        residual_state[
            permutation_indices
        ]
    )

    # ---------------------------------------------------------------
    # Reconstruct response under the reduced model
    # ---------------------------------------------------------------

    Y_permuted = (
        Yhat_reduced
        + permuted_residuals
    )

    # ---------------------------------------------------------------
    # Reduced-model residual SS
    # ---------------------------------------------------------------

    residual_reduced_perm = (
        Y_permuted
        - Yhat_reduced
    )

    SS_reduced_perm = np.sum(
        residual_reduced_perm ** 2
    )

    # ---------------------------------------------------------------
    # Full-model fitted values
    #
    # IMPORTANT:
    # H_full is n × n, so this returns n × q fitted values.
    # ---------------------------------------------------------------

    Yhat_full_perm = (
        H_full
        @ Y_permuted
    )

    assert Yhat_full_perm.shape == Y_permuted.shape

    # ---------------------------------------------------------------
    # Full-model residual SS
    # ---------------------------------------------------------------

    residual_full_perm = (
        Y_permuted
        - Yhat_full_perm
    )

    SS_full_perm = np.sum(
        residual_full_perm ** 2
    )

    # ---------------------------------------------------------------
    # Additional species SS
    # ---------------------------------------------------------------

    SS_species_perm = (
        SS_reduced_perm
        - SS_full_perm
    )

    # Numerical precision guard
    if (
        SS_species_perm < 0
        and np.isclose(
            SS_species_perm,
            0.0,
            atol=1e-10
        )
    ):
        SS_species_perm = 0.0

    # ---------------------------------------------------------------
    # Permuted pseudo-F
    # ---------------------------------------------------------------

    permutation_F[i] = (
        (SS_species_perm / df_species)
        /
        (SS_full_perm / df_residual)
    )

# -------------------------------------------------------------------
# 11. Validate permutation output
# -------------------------------------------------------------------

assert permutation_F.shape == (
    N_PERMUTATIONS,
)

assert np.isfinite(
    permutation_F
).all()

assert np.all(
    permutation_F >= 0
)

# -------------------------------------------------------------------
# 12. Empirical permutation p-value
# -------------------------------------------------------------------

n_exceedances = int(
    np.sum(
        permutation_F >= pseudo_F_adjusted
    )
)

p_value = (
    1 + n_exceedances
) / (
    N_PERMUTATIONS + 1
)

# -------------------------------------------------------------------
# 13. Null distribution summary
# -------------------------------------------------------------------

null_summary = pd.DataFrame({
    "statistic": [
        "Observed pseudo-F",
        "Null mean",
        "Null median",
        "Null 95th percentile",
        "Null 99th percentile",
        "Permutation p-value"
    ],
    "value": [
        pseudo_F_adjusted,
        np.mean(permutation_F),
        np.median(permutation_F),
        np.quantile(
            permutation_F,
            0.95
        ),
        np.quantile(
            permutation_F,
            0.99
        ),
        p_value
    ]
})

print("\n" + "=" * 80)
print("FREEDMAN–LANE PERMUTATION TEST")
print("=" * 80)

print(
    "Permutations:",
    N_PERMUTATIONS
)

print(
    "Seed:",
    PERMUTATION_SEED
)

display(null_summary)

print(
    "Permuted statistics ≥ observed:",
    n_exceedances
)

# -------------------------------------------------------------------
# 14. Save results
# -------------------------------------------------------------------

species_association_result = {
    "n_recordings": int(n),
    "n_species": int(len(species_categories)),
    "n_states": int(len(state_categories)),
    "n_retained_pcs": int(N_RETAINED_PCS),
    "species_df": int(df_species),
    "residual_df": int(df_residual),
    "ss_species_adjusted": float(
        SS_species_adjusted
    ),
    "ss_residual_full": float(
        SS_resid_full
    ),
    "pseudo_F": float(
        pseudo_F_adjusted
    ),
    "partial_R2": float(
        partial_R2_species
    ),
    "n_permutations": int(
        N_PERMUTATIONS
    ),
    "permutation_seed": int(
        PERMUTATION_SEED
    ),
    "p_value": float(
        p_value
    ),
    "n_exceedances": int(
        n_exceedances
    )
}

species_association_result_path = (
    PCA_ANALYSIS_DIR
    / "species_association_state_adjusted.json"
)

species_association_result_path.write_text(
    json.dumps(
        species_association_result,
        indent=2
    )
)

np.save(
    PCA_ANALYSIS_DIR
    / "species_association_permutation_F.npy",
    permutation_F
)

print("\nSaved:")
print(
    species_association_result_path
)

print(
    PCA_ANALYSIS_DIR
    / "species_association_permutation_F.npy"
)

# -------------------------------------------------------------------
# 15. Interpretation guardrail
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("INTERPRETATION GUARDRAIL")
print("=" * 80)

print(
    "This test asks whether species explains additional multivariate "
    "PCA structure after accounting for state."
)

print(
    "A significant result would support an association between acoustic "
    "feature structure and species identity, not a direct measure of "
    "species richness or biodiversity."
)

print(
    "A non-significant result would also be scientifically informative: "
    "it would indicate that the apparent species structure is not stronger "
    "than expected under the specified state-adjusted null model."
)

MODEL DESIGN
Observations: 172
PCA dimensions: 9
Species groups: 56
State groups: 3

State-only design shape: (172, 3)
State-only rank: 3

Full design shape: (172, 58)
Full model rank: 58

Species degrees of freedom: 55
Residual degrees of freedom: 114

OBSERVED STATE-ADJUSTED SPECIES ASSOCIATION
Species SS after accounting for state: 2787.625769
Residual SS after state + species: 3060.476784
Species degrees of freedom: 55
Residual degrees of freedom: 114
State-adjusted pseudo-F: 1.887937
Partial R² for species: 0.476672
Partial R² (%): 47.667%

Observed model dimension checks: PASSED

FREEDMAN–LANE PERMUTATION TEST
Permutations: 4999
Seed: 42


,statistic,value
0,Observed pseudo-F,1.887937
1,Null mean,1.041180
2,Null median,1.034201
3,Null 95th percentile,1.218595
4,Null 99th percentile,1.315092
5,Permutation p-value,0.000200


Permuted statistics ≥ observed: 0

Saved:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\species_association_state_adjusted.json
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\species_association_permutation_F.npy

INTERPRETATION GUARDRAIL
This test asks whether species explains additional multivariate PCA structure after accounting for state.
A significant result would support an association between acoustic feature structure and species identity, not a direct measure of species richness or biodiversity.
A non-significant result would also be scientifically informative: it would indicate that the apparent species structure is not stronger than expected under the specified state-adjusted null model.


In [27]:
# 06F.7 — Freedman–Lane Null Diagnostics
#
# Purpose:
# Validate the state-adjusted permutation framework used in 06F.6.
#
# We specifically examine:
#   1. Residual dispersion across states
#   2. Residual centroids across states
#   3. Distribution and range of the permutation pseudo-F statistics
#   4. Whether the observed statistic lies beyond the simulated null
#
# This cell does NOT rerun the permutation test.

# -------------------------------------------------------------------
# 1. Validate required objects from 06F.6
# -------------------------------------------------------------------

required_objects = [
    "Y",
    "Yhat_state",
    "residual_state",
    "state_labels",
    "permutation_F",
    "pseudo_F_adjusted",
    "p_value",
    "N_PERMUTATIONS"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "06F.6 objects are missing: "
        + ", ".join(missing_objects)
        + ". Rerun 06F.6 before running 06F.7."
    )

assert residual_state.shape == Y.shape
assert np.isfinite(residual_state).all()
assert np.isfinite(permutation_F).all()

# -------------------------------------------------------------------
# 2. Residual magnitude for each recording
# -------------------------------------------------------------------
#
# Euclidean norm across the nine retained PCA dimensions.

residual_norm = np.linalg.norm(
    residual_state,
    axis=1
)

residual_diagnostics = pd.DataFrame({
    "state": state_labels,
    "residual_norm": residual_norm
})

state_residual_summary = (
    residual_diagnostics
    .groupby("state")["residual_norm"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max"
    )
    .reset_index()
)

print("=" * 80)
print("STATE-WISE REDUCED-MODEL RESIDUAL DISPERSION")
print("=" * 80)

display(state_residual_summary)

# -------------------------------------------------------------------
# 3. Compare residual scale across states
# -------------------------------------------------------------------

state_mean_residual = (
    state_residual_summary["mean"]
)

residual_scale_ratio = (
    state_mean_residual.max()
    /
    state_mean_residual.min()
)

print(
    "Maximum / minimum state mean residual norm:",
    f"{residual_scale_ratio:.4f}"
)

print(
    "\nThis ratio is descriptive only. It is used to identify whether "
    "one state has markedly different residual dispersion."
)

# -------------------------------------------------------------------
# 4. Residual centroids by state
# -------------------------------------------------------------------
#
# Under the state-only model, residuals should be centered approximately
# around zero within each state.

residual_columns = [
    f"residual_PC{i}"
    for i in range(1, residual_state.shape[1] + 1)
]

residual_df = pd.DataFrame(
    residual_state,
    columns=residual_columns
)

residual_df["state"] = state_labels

residual_state_centroids = (
    residual_df
    .groupby("state")[residual_columns]
    .mean()
)

print("\n" + "=" * 80)
print("STATE-WISE RESIDUAL CENTROIDS")
print("=" * 80)

display(residual_state_centroids)

max_abs_residual_centroid = (
    np.abs(
        residual_state_centroids.to_numpy()
    ).max()
)

print(
    "\nMaximum absolute residual centroid:",
    f"{max_abs_residual_centroid:.10f}"
)

# -------------------------------------------------------------------
# 5. Overall residual centroid
# -------------------------------------------------------------------

overall_residual_centroid = (
    residual_state.mean(axis=0)
)

print(
    "Maximum absolute overall residual centroid:",
    f"{np.abs(overall_residual_centroid).max():.10f}"
)

# -------------------------------------------------------------------
# 6. Permutation null summary
# -------------------------------------------------------------------

null_quantiles = np.quantile(
    permutation_F,
    [
        0.00,
        0.01,
        0.05,
        0.50,
        0.95,
        0.99,
        0.999,
        1.00
    ]
)

null_quantile_table = pd.DataFrame({
    "quantile": [
        "0%",
        "1%",
        "5%",
        "50%",
        "95%",
        "99%",
        "99.9%",
        "100%"
    ],
    "pseudo_F": null_quantiles
})

print("\n" + "=" * 80)
print("PERMUTATION NULL DISTRIBUTION")
print("=" * 80)

display(null_quantile_table)

# -------------------------------------------------------------------
# 7. Null mean / variance / coefficient of variation
# -------------------------------------------------------------------

null_mean = np.mean(permutation_F)
null_std = np.std(
    permutation_F,
    ddof=1
)

null_cv = (
    null_std / null_mean
)

print(
    "Null mean:",
    f"{null_mean:.6f}"
)

print(
    "Null standard deviation:",
    f"{null_std:.6f}"
)

print(
    "Null coefficient of variation:",
    f"{null_cv:.6f}"
)

# -------------------------------------------------------------------
# 8. Observed statistic relative to null
# -------------------------------------------------------------------

null_95 = np.quantile(
    permutation_F,
    0.95
)

null_99 = np.quantile(
    permutation_F,
    0.99
)

observed_to_null95 = (
    pseudo_F_adjusted / null_95
)

observed_to_null_mean = (
    pseudo_F_adjusted / null_mean
)

print("\n" + "=" * 80)
print("OBSERVED VS. NULL")
print("=" * 80)

print(
    "Observed pseudo-F:",
    f"{pseudo_F_adjusted:.6f}"
)

print(
    "Null 95th percentile:",
    f"{null_95:.6f}"
)

print(
    "Null 99th percentile:",
    f"{null_99:.6f}"
)

print(
    "Observed / null 95th percentile:",
    f"{observed_to_null95:.4f}"
)

print(
    "Observed / null mean:",
    f"{observed_to_null_mean:.4f}"
)

# -------------------------------------------------------------------
# 9. Check whether observed exceeds the simulated null
# -------------------------------------------------------------------

n_exceedances = int(
    np.sum(
        permutation_F >= pseudo_F_adjusted
    )
)

print(
    "\nPermuted statistics ≥ observed:",
    n_exceedances,
    "/",
    N_PERMUTATIONS
)

# -------------------------------------------------------------------
# 10. Create a compact diagnostic table
# -------------------------------------------------------------------

diagnostic_summary = pd.DataFrame({
    "diagnostic": [
        "Observed pseudo-F",
        "Null mean",
        "Null SD",
        "Null 95th percentile",
        "Null 99th percentile",
        "Null maximum",
        "Exceedances",
        "Permutation p-value",
        "Max/min state residual scale"
    ],
    "value": [
        pseudo_F_adjusted,
        null_mean,
        null_std,
        null_95,
        null_99,
        permutation_F.max(),
        n_exceedances,
        p_value,
        residual_scale_ratio
    ]
})

print("\n" + "=" * 80)
print("06F.7 DIAGNOSTIC SUMMARY")
print("=" * 80)

display(diagnostic_summary)

# -------------------------------------------------------------------
# 11. Interpretation guardrail
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("INTERPRETATION GUARDRAIL")
print("=" * 80)

print(
    "Residual centroids should be approximately zero within each state "
    "because state is explicitly included in the reduced model."
)

print(
    "The permutation null should be inspected as a distribution rather "
    "than interpreted solely through the p-value."
)

print(
    "The state residual-dispersion comparison is descriptive and does not "
    "by itself establish or reject exchangeability."
)

print(
    "If the null diagnostics are satisfactory, the state-adjusted species "
    "association can be reported as a permutation-based multivariate result."
)

STATE-WISE REDUCED-MODEL RESIDUAL DISPERSION


,state,n,mean,median,std,min,max
0,AZ,72,4.949452,4.336854,2.118697,1.685773,11.759519
1,CA,79,5.491275,5.068782,2.301871,2.323763,18.169337
2,TX,21,5.958735,5.602413,3.350373,2.420843,17.992011


Maximum / minimum state mean residual norm: 1.2039

This ratio is descriptive only. It is used to identify whether one state has markedly different residual dispersion.

STATE-WISE RESIDUAL CENTROIDS


,residual_PC1,residual_PC2,residual_PC3,residual_PC4,residual_PC5,residual_PC6,residual_PC7,residual_PC8,residual_PC9
state,,,,,,,,,
AZ,1.480297e-16,-4.857226e-16,1.850372e-16,9.116936e-17,1.572816e-16,1.657625e-17,-4.317534e-17,2.312965e-17,-2.158767e-17
CA,4.497106e-17,3.372829e-16,1.925323e-16,1.475613e-16,-2.793124e-16,1.025902e-16,-1.264811e-16,7.026728e-18,7.026728e-18
TX,-3.489272e-16,3.383537e-16,-2.008975e-16,1.057355e-17,3.383537e-16,2.114711e-17,4.229421e-17,-4.890268e-17,4.229421e-17



Maximum absolute residual centroid: 0.0000000000
Maximum absolute overall residual centroid: 0.0000000000

PERMUTATION NULL DISTRIBUTION


,quantile,pseudo_F
0,0%,0.740378
1,1%,0.823942
2,5%,0.882166
3,50%,1.034201
4,95%,1.218595
5,99%,1.315092
6,99.9%,1.438345
7,100%,1.498688


Null mean: 1.041180
Null standard deviation: 0.103660
Null coefficient of variation: 0.099560

OBSERVED VS. NULL
Observed pseudo-F: 1.887937
Null 95th percentile: 1.218595
Null 99th percentile: 1.315092
Observed / null 95th percentile: 1.5493
Observed / null mean: 1.8133

Permuted statistics ≥ observed: 0 / 4999

06F.7 DIAGNOSTIC SUMMARY


,diagnostic,value
0,Observed pseudo-F,1.887937
1,Null mean,1.041180
2,Null SD,0.103660
3,Null 95th percentile,1.218595
4,Null 99th percentile,1.315092
5,Null maximum,1.498688
6,Exceedances,0.000000
7,Permutation p-value,0.000200
8,Max/min state residual scale,1.203918



INTERPRETATION GUARDRAIL
Residual centroids should be approximately zero within each state because state is explicitly included in the reduced model.
The permutation null should be inspected as a distribution rather than interpreted solely through the p-value.
The state residual-dispersion comparison is descriptive and does not by itself establish or reject exchangeability.
If the null diagnostics are satisfactory, the state-adjusted species association can be reported as a permutation-based multivariate result.


In [28]:
# ============================================================
# 06F.8 — Representation Sensitivity
# Setup and Validation
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PCA_ANALYSIS_DIR = DATA_PROCESSED_DIR / "pca_analysis"
PCA_REPRESENTATION_DIR = DATA_PROCESSED_DIR / "pca_representations"

# ------------------------------------------------------------------
# Analysis configuration
# ------------------------------------------------------------------

REPRESENTATIONS = [
    "baseline",
    "distribution_aware",
    "yeo_johnson",
    "block_balanced",
    "ndsi_derived_excluded",
    "ndsi_family_excluded",
]

RANDOM_SEED = 42
N_PERMUTATIONS = 4999

N_RECORDINGS_TARGET = 172
N_SPECIES_TARGET = 56

# ------------------------------------------------------------------
# Load recording-level metadata
# ------------------------------------------------------------------

metadata = pd.read_csv(
    DATA_PROCESSED_DIR / "master_metadata.csv"
)

metadata["recording_id"] = metadata["recording_id"].astype(str)

# Reconstruct the reproducible >=2-recording species cohort
species_counts = metadata["species_id"].value_counts()

eligible_species = species_counts[
    species_counts >= 2
].index

analysis_metadata = metadata[
    metadata["species_id"].isin(eligible_species)
].copy()

analysis_metadata = analysis_metadata.reset_index(drop=True)

# ------------------------------------------------------------------
# Load PCA artifacts
# ------------------------------------------------------------------

pca_results = {}

for name in REPRESENTATIONS:

    scores_path = PCA_ANALYSIS_DIR / f"{name}_scores.npy"
    feature_names_path = PCA_ANALYSIS_DIR / f"{name}_feature_names.json"
    retained_path = PCA_ANALYSIS_DIR / f"{name}_pa_retained.npy"

    scores = np.load(scores_path)

    with open(feature_names_path, "r") as f:
        feature_names = json.load(f)

    retained_mask = np.load(retained_path)

    retained_indices = np.where(retained_mask)[0]

    pca_results[name] = {
        "scores": scores,
        "feature_names": feature_names,
        "retained_mask": retained_mask,
        "retained_indices": retained_indices,
    }

# ------------------------------------------------------------------
# Validation
# ------------------------------------------------------------------

assert len(analysis_metadata) == N_RECORDINGS_TARGET
assert analysis_metadata["species_id"].nunique() == N_SPECIES_TARGET

print("=" * 80)
print("06F.8 REPRESENTATION SENSITIVITY — VALIDATION")
print("=" * 80)

print(
    f"Analysis cohort: {len(analysis_metadata)} recordings, "
    f"{analysis_metadata['species_id'].nunique()} species"
)

for name in REPRESENTATIONS:

    scores = pca_results[name]["scores"]
    retained = pca_results[name]["retained_indices"]

    assert scores.shape[0] == len(metadata)
    assert len(retained) > 0
    assert np.isfinite(scores).all()

    print(
        f"{name:28s} | "
        f"full scores: {scores.shape} | "
        f"retained PCs: {len(retained)}"
    )

print("=" * 80)
print("Validation passed.")

06F.8 REPRESENTATION SENSITIVITY — VALIDATION
Analysis cohort: 172 recordings, 56 species
baseline                     | full scores: (275, 50) | retained PCs: 9
distribution_aware           | full scores: (275, 50) | retained PCs: 9
yeo_johnson                  | full scores: (275, 50) | retained PCs: 9
block_balanced               | full scores: (275, 50) | retained PCs: 7
ndsi_derived_excluded        | full scores: (275, 48) | retained PCs: 9
ndsi_family_excluded         | full scores: (275, 46) | retained PCs: 9
Validation passed.


In [29]:
# ============================================================
# 06F.8 — Representation Sensitivity
# State-adjusted species association
# ============================================================

# ------------------------------------------------------------------
# Align the 172-recording cohort to PCA score rows
# ------------------------------------------------------------------

metadata_full = metadata.copy()

# PCA artifacts use the same 275-recording ordering as master_metadata.
# Build recording_id -> row-position mapping.
recording_to_position = {
    rid: i
    for i, rid in enumerate(metadata_full["recording_id"])
}

analysis_positions = np.array([
    recording_to_position[rid]
    for rid in analysis_metadata["recording_id"]
])

# ------------------------------------------------------------------
# Build categorical design matrices
# ------------------------------------------------------------------

species_codes = pd.Categorical(
    analysis_metadata["species_id"]
).codes

state_codes = pd.Categorical(
    analysis_metadata["state"]
).codes

n = len(analysis_metadata)

# Intercept + state
X_reduced = np.column_stack([
    np.ones(n),
    pd.get_dummies(
        analysis_metadata["state"],
        drop_first=True
    ).to_numpy(dtype=float)
])

# Intercept + state + species
species_dummies = pd.get_dummies(
    analysis_metadata["species_id"],
    drop_first=True
).to_numpy(dtype=float)

X_full = np.column_stack([
    X_reduced,
    species_dummies
])

# ------------------------------------------------------------------
# Projection matrices
# ------------------------------------------------------------------

H_reduced = (
    X_reduced
    @ np.linalg.pinv(X_reduced)
)

H_full = (
    X_full
    @ np.linalg.pinv(X_full)
)

# ------------------------------------------------------------------
# Permutation function
# ------------------------------------------------------------------

def state_adjusted_species_test(
    Y,
    n_permutations=4999,
    seed=42
):
    """
    Freedman–Lane permutation test for the additional contribution
    of species after accounting for state.

    Y:
        n_recordings × n_PCs retained PCA scores
    """

    rng = np.random.default_rng(seed)

    # Reduced-model fitted values and residuals
    Y_reduced_hat = H_reduced @ Y
    E_reduced = Y - Y_reduced_hat

    # Observed full-model and reduced-model residual SS
    E_full = Y - (H_full @ Y)

    ss_reduced_residual = np.sum(E_reduced ** 2)
    ss_full_residual = np.sum(E_full ** 2)

    # Additional SS attributable to species after state
    ss_species = (
        ss_reduced_residual
        - ss_full_residual
    )

    # Degrees of freedom
    df_species = np.linalg.matrix_rank(X_full) - np.linalg.matrix_rank(X_reduced)
    df_residual = n - np.linalg.matrix_rank(X_full)

    pseudo_F = (
        (ss_species / df_species)
        / (ss_full_residual / df_residual)
    )

    # --------------------------------------------------------------
    # Freedman–Lane permutation
    # --------------------------------------------------------------

    permutation_F = np.empty(n_permutations)

    for i in range(n_permutations):

        permuted_residuals = E_reduced[
            rng.permutation(n)
        ]

        Y_permuted = (
            Y_reduced_hat
            + permuted_residuals
        )

        E_full_permuted = (
            Y_permuted
            - H_full @ Y_permuted
        )

        ss_full_permuted = np.sum(
            E_full_permuted ** 2
        )

        ss_species_permuted = (
            ss_reduced_residual
            - ss_full_permuted
        )

        permutation_F[i] = (
            (ss_species_permuted / df_species)
            / (ss_full_permuted / df_residual)
        )

    exceedances = np.sum(
        permutation_F >= pseudo_F
    )

    p_value = (
        exceedances + 1
    ) / (
        n_permutations + 1
    )

    return {
        "n_recordings": n,
        "n_species": analysis_metadata["species_id"].nunique(),
        "n_states": analysis_metadata["state"].nunique(),
        "n_pcs": Y.shape[1],
        "df_species": df_species,
        "df_residual": df_residual,
        "species_ss": ss_species,
        "residual_ss": ss_full_residual,
        "pseudo_F": pseudo_F,
        "null_mean": np.mean(permutation_F),
        "null_sd": np.std(permutation_F, ddof=1),
        "null_p95": np.quantile(permutation_F, 0.95),
        "null_p99": np.quantile(permutation_F, 0.99),
        "null_max": np.max(permutation_F),
        "exceedances": int(exceedances),
        "p_value": p_value,
        "permutation_F": permutation_F,
    }


# ------------------------------------------------------------------
# Run across all representations
# ------------------------------------------------------------------

results = {}
permutation_distributions = {}

print("=" * 80)
print("06F.8 — STATE-ADJUSTED SPECIES ASSOCIATION")
print("=" * 80)

for name in REPRESENTATIONS:

    scores = pca_results[name]["scores"]

    retained_indices = pca_results[name]["retained_indices"]

    # Restrict to retained PCA dimensions
    Y = scores[
        analysis_positions,
        :
    ][:, retained_indices]

    result = state_adjusted_species_test(
        Y,
        n_permutations=N_PERMUTATIONS,
        seed=RANDOM_SEED
    )

    results[name] = result
    permutation_distributions[name] = result.pop(
        "permutation_F"
    )

    print(
        f"{name:28s} | "
        f"PCs={result['n_pcs']:2d} | "
        f"F={result['pseudo_F']:.6f} | "
        f"p={result['p_value']:.6f} | "
        f"null95={result['null_p95']:.6f} | "
        f"nullMax={result['null_max']:.6f}"
    )

print("=" * 80)

06F.8 — STATE-ADJUSTED SPECIES ASSOCIATION
baseline                     | PCs= 9 | F=1.238181 | p=0.027600 | null95=1.206826 | nullMax=1.412377
distribution_aware           | PCs= 9 | F=1.236056 | p=0.028800 | null95=1.209677 | nullMax=1.409002
yeo_johnson                  | PCs= 9 | F=1.231444 | p=0.032800 | null95=1.208259 | nullMax=1.432092
block_balanced               | PCs= 7 | F=1.251718 | p=0.026200 | null95=1.214196 | nullMax=1.477242
ndsi_derived_excluded        | PCs= 9 | F=1.227972 | p=0.033000 | null95=1.206886 | nullMax=1.436872
ndsi_family_excluded         | PCs= 9 | F=1.223643 | p=0.038200 | null95=1.209327 | nullMax=1.438012


In [30]:
# ============================================================
# 06F.8.2 — Baseline calculation audit
# ============================================================

print("=" * 80)
print("06F.8.2 — BASELINE CALCULATION AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# Baseline retained scores
# ------------------------------------------------------------

baseline_scores = pca_results["baseline"]["scores"]

baseline_retained = pca_results["baseline"]["retained_indices"]

Y_baseline = baseline_scores[
    analysis_positions,
    :
][:, baseline_retained]

print("Y_baseline shape:", Y_baseline.shape)
print("Expected shape:  (172, 9)")

# ------------------------------------------------------------
# Design matrix diagnostics
# ------------------------------------------------------------

print("\nReduced design")
print("Shape:", X_reduced.shape)
print("Rank:", np.linalg.matrix_rank(X_reduced))

print("\nFull design")
print("Shape:", X_full.shape)
print("Rank:", np.linalg.matrix_rank(X_full))

print("\nDegrees of freedom")
print(
    "Species:",
    np.linalg.matrix_rank(X_full)
    - np.linalg.matrix_rank(X_reduced)
)

print(
    "Residual:",
    n
    - np.linalg.matrix_rank(X_full)
)

# ------------------------------------------------------------
# Projection-matrix checks
# ------------------------------------------------------------

print("\nProjection matrix checks")

print(
    "H_reduced symmetric:",
    np.allclose(H_reduced, H_reduced.T)
)

print(
    "H_reduced idempotent:",
    np.allclose(
        H_reduced @ H_reduced,
        H_reduced
    )
)

print(
    "H_full symmetric:",
    np.allclose(H_full, H_full.T)
)

print(
    "H_full idempotent:",
    np.allclose(
        H_full @ H_full,
        H_full
    )
)

# ------------------------------------------------------------
# Observed sums of squares
# ------------------------------------------------------------

Y_reduced_hat = H_reduced @ Y_baseline
Y_full_hat = H_full @ Y_baseline

E_reduced = Y_baseline - Y_reduced_hat
E_full = Y_baseline - Y_full_hat

ss_reduced = np.sum(E_reduced ** 2)
ss_full = np.sum(E_full ** 2)

ss_species = ss_reduced - ss_full

df_species = (
    np.linalg.matrix_rank(X_full)
    - np.linalg.matrix_rank(X_reduced)
)

df_residual = (
    n
    - np.linalg.matrix_rank(X_full)
)

F_audit = (
    (ss_species / df_species)
    /
    (ss_full / df_residual)
)

print("\nObserved sums of squares")
print("Reduced residual SS:", ss_reduced)
print("Full residual SS:   ", ss_full)
print("Species SS:         ", ss_species)

print("\nObserved pseudo-F")
print("F =", F_audit)

# ------------------------------------------------------------
# Compare with 06F.8 result
# ------------------------------------------------------------

print("\nStored 06F.8 result")
print(
    "F =",
    results["baseline"]["pseudo_F"]
)

print(
    "Difference =",
    F_audit - results["baseline"]["pseudo_F"]
)

print("=" * 80)

06F.8.2 — BASELINE CALCULATION AUDIT
Y_baseline shape: (172, 9)
Expected shape:  (172, 9)

Reduced design
Shape: (172, 3)
Rank: 3

Full design
Shape: (172, 58)
Rank: 58

Degrees of freedom
Species: 55
Residual: 114

Projection matrix checks
H_reduced symmetric: True
H_reduced idempotent: True
H_full symmetric: True
H_full idempotent: True

Observed sums of squares
Reduced residual SS: 5141.0889111886445
Full residual SS:    3218.4753916810087
Species SS:          1922.6135195076358

Observed pseudo-F
F = 1.2381805022023962

Stored 06F.8 result
F = 1.2381805022023962
Difference = 0.0


# RESTARTING ANALYSIS

06F — PCA Species Association
Objective
This notebook tests whether the multivariate acoustic structure identified
without ecological labels is systematically associated with species identity.
The scientific question is:
Do the dominant acoustic dimensions identified from unsupervised acoustic
features show systematic association with species identity?

Species identity is introduced only after PCA construction. It is therefore
used for post-hoc ecological interpretation rather than as an input to the
unsupervised representation.
Primary analysis unit
The recording is the primary unit of analysis.
The focal species-association subset contains:
- 56 species
- 172 recordings
This restriction is applied before statistical testing to ensure that species
with sufficient recording representation contribute to the association analysis.
Statistical principle
Observed species–PCA relationships will be compared against a permutation null
distribution in which species labels are shuffled across recordings while the
acoustic PCA structure is held fixed.
This tests whether the observed association is stronger than expected from
the marginal species frequencies alone.
Important distinction
A significant species association does not mean that PCA provides a direct
measure of biodiversity or species richness.
It indicates that acoustic variation contains reproducible structure associated
with species identity.

In [31]:
# ============================================================
# 06F.1 — Setup and locate ecological-analysis inputs
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

DATA_PROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

PCA_ANALYSIS_DIR = (
    DATA_PROCESSED_DIR / "pca_analysis"
)

PCA_REPRESENTATION_DIR = (
    DATA_PROCESSED_DIR / "pca_representations"
)

BOOTSTRAP_DIR = (
    PCA_ANALYSIS_DIR / "bootstrap_stability"
)


# ------------------------------------------------------------
# Analysis constants
# ------------------------------------------------------------

N_RECORDINGS_FULL = 275
N_SPECIES_TARGET = 56
N_RECORDINGS_TARGET = 172

RANDOM_SEED = 42


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("06F — PCA SPECIES ASSOCIATION")
print("=" * 70)

print(f"Project root:")
print(PROJECT_ROOT)

print(f"\nProcessed-data directory:")
print(DATA_PROCESSED_DIR)

print(f"\nPCA analysis directory:")
print(PCA_ANALYSIS_DIR)

print(f"\nPCA representation directory:")
print(PCA_REPRESENTATION_DIR)

print(f"\nBootstrap directory:")
print(BOOTSTRAP_DIR)

print("\n" + "=" * 70)
print("TARGET ECOLOGICAL SUBSET")
print("=" * 70)

print(f"Target species:    {N_SPECIES_TARGET}")
print(f"Target recordings: {N_RECORDINGS_TARGET}")
print(f"Random seed:       {RANDOM_SEED}")

06F — PCA SPECIES ASSOCIATION
Project root:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity

Processed-data directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed

PCA analysis directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis

PCA representation directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_representations

Bootstrap directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\bootstrap_stability

TARGET ECOLOGICAL SUBSET
Target species:    56
Target recordings: 172
Random seed:       42


In [32]:
# ============================================================
# 06F.1 — Cell 2: Inventory relevant analysis artifacts
# ============================================================

print("=" * 70)
print("PCA ANALYSIS ARTIFACTS")
print("=" * 70)

for path in sorted(PCA_ANALYSIS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))


print("\n" + "=" * 70)
print("PCA REPRESENTATION ARTIFACTS")
print("=" * 70)

for path in sorted(PCA_REPRESENTATION_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))


print("\n" + "=" * 70)
print("LIKELY METADATA / FEATURE TABLES")
print("=" * 70)

# Search processed data for CSV/Parquet files whose names
# suggest recording-level metadata or feature tables.
candidate_files = []

for path in sorted(DATA_PROCESSED_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in {".csv", ".parquet"}:
        name = path.name.lower()

        keywords = [
            "feature",
            "metadata",
            "recording",
            "master",
            "acoustic",
            "species",
            "pca",
        ]

        if any(keyword in name for keyword in keywords):
            candidate_files.append(path)

for path in candidate_files:
    print(path.relative_to(PROJECT_ROOT))

print(f"\nCandidate metadata/feature files found: {len(candidate_files)}")

PCA ANALYSIS ARTIFACTS
data\processed\pca_analysis\baseline_components.npy
data\processed\pca_analysis\baseline_explained_variance.npy
data\processed\pca_analysis\baseline_explained_variance_ratio.npy
data\processed\pca_analysis\baseline_feature_names.json
data\processed\pca_analysis\baseline_pa_null_95pct.npy
data\processed\pca_analysis\baseline_pa_null_mean.npy
data\processed\pca_analysis\baseline_pa_observed_eigenvalues.npy
data\processed\pca_analysis\baseline_pa_retained.npy
data\processed\pca_analysis\baseline_scores.npy
data\processed\pca_analysis\baseline_singular_values.npy
data\processed\pca_analysis\block_balanced_components.npy
data\processed\pca_analysis\block_balanced_explained_variance.npy
data\processed\pca_analysis\block_balanced_explained_variance_ratio.npy
data\processed\pca_analysis\block_balanced_feature_names.json
data\processed\pca_analysis\block_balanced_pa_null_95pct.npy
data\processed\pca_analysis\block_balanced_pa_null_mean.npy
data\processed\pca_analysis\bloc

In [33]:
# ============================================================
# 06F.1 — Cell 3: Inspect candidate recording-level tables
# ============================================================

print("=" * 70)
print("CANDIDATE RECORDING / FEATURE TABLES")
print("=" * 70)

for path in candidate_files:
    print(f"\n{'-' * 70}")
    print(f"FILE: {path.relative_to(PROJECT_ROOT)}")

    try:
        if path.suffix.lower() == ".csv":
            df_tmp = pd.read_csv(path, nrows=5)
        elif path.suffix.lower() == ".parquet":
            df_tmp = pd.read_parquet(path).head(5)
        else:
            continue

        print(f"Shape preview: {df_tmp.shape}")
        print("Columns:")
        print(list(df_tmp.columns))

    except Exception as e:
        print(f"Could not inspect file: {e}")

CANDIDATE RECORDING / FEATURE TABLES

----------------------------------------------------------------------
FILE: data\processed\acoustic_features_recording.csv
Shape preview: (5, 114)
Columns:
['recording_id', 'state', 'species_id', 'common_name', 'scientific_name', 'original_sample_rate', 'original_channels', 'flag_short_duration', 'flag_low_amplitude', 'flag_clipping', 'flag_low_quality', 'flag_any', 'flag_bandlimit_incompatible', 'n_windows', 'aci_mean', 'adi_mean', 'bioacoustic_index_mean', 'ndsi_mean', 'ndsi_ratio_ba_mean', 'anthropophony_mean', 'biophony_mean', 'spectral_entropy_eas_mean', 'spectral_entropy_ecu_mean', 'spectral_entropy_ecv_mean', 'spectral_entropy_eps_mean', 'spectral_entropy_eps_kurt_mean', 'spectral_entropy_eps_skew_mean', 'temporal_entropy_mean', 'rms_mean_mean', 'rms_std_mean', 'zcr_mean_mean', 'zcr_std_mean', 'spectral_centroid_mean_mean', 'spectral_centroid_std_mean', 'spectral_bandwidth_mean_mean', 'spectral_bandwidth_std_mean', 'spectral_rolloff_mean_me

In [34]:
# ============================================================
# 06F.1 — Cell 4: Audit species representation
# ============================================================

FEATURES_PATH = DATA_PROCESSED_DIR / "acoustic_features_recording.csv"

recording_features = pd.read_csv(FEATURES_PATH)

print("=" * 70)
print("RECORDING-LEVEL FEATURE TABLE")
print("=" * 70)

print(f"Shape: {recording_features.shape}")
print(f"Unique recordings: {recording_features['recording_id'].nunique()}")
print(f"Unique species: {recording_features['species_id'].nunique()}")

print("\n" + "=" * 70)
print("SPECIES RECORDING COUNTS")
print("=" * 70)

species_counts = (
    recording_features
    .groupby(
        ["species_id", "common_name", "scientific_name"],
        dropna=False
    )
    .size()
    .reset_index(name="n_recordings")
    .sort_values(
        ["n_recordings", "scientific_name"],
        ascending=[False, True]
    )
)

print(species_counts.to_string(index=False))

print("\n" + "=" * 70)
print("SPECIES-COUNT SUMMARY")
print("=" * 70)

print(species_counts["n_recordings"].describe())

print("\nSpecies by minimum recording threshold:")

for threshold in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    eligible = species_counts[
        species_counts["n_recordings"] >= threshold
    ]

    n_species = len(eligible)
    n_recordings = eligible["n_recordings"].sum()

    print(
        f"  >= {threshold:2d} recordings/species: "
        f"{n_species:3d} species, "
        f"{n_recordings:3d} recordings"
    )

RECORDING-LEVEL FEATURE TABLE
Shape: (275, 114)
Unique recordings: 275
Unique species: 159

SPECIES RECORDING COUNTS
                species_id                  common_name            scientific_name  n_recordings
         Anas carolinensis            Green-winged Teal          Anas carolinensis             5
     Archilochus alexandri    Black-chinned Hummingbird      Archilochus alexandri             5
          Chordeiles minor             Common Nighthawk           Chordeiles minor             5
   Geococcyx californianus           Greater Roadrunner    Geococcyx californianus             5
           Mareca strepera                      Gadwall            Mareca strepera             5
  Phalaenoptilus nuttallii              Common Poorwill   Phalaenoptilus nuttallii             5
   Selasphorus platycercus     Broad-tailed Hummingbird    Selasphorus platycercus             5
        Anas platyrhynchos                      Mallard         Anas platyrhynchos             4
          

In [35]:
# ============================================================
# 06F.1 — Cell 5: Audit QC and state structure
# ============================================================

print("=" * 70)
print("QC FLAG DISTRIBUTION")
print("=" * 70)

qc_cols = [
    "flag_short_duration",
    "flag_low_amplitude",
    "flag_clipping",
    "flag_low_quality",
    "flag_any",
    "flag_bandlimit_incompatible",
]

for col in qc_cols:
    if col in recording_features.columns:
        print(f"\n{col}:")
        print(recording_features[col].value_counts(dropna=False).to_string())


print("\n" + "=" * 70)
print("STATE × QC")
print("=" * 70)

print(
    recording_features
    .groupby("state")
    .agg(
        n_recordings=("recording_id", "nunique"),
        n_species=("species_id", "nunique"),
        n_flagged=("flag_any", "sum"),
        n_bandlimit_incompatible=("flag_bandlimit_incompatible", "sum"),
    )
    .to_string()
)


print("\n" + "=" * 70)
print("CLEAN / UNFLAGGED DATASET")
print("=" * 70)

clean = recording_features[
    ~recording_features["flag_any"].astype(bool)
].copy()

print(f"Clean recordings: {len(clean)}")
print(f"Clean species:    {clean['species_id'].nunique()}")

clean_species_counts = (
    clean.groupby("species_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nNumber of species by recording count:")
print(clean_species_counts.to_string())


print("\n" + "=" * 70)
print("BANDLIMIT-COMPATIBLE DATASET")
print("=" * 70)

bandlimit_compatible = recording_features[
    ~recording_features["flag_bandlimit_incompatible"].astype(bool)
].copy()

print(f"Bandlimit-compatible recordings: {len(bandlimit_compatible)}")
print(f"Bandlimit-compatible species:    {bandlimit_compatible['species_id'].nunique()}")

QC FLAG DISTRIBUTION

flag_short_duration:
flag_short_duration
False    267
True       8

flag_low_amplitude:
flag_low_amplitude
False    222
True      53

flag_clipping:
flag_clipping
False    267
True       8

flag_low_quality:
flag_low_quality
False    275

flag_any:
flag_any
False    208
True      67

flag_bandlimit_incompatible:
flag_bandlimit_incompatible
False    274
True       1

STATE × QC
       n_recordings  n_species  n_flagged  n_bandlimit_incompatible
state                                                              
AZ               75         44         30                         0
CA              100         66         15                         0
TX              100        100         22                         1

CLEAN / UNFLAGGED DATASET
Clean recordings: 208
Clean species:    134

Number of species by recording count:
1    92
2    20
3    13
4     8
5     1

BANDLIMIT-COMPATIBLE DATASET
Bandlimit-compatible recordings: 274
Bandlimit-compatible species:    158


In [36]:
# ============================================================
# 06F.1 — Cell 6: Trace the origin of the 56-species /
#             172-recording subset
# ============================================================

import re

SEARCH_TERMS = [
    "172",
    "56 species",
    "56_species",
    "species association",
    "species_association",
    "species association subset",
    "eligible species",
]

SEARCH_EXTENSIONS = {
    ".ipynb",
    ".py",
    ".md",
    ".txt",
    ".json",
    ".csv",
}

matches = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    # Skip large/raw/generated data directories where possible
    relative = path.relative_to(PROJECT_ROOT)

    if any(
        part.lower() in {
            ".git",
            "audio",
            "raw",
            "__pycache__",
        }
        for part in relative.parts
    ):
        continue

    if path.suffix.lower() not in SEARCH_EXTENSIONS:
        continue

    try:
        text_content = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    for term in SEARCH_TERMS:
        if term.lower() in text_content.lower():
            matches.append(
                (
                    str(relative),
                    term
                )
            )

print("=" * 70)
print("SUBSET-PROVENANCE SEARCH")
print("=" * 70)

if matches:
    # Remove duplicate file/term pairs while preserving order
    seen = set()

    for filename, term in matches:
        key = (filename, term)

        if key not in seen:
            seen.add(key)
            print(f"{filename}  <--  {term}")

    print(f"\nTotal matches: {len(seen)}")

else:
    print("No matches found.")

SUBSET-PROVENANCE SEARCH
data\interim\xenocanto_selected\xc_a1_selected_AZ.json  <--  172
data\interim\xenocanto_selected\xc_a1_selected_CA.json  <--  172
data\manifests\a1_xenocanto_selection_manifest.csv  <--  172
data\manifests\cleaning_log.csv  <--  172
data\processed\acoustic_features_recording.csv  <--  172
data\processed\acoustic_features_window.csv  <--  172
data\processed\ebird_observations.csv  <--  172
data\processed\master_metadata.csv  <--  172
data\processed\pca_analysis\parallel_analysis_detail.csv  <--  172
data\processed\pca_analysis\pca_first10.csv  <--  172
data\processed\pca_analysis\pca_variance_summary.csv  <--  172
data\processed\pca_analysis\recording_ids.json  <--  172
data\processed\pca_analysis\species_association_state_adjusted.json  <--  172
data\processed\pca_analysis\bootstrap_stability\bootstrap_explained_variance_summary.csv  <--  172
data\processed\pca_analysis\bootstrap_stability\principal_angle_summary.csv  <--  172
data\processed\pca_analysis\bootst

In [37]:
# ============================================================
# 06F.1 — Cell 7: Validate PCA recording-ID lineage
# ============================================================

PCA_IDS_PATH = PCA_ANALYSIS_DIR / "recording_ids.json"
REP_IDS_PATH = PCA_REPRESENTATION_DIR / "recording_ids.json"

print("=" * 70)
print("PCA RECORDING-ID FILES")
print("=" * 70)

print(f"PCA analysis IDs exist:       {PCA_IDS_PATH.exists()}")
print(f"PCA representation IDs exist: {REP_IDS_PATH.exists()}")


# ------------------------------------------------------------
# Load IDs
# ------------------------------------------------------------

with open(PCA_IDS_PATH, "r", encoding="utf-8") as f:
    pca_recording_ids = json.load(f)

with open(REP_IDS_PATH, "r", encoding="utf-8") as f:
    representation_recording_ids = json.load(f)


# ------------------------------------------------------------
# Normalize to simple lists
# ------------------------------------------------------------

def extract_id_list(obj):
    """
    Handle either:
      - a plain JSON list
      - a dictionary containing recording IDs
    """
    if isinstance(obj, list):
        return obj

    if isinstance(obj, dict):
        for key in [
            "recording_ids",
            "ids",
            "recording_id",
        ]:
            if key in obj:
                return obj[key]

    raise ValueError(
        f"Could not identify recording IDs. "
        f"JSON object type: {type(obj)}"
    )


pca_ids = extract_id_list(pca_recording_ids)
representation_ids = extract_id_list(
    representation_recording_ids
)

pca_ids = [str(x) for x in pca_ids]
representation_ids = [str(x) for x in representation_ids]


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ID COUNTS")
print("=" * 70)

print(f"PCA IDs:            {len(pca_ids)}")
print(f"Unique PCA IDs:     {len(set(pca_ids))}")

print(f"Representation IDs: {len(representation_ids)}")
print(f"Unique repr IDs:    {len(set(representation_ids))}")

print(f"Feature-table IDs:  "
      f"{recording_features['recording_id'].nunique()}")


# ------------------------------------------------------------
# Compare PCA and representation ordering
# ------------------------------------------------------------

same_length = len(pca_ids) == len(representation_ids)
same_order = pca_ids == representation_ids

print("\n" + "=" * 70)
print("PCA ↔ REPRESENTATION ALIGNMENT")
print("=" * 70)

print(f"Same length: {same_length}")
print(f"Same order:  {same_order}")


# ------------------------------------------------------------
# Compare against recording-level feature table
# ------------------------------------------------------------

feature_ids = recording_features["recording_id"].astype(str).tolist()
feature_id_set = set(feature_ids)

pca_id_set = set(pca_ids)

print("\n" + "=" * 70)
print("PCA ↔ FEATURE-TABLE COVERAGE")
print("=" * 70)

print(
    f"PCA IDs found in feature table: "
    f"{len(pca_id_set & feature_id_set)} / {len(pca_id_set)}"
)

print(
    f"Feature-table IDs represented in PCA: "
    f"{len(feature_id_set & pca_id_set)} / {len(feature_id_set)}"
)

missing_from_features = sorted(
    pca_id_set - feature_id_set
)

missing_from_pca = sorted(
    feature_id_set - pca_id_set
)

print(f"\nPCA IDs missing from feature table: {len(missing_from_features)}")
print(f"Feature IDs missing from PCA:        {len(missing_from_pca)}")


# ------------------------------------------------------------
# Show first few IDs for manual sanity check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 10 PCA IDS")
print("=" * 70)

print(pca_ids[:10])

PCA RECORDING-ID FILES
PCA analysis IDs exist:       True
PCA representation IDs exist: True

ID COUNTS
PCA IDs:            275
Unique PCA IDs:     275
Representation IDs: 275
Unique repr IDs:    275
Feature-table IDs:  275

PCA ↔ REPRESENTATION ALIGNMENT
Same length: True
Same order:  True

PCA ↔ FEATURE-TABLE COVERAGE
PCA IDs found in feature table: 275 / 275
Feature-table IDs represented in PCA: 275 / 275

PCA IDs missing from feature table: 0
Feature IDs missing from PCA:        0

FIRST 10 PCA IDS
['1035206', '1080401', '1080405', '1089596', '109164', '109171', '1101338', '1106496', '110689', '114789']


In [38]:
# ============================================================
# 06F.1 — Cell 8: Search existing PCA artifacts for
#             an explicit ecological subset
# ============================================================

print("=" * 70)
print("SEARCHING PCA ARTIFACTS FOR RECORDING SUBSETS")
print("=" * 70)

# ------------------------------------------------------------
# Inspect CSV artifacts that may contain recording-level rows
# ------------------------------------------------------------

pca_csvs = sorted(PCA_ANALYSIS_DIR.glob("*.csv"))

for path in pca_csvs:
    print(f"\n{'-' * 70}")
    print(path.name)

    try:
        df_tmp = pd.read_csv(path)

        print(f"Shape: {df_tmp.shape}")
        print("Columns:", list(df_tmp.columns))

        # If recording IDs are present, report their count.
        id_columns = [
            c for c in df_tmp.columns
            if "recording" in c.lower() and "id" in c.lower()
        ]

        for col in id_columns:
            ids = df_tmp[col].dropna().astype(str)

            print(
                f"  {col}: "
                f"{len(ids)} rows, "
                f"{ids.nunique()} unique IDs"
            )

    except Exception as e:
        print(f"Could not read: {e}")


# ------------------------------------------------------------
# Inspect JSON artifacts for recording-level ID collections
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("JSON ARTIFACTS")
print("=" * 70)

for path in sorted(PCA_ANALYSIS_DIR.glob("*.json")):
    print(f"\n{path.name}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, list):
            print(f"  JSON list length: {len(obj)}")

        elif isinstance(obj, dict):
            print(f"  JSON keys: {list(obj.keys())}")

            for key, value in obj.items():
                if isinstance(value, list):
                    print(
                        f"  {key}: list length = {len(value)}"
                    )

    except Exception as e:
        print(f"  Could not read: {e}")

SEARCHING PCA ARTIFACTS FOR RECORDING SUBSETS

----------------------------------------------------------------------
parallel_analysis_detail.csv
Shape: (294, 6)
Columns: ['representation', 'PC', 'observed_eigenvalue', 'null_mean', 'null_95pct', 'retained']

----------------------------------------------------------------------
parallel_analysis_summary.csv
Shape: (6, 10)
Columns: ['representation', 'n_features', 'retained_PCs', 'first_PC_below_null', 'PC1_observed', 'PC1_null_95', 'PC2_observed', 'PC2_null_95', 'PC5_observed', 'PC5_null_95']

----------------------------------------------------------------------
pca_first10.csv
Shape: (60, 4)
Columns: ['representation', 'PC', 'explained_variance_ratio', 'cumulative_variance']

----------------------------------------------------------------------
pca_variance_summary.csv
Shape: (6, 16)
Columns: ['representation', 'n_features', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC10', 'cumulative_5', 'cumulative_10', 'cumulative_20', 'PCs_to_50pct',

In [39]:
# ============================================================
# 06F.1 — Cell 9: Trace the 56-species ecological subset
# ============================================================

NOTEBOOK_DIR = PROJECT_ROOT / "research" / "notebooks"

TARGET_TERMS = [
    "56 species",
    "56_species",
    "species_count",
    "species threshold",
    "minimum species",
    "species subset",
    "eBird",
    "ebird",
    "172 recordings",
]

print("=" * 70)
print("TARGETED ECOLOGICAL-SUBSET SEARCH")
print("=" * 70)

results = []

for path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):

    try:
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
    except Exception:
        continue

    # Combine markdown + code cells
    cell_texts = []

    for cell in notebook.get("cells", []):
        source = "".join(cell.get("source", []))
        cell_texts.append(source)

    notebook_text = "\n".join(cell_texts).lower()

    found = [
        term for term in TARGET_TERMS
        if term.lower() in notebook_text
    ]

    if found:
        results.append((path.name, found))

for filename, found in results:
    print(f"\n{filename}")
    print("  " + ", ".join(found))

print(f"\nNotebooks containing relevant terms: {len(results)}")

TARGETED ECOLOGICAL-SUBSET SEARCH

02_data_acquisition.ipynb
  species_count, eBird, ebird

03_data_audit.ipynb
  eBird, ebird

04_cleaning.ipynb
  eBird, ebird

06_eda.ipynb
  eBird, ebird

06_F_species_association.ipynb
  56 species, 56_species, species_count, species threshold, minimum species, species subset, eBird, ebird, 172 recordings

Notebooks containing relevant terms: 5


In [40]:
# Find the existing 06F notebook
for p in sorted((PROJECT_ROOT / "research" / "notebooks").glob("*06*F*species*")):
    print(p)

c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\notebooks\06_F_species_association.ipynb


In [41]:
# ============================================================
# 06F.1 — Cell 10: Find code defining 56 / 172 together
# ============================================================

import json

NOTEBOOK_DIR = PROJECT_ROOT / "research" / "notebooks"

print("=" * 70)
print("SEARCHING FOR THE ORIGINAL 56 / 172 SELECTION LOGIC")
print("=" * 70)

for path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):

    try:
        with open(path, "r", encoding="utf-8") as f:
            nb_tmp = json.load(f)
    except Exception:
        continue

    for cell_num, cell in enumerate(nb_tmp.get("cells", [])):

        source = "".join(cell.get("source", []))

        # Look for cells that mention both target quantities.
        lower = source.lower()

        has_56 = (
            "56" in lower
            or "56_species" in lower
            or "56 species" in lower
        )

        has_172 = (
            "172" in lower
            or "172 recordings" in lower
        )

        if has_56 and has_172:
            print("\n" + "-" * 70)
            print(f"FILE: {path.name}")
            print(f"CELL: {cell_num}")
            print("-" * 70)
            print(source[:8000])

SEARCHING FOR THE ORIGINAL 56 / 172 SELECTION LOGIC

----------------------------------------------------------------------
FILE: 06_F_species_association.ipynb
CELL: 1
----------------------------------------------------------------------
# 06F — PCA Species Association

## Objective

This notebook tests whether the multivariate acoustic structure identified
without ecological labels is systematically associated with species identity.

The scientific question is:

> Do the dominant acoustic dimensions identified from unsupervised acoustic
> features show systematic association with species identity?

Species identity is introduced only after PCA construction. It is therefore
used for post-hoc ecological interpretation rather than as an input to the
unsupervised representation.

## Primary analysis unit

The recording is the primary unit of analysis.

The focal species-association subset contains:

- 56 species
- 172 recordings

This restriction is applied before statistical testing to

In [42]:
# ============================================================
# 06F.1 — Cell 10: Search EDA notebook for species-selection
#             logic involving eBird
# ============================================================

EDA_PATH = PROJECT_ROOT / "research" / "notebooks" / "06_eda.ipynb"

with open(EDA_PATH, "r", encoding="utf-8") as f:
    eda_nb = json.load(f)

print("=" * 70)
print("06_EDA — eBird / SPECIES SELECTION CELLS")
print("=" * 70)

terms = [
    "ebird",
    "species",
    "threshold",
    "merge",
    "common_species",
    "overlap",
    "shared",
]

for i, cell in enumerate(eda_nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    lower = source.lower()

    found = [term for term in terms if term.lower() in lower]

    if found:
        print("\n" + "-" * 70)
        print(f"CELL {i} | matches: {found}")
        print("-" * 70)
        print(source[:6000])

06_EDA — eBird / SPECIES SELECTION CELLS

----------------------------------------------------------------------
CELL 0 | matches: ['ebird']
----------------------------------------------------------------------
# Notebook 06 — Exploratory Data Analysis

## From Soundscapes to Biodiversity

This notebook performs exploratory data analysis on the cleaned acoustic and independent eBird observation datasets produced by the preceding data preparation and audio feature extraction workflows.

The analysis examines sampling coverage, geographic and temporal structure, recording characteristics, acoustic feature distributions, data-quality patterns, and relationships among acoustic descriptors. The resulting figures and structured JSON outputs are used both for the project website and as the exploratory foundation for subsequent statistical learning and modeling modules.

The analysis distinguishes between **dataset representation** and **ecological inference**. Xeno-canto recordings represent

In [43]:
# ============================================================
# 06F.1 — Cell 11: Find the original acoustic species-selection
#             logic in acquisition / cleaning notebooks
# ============================================================

SEARCH_NOTEBOOKS = [
    "02_data_acquisition.ipynb",
    "03_data_audit.ipynb",
    "04_cleaning.ipynb",
    "05_feature_extraction.ipynb",
]

SEARCH_TERMS = [
    "species_stratified",
    "round_robin",
    "species selection",
    "species_count",
    "recordings_per_species",
    "minimum",
    "56",
    "172",
    "eBird",
]

for notebook_name in SEARCH_NOTEBOOKS:

    path = PROJECT_ROOT / "research" / "notebooks" / notebook_name

    if not path.exists():
        print(f"\nNOT FOUND: {notebook_name}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    print("\n" + "=" * 80)
    print(f"{notebook_name}")
    print("=" * 80)

    for cell_num, cell in enumerate(nb.get("cells", [])):

        source = "".join(cell.get("source", []))
        lower = source.lower()

        matches = [
            term for term in SEARCH_TERMS
            if term.lower() in lower
        ]

        if matches:
            print("\n" + "-" * 80)
            print(f"CELL {cell_num} | matches: {matches}")
            print("-" * 80)
            print(source[:10000])


02_data_acquisition.ipynb

--------------------------------------------------------------------------------
CELL 0 | matches: ['eBird']
--------------------------------------------------------------------------------
import ebird

print("Loaded from:", ebird.__file__)
print("fetch_region_observations:", hasattr(ebird, "fetch_region_observations"))
print("save_raw_observations:", hasattr(ebird, "save_raw_observations"))
print("REGIONS:", hasattr(ebird, "REGIONS"))

--------------------------------------------------------------------------------
CELL 1 | matches: ['eBird']
--------------------------------------------------------------------------------
import sys
from pathlib import Path

ACQUISITION_DIR = Path.cwd()

if str(ACQUISITION_DIR) not in sys.path:
    sys.path.insert(0, str(ACQUISITION_DIR))

from xenocanto import (
    fetch_metadata,
    save_raw_metadata,
    download_audio_sample,
    A1_TARGETS,
)

from ebird import (
    fetch_region_observations,
    save_raw_observati

In [44]:
# ============================================================
# 06F.1 — Cell 12: Inspect species-stratified acquisition logic
# ============================================================

ACQ_PATH = PROJECT_ROOT / "research" / "notebooks" / "02_data_acquisition.ipynb"

with open(ACQ_PATH, "r", encoding="utf-8") as f:
    acq_nb = json.load(f)

KEYWORDS = [
    "species_stratified",
    "round_robin",
    "species",
    "selection",
    "recordings_per_species",
    "selected",
    "manifest",
    "sampling",
]

for i, cell in enumerate(acq_nb.get("cells", [])):

    source = "".join(cell.get("source", []))
    lower = source.lower()

    # Prioritize cells containing actual code rather than markdown-only
    # descriptions.
    if any(keyword in lower for keyword in KEYWORDS):

        print("\n" + "=" * 90)
        print(f"CELL {i}")
        print("=" * 90)
        print(source[:15000])


CELL 5
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
RAW_XC_DIR = PROJECT_ROOT / "data" / "raw" / "xenocanto"

for state in A1_TARGETS:

    metadata_path = RAW_XC_DIR / f"xc_metadata_raw_{state}.json"

    with open(metadata_path, encoding="utf-8") as f:
        recs = json.load(f)[:A1_TARGETS[state]]

    species = {
        r["en"]
        for r in recs
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(recs)} recordings, "
        f"{len(species)} unique species"
    )

CELL 6
print("A1 Xeno-canto sample")
print("=" * 40)

sample_species = {
    "CA": 6,
    "AZ": 16,
    "TX": 15,
}

for state, target in A1_TARGETS.items():
    print(
        f"{state}: "
        f"{target} recordings | "
        f"{sample_species[state]} unique species"
    )

print(f"\nTotal recordings: {sum(A1_TARGETS.values())}")
print("Audio download failures: 0")
print("Restricted/skipped recordings: 0")

CELL 7
for state in A1_TARGETS:
    with open(RA

In [45]:
# ============================================================
# 06F.2 — Full species representation audit
# ============================================================

print("=" * 80)
print("FULL SPECIES REPRESENTATION AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# Load recording-level acoustic feature table
# ------------------------------------------------------------

FEATURE_TABLE_PATH = (
    DATA_PROCESSED_DIR
    / "acoustic_features_recording.csv"
)

recording_df = pd.read_csv(FEATURE_TABLE_PATH)

required_cols = [
    "recording_id",
    "species_id",
]

missing_cols = [
    col for col in required_cols
    if col not in recording_df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

n_recordings = recording_df["recording_id"].nunique()
n_species = recording_df["species_id"].nunique()

print(f"Rows:                         {len(recording_df):,}")
print(f"Unique recordings:            {n_recordings:,}")
print(f"Unique species:               {n_species:,}")

assert len(recording_df) == 275
assert n_recordings == 275
assert n_species == 159

# ------------------------------------------------------------
# Species recording counts
# ------------------------------------------------------------

species_counts = (
    recording_df
    .groupby("species_id", dropna=False)
    .agg(
        n_recordings=("recording_id", "nunique")
    )
    .sort_values(
        "n_recordings",
        ascending=False
    )
)

species_counts["n_species_with_this_count"] = (
    species_counts["n_recordings"]
    .map(
        species_counts["n_recordings"]
        .value_counts()
    )
)

print("\nSpecies-level recording counts:")
display(species_counts)

# ------------------------------------------------------------
# Frequency distribution
# ------------------------------------------------------------

count_distribution = (
    species_counts["n_recordings"]
    .value_counts()
    .sort_index()
    .rename_axis("recordings_per_species")
    .reset_index(name="n_species")
)

count_distribution["n_recordings"] = (
    count_distribution["recordings_per_species"]
    * count_distribution["n_species"]
)

count_distribution["pct_species"] = (
    100
    * count_distribution["n_species"]
    / n_species
)

count_distribution["pct_recordings"] = (
    100
    * count_distribution["n_recordings"]
    / n_recordings
)

print("\n" + "-" * 80)
print("SPECIES-COUNT DISTRIBUTION")
print("-" * 80)

display(count_distribution)

# ------------------------------------------------------------
# Cumulative thresholds
# ------------------------------------------------------------

threshold_rows = []

for minimum_recordings in range(
    1,
    int(species_counts["n_recordings"].max()) + 1
):

    eligible = (
        species_counts["n_recordings"]
        >= minimum_recordings
    )

    threshold_rows.append({
        "minimum_recordings_per_species":
            minimum_recordings,

        "n_species":
            int(eligible.sum()),

        "n_recordings":
            int(
                species_counts.loc[
                    eligible,
                    "n_recordings"
                ].sum()
            ),

        "pct_species":
            100 * eligible.sum() / n_species,

        "pct_recordings":
            100
            * species_counts.loc[
                eligible,
                "n_recordings"
            ].sum()
            / n_recordings,
    })

threshold_summary = pd.DataFrame(threshold_rows)

print("\n" + "-" * 80)
print("THRESHOLD SUMMARY")
print("-" * 80)

display(threshold_summary)

# ------------------------------------------------------------
# State × species representation
# ------------------------------------------------------------

state_species = (
    recording_df
    .groupby(
        ["state", "species_id"],
        dropna=False
    )
    .agg(
        n_recordings=("recording_id", "nunique")
    )
    .reset_index()
)

state_species_summary = (
    state_species
    .groupby("state")
    .agg(
        n_species=("species_id", "nunique"),
        n_recordings=("n_recordings", "sum"),
        median_recordings_per_species=(
            "n_recordings",
            "median"
        ),
        max_recordings_per_species=(
            "n_recordings",
            "max"
        ),
    )
    .reset_index()
)

print("\n" + "-" * 80)
print("STATE-LEVEL SPECIES REPRESENTATION")
print("-" * 80)

display(state_species_summary)

# ------------------------------------------------------------
# Identify sparse species explicitly
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SPARSE SPECIES")
print("-" * 80)

for threshold in [1, 2, 3, 4, 5]:

    n_species_threshold = int(
        (species_counts["n_recordings"] >= threshold).sum()
    )

    n_recordings_threshold = int(
        species_counts.loc[
            species_counts["n_recordings"] >= threshold,
            "n_recordings"
        ].sum()
    )

    print(
        f">= {threshold} recordings/species: "
        f"{n_species_threshold} species, "
        f"{n_recordings_threshold} recordings"
    )

print("\nAudit complete.")

FULL SPECIES REPRESENTATION AUDIT
Rows:                         275
Unique recordings:            275
Unique species:               159

Species-level recording counts:


,n_recordings,n_species_with_this_count
species_id,,
Anas carolinensis,5,7
Archilochus alexandri,5,7
Selasphorus platycercus,5,7
Mareca strepera,5,7
Phalaenoptilus nuttallii,5,7
...,...,...
Tyrannus tyrannus,1,103
Tyrannus vociferans,1,103
Tyto furcata,1,103



--------------------------------------------------------------------------------
SPECIES-COUNT DISTRIBUTION
--------------------------------------------------------------------------------


,recordings_per_species,n_species,n_recordings,pct_species,pct_recordings
0,1,103,103,64.779874,37.454545
1,2,23,46,14.465409,16.727273
2,3,13,39,8.176101,14.181818
3,4,13,52,8.176101,18.909091
4,5,7,35,4.402516,12.727273



--------------------------------------------------------------------------------
THRESHOLD SUMMARY
--------------------------------------------------------------------------------


,minimum_recordings_per_species,n_species,n_recordings,pct_species,pct_recordings
0,1,159,275,100.000000,100.000000
1,2,56,172,35.220126,62.545455
2,3,33,126,20.754717,45.818182
3,4,20,87,12.578616,31.636364
4,5,7,35,4.402516,12.727273



--------------------------------------------------------------------------------
STATE-LEVEL SPECIES REPRESENTATION
--------------------------------------------------------------------------------


,state,n_species,n_recordings,median_recordings_per_species,max_recordings_per_species
0,AZ,44,75,2.0,2
1,CA,66,100,2.0,2
2,TX,100,100,1.0,1



--------------------------------------------------------------------------------
SPARSE SPECIES
--------------------------------------------------------------------------------
>= 1 recordings/species: 159 species, 275 recordings
>= 2 recordings/species: 56 species, 172 recordings
>= 3 recordings/species: 33 species, 126 recordings
>= 4 recordings/species: 20 species, 87 recordings
>= 5 recordings/species: 7 species, 35 recordings

Audit complete.


In [46]:
# ============================================================
# 06F.2 — Full species representation audit
#              corrected distribution calculation
# ============================================================

print("=" * 80)
print("FULL SPECIES REPRESENTATION AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# Load recording-level acoustic feature table
# ------------------------------------------------------------

FEATURE_TABLE_PATH = (
    DATA_PROCESSED_DIR
    / "acoustic_features_recording.csv"
)

recording_df = pd.read_csv(FEATURE_TABLE_PATH)

required_cols = [
    "recording_id",
    "species_id",
    "state",
]

missing_cols = [
    col for col in required_cols
    if col not in recording_df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

n_recordings = recording_df["recording_id"].nunique()
n_species = recording_df["species_id"].nunique()

print(f"Rows:                         {len(recording_df):,}")
print(f"Unique recordings:            {n_recordings:,}")
print(f"Unique species:               {n_species:,}")

assert len(recording_df) == 275
assert n_recordings == 275
assert n_species == 159

# ------------------------------------------------------------
# Species-level recording counts
# ------------------------------------------------------------

species_counts = (
    recording_df
    .groupby("species_id", dropna=False)
    .agg(
        n_recordings=("recording_id", "nunique")
    )
    .sort_values(
        "n_recordings",
        ascending=False
    )
)

print("\nSpecies-level recording counts:")
display(species_counts)

# ------------------------------------------------------------
# Correct frequency distribution
# ------------------------------------------------------------

count_distribution = (
    species_counts["n_recordings"]
    .value_counts()
    .sort_index()
    .rename_axis("recordings_per_species")
    .reset_index(name="n_species")
)

count_distribution["n_recordings"] = (
    count_distribution["recordings_per_species"]
    * count_distribution["n_species"]
)

count_distribution["pct_species"] = (
    100
    * count_distribution["n_species"]
    / n_species
)

count_distribution["pct_recordings"] = (
    100
    * count_distribution["n_recordings"]
    / n_recordings
)

print("\n" + "-" * 80)
print("CORRECT SPECIES-COUNT DISTRIBUTION")
print("-" * 80)

display(count_distribution)

# ------------------------------------------------------------
# Threshold summary
# ------------------------------------------------------------

threshold_rows = []

max_reps = int(
    species_counts["n_recordings"].max()
)

for minimum_recordings in range(1, max_reps + 1):

    eligible = (
        species_counts["n_recordings"]
        >= minimum_recordings
    )

    n_species_threshold = int(eligible.sum())

    n_recordings_threshold = int(
        species_counts.loc[
            eligible,
            "n_recordings"
        ].sum()
    )

    threshold_rows.append({
        "minimum_recordings_per_species":
            minimum_recordings,

        "n_species":
            n_species_threshold,

        "n_recordings":
            n_recordings_threshold,

        "pct_species":
            100 * n_species_threshold / n_species,

        "pct_recordings":
            100 * n_recordings_threshold / n_recordings,
    })

threshold_summary = pd.DataFrame(threshold_rows)

print("\n" + "-" * 80)
print("THRESHOLD SUMMARY")
print("-" * 80)

display(threshold_summary)

# ------------------------------------------------------------
# State-level species representation
# ------------------------------------------------------------

state_species = (
    recording_df
    .groupby(
        ["state", "species_id"],
        dropna=False
    )
    .agg(
        n_recordings=("recording_id", "nunique")
    )
    .reset_index()
)

state_species_summary = (
    state_species
    .groupby("state")
    .agg(
        n_species=("species_id", "nunique"),
        n_recordings=("n_recordings", "sum"),
        median_recordings_per_species=(
            "n_recordings",
            "median"
        ),
        max_recordings_per_species=(
            "n_recordings",
            "max"
        ),
    )
    .reset_index()
)

print("\n" + "-" * 80)
print("STATE-LEVEL SPECIES REPRESENTATION")
print("-" * 80)

display(state_species_summary)

# ------------------------------------------------------------
# State composition of each threshold cohort
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("STATE COMPOSITION OF THRESHOLD COHORTS")
print("-" * 80)

for minimum_recordings in range(1, max_reps + 1):

    eligible_species = set(
        species_counts.index[
            species_counts["n_recordings"]
            >= minimum_recordings
        ]
    )

    cohort = recording_df[
        recording_df["species_id"]
        .isin(eligible_species)
    ]

    print(
        f"\n>= {minimum_recordings} recordings/species"
    )

    print(
        cohort["state"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print(
        f"Total recordings: {len(cohort)}"
    )

    print(
        f"Total species: "
        f"{cohort['species_id'].nunique()}"
    )

print("\nAudit complete.")

FULL SPECIES REPRESENTATION AUDIT
Rows:                         275
Unique recordings:            275
Unique species:               159

Species-level recording counts:


,n_recordings
species_id,
Anas carolinensis,5
Archilochus alexandri,5
Selasphorus platycercus,5
Mareca strepera,5
Phalaenoptilus nuttallii,5
...,...
Tyrannus tyrannus,1
Tyrannus vociferans,1
Tyto furcata,1



--------------------------------------------------------------------------------
CORRECT SPECIES-COUNT DISTRIBUTION
--------------------------------------------------------------------------------


,recordings_per_species,n_species,n_recordings,pct_species,pct_recordings
0,1,103,103,64.779874,37.454545
1,2,23,46,14.465409,16.727273
2,3,13,39,8.176101,14.181818
3,4,13,52,8.176101,18.909091
4,5,7,35,4.402516,12.727273



--------------------------------------------------------------------------------
THRESHOLD SUMMARY
--------------------------------------------------------------------------------


,minimum_recordings_per_species,n_species,n_recordings,pct_species,pct_recordings
0,1,159,275,100.000000,100.000000
1,2,56,172,35.220126,62.545455
2,3,33,126,20.754717,45.818182
3,4,20,87,12.578616,31.636364
4,5,7,35,4.402516,12.727273



--------------------------------------------------------------------------------
STATE-LEVEL SPECIES REPRESENTATION
--------------------------------------------------------------------------------


,state,n_species,n_recordings,median_recordings_per_species,max_recordings_per_species
0,AZ,44,75,2.0,2
1,CA,66,100,2.0,2
2,TX,100,100,1.0,1



--------------------------------------------------------------------------------
STATE COMPOSITION OF THRESHOLD COHORTS
--------------------------------------------------------------------------------

>= 1 recordings/species
state
AZ     75
CA    100
TX    100
Total recordings: 275
Total species: 159

>= 2 recordings/species
state
AZ    72
CA    79
TX    21
Total recordings: 172
Total species: 56

>= 3 recordings/species
state
AZ    48
CA    59
TX    19
Total recordings: 126
Total species: 33

>= 4 recordings/species
state
AZ    40
CA    37
TX    10
Total recordings: 87
Total species: 20

>= 5 recordings/species
state
AZ    14
CA    14
TX     7
Total recordings: 35
Total species: 7

Audit complete.


In [47]:
# 06F.3 — Species × State Structure
#
# Purpose:
# Before running the PCA–species association test, quantify how species
# representation is distributed across states.
#
# This determines whether an unrestricted species-label permutation would
# inadvertently confound species association with geographic/state structure.

# -------------------------------------------------------------------
# 1. Build species × state recording-count table
# -------------------------------------------------------------------

species_state_counts = pd.crosstab(
    recording_df["species_id"],
    recording_df["state"]
).sort_index()

print("=" * 80)
print("SPECIES × STATE RECORDING STRUCTURE")
print("=" * 80)

print(f"Species: {species_state_counts.shape[0]}")
print(f"States:  {species_state_counts.shape[1]}")
print(f"Total recordings: {species_state_counts.to_numpy().sum()}")

display(species_state_counts)

# -------------------------------------------------------------------
# 2. Number of states represented by each species
# -------------------------------------------------------------------

species_n_states = (
    (species_state_counts > 0)
    .sum(axis=1)
    .rename("n_states")
)

state_span_summary = (
    species_n_states
    .value_counts()
    .sort_index()
    .rename_axis("n_states")
    .reset_index(name="n_species")
)

print("\n" + "=" * 80)
print("SPECIES GEOGRAPHIC SPAN")
print("=" * 80)

display(state_span_summary)

# -------------------------------------------------------------------
# 3. How many species are state-specific vs shared?
# -------------------------------------------------------------------

n_species_total = len(species_n_states)
n_species_one_state = int((species_n_states == 1).sum())
n_species_two_states = int((species_n_states == 2).sum())
n_species_three_states = int((species_n_states == 3).sum())

print("\nSpecies represented in exactly one state:",
      f"{n_species_one_state} / {n_species_total}"
      f" ({100 * n_species_one_state / n_species_total:.1f}%)")

print("Species represented in exactly two states:",
      f"{n_species_two_states} / {n_species_total}"
      f" ({100 * n_species_two_states / n_species_total:.1f}%)")

print("Species represented in all three states:",
      f"{n_species_three_states} / {n_species_total}"
      f" ({100 * n_species_three_states / n_species_total:.1f}%)")

# -------------------------------------------------------------------
# 4. State-specific species in the ≥2-recording cohort
# -------------------------------------------------------------------

species_state_subset = species_state_counts.loc[
    species_state_counts.index.isin(species_counts.loc[
        species_counts["n_recordings"] >= 2
    ].index)
].copy()

print("\n" + "=" * 80)
print("≥2-RECORDING SPECIES COHORT: SPECIES × STATE")
print("=" * 80)

print(f"Species: {species_state_subset.shape[0]}")
print(f"Recordings: {int(species_state_subset.to_numpy().sum())}")

display(species_state_subset)

# -------------------------------------------------------------------
# 5. State distribution of the ≥2 cohort
# -------------------------------------------------------------------

cohort_state_summary = (
    species_state_subset
    .sum(axis=0)
    .rename("n_recordings")
    .to_frame()
)

cohort_state_summary["pct_recordings"] = (
    100
    * cohort_state_summary["n_recordings"]
    / cohort_state_summary["n_recordings"].sum()
)

print("\n" + "=" * 80)
print("≥2-RECORDING COHORT: STATE COMPOSITION")
print("=" * 80)

display(cohort_state_summary)

# -------------------------------------------------------------------
# 6. Identify species occurring in multiple states
# -------------------------------------------------------------------

shared_species = species_state_subset.loc[
    (species_state_subset > 0).sum(axis=1) >= 2
]

print("\n" + "=" * 80)
print("SPECIES SHARED ACROSS ≥2 STATES")
print("=" * 80)

print(f"Shared species: {len(shared_species)}")

if len(shared_species) > 0:
    display(shared_species)

# -------------------------------------------------------------------
# 7. Check whether each state contains repeated species
# -------------------------------------------------------------------

state_species_repeat_summary = []

for state in sorted(recording_df["state"].dropna().unique()):
    state_counts = (
        recording_df.loc[
            recording_df["state"] == state
        ]
        .groupby("species_id")["recording_id"]
        .nunique()
    )

    state_species_repeat_summary.append({
        "state": state,
        "n_species": int(state_counts.shape[0]),
        "n_species_with_2plus": int((state_counts >= 2).sum()),
        "n_recordings": int(state_counts.sum()),
        "max_recordings_per_species": int(state_counts.max()),
    })

state_species_repeat_summary = pd.DataFrame(
    state_species_repeat_summary
)

print("\n" + "=" * 80)
print("WITHIN-STATE SPECIES REPLICATION")
print("=" * 80)

display(state_species_repeat_summary)

# -------------------------------------------------------------------
# 8. Preliminary interpretation
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("PRELIMINARY PERMUTATION IMPLICATION")
print("=" * 80)

print(
    "Species and state are not independent grouping variables in this "
    "sampling design. Therefore, an unrestricted permutation of species "
    "labels could partially convert geographic structure into apparent "
    "species association."
)

print(
    "The next analysis should therefore compare an unrestricted permutation "
    "scheme with a state-restricted permutation scheme, provided that the "
    "within-state replication structure supports the latter."
)

SPECIES × STATE RECORDING STRUCTURE
Species: 159
States:  3
Total recordings: 275


state,AZ,CA,TX
species_id,,,
Accipiter cooperii,0,0,1
Aechmophorus clarkii,0,0,1
Aeronautes saxatalis,1,1,0
Aix sponsa,0,1,0
Alectoris chukar,0,2,1
...,...,...,...
Tyrannus tyrannus,0,0,1
Tyrannus vociferans,0,0,1
Tyto furcata,0,0,1



SPECIES GEOGRAPHIC SPAN


,n_states,n_species
0,1,118
1,2,31
2,3,10



Species represented in exactly one state: 118 / 159 (74.2%)
Species represented in exactly two states: 31 / 159 (19.5%)
Species represented in all three states: 10 / 159 (6.3%)

≥2-RECORDING SPECIES COHORT: SPECIES × STATE
Species: 56
Recordings: 172


...


SPECIES SHARED ACROSS ≥2 STATES
Shared species: 41


state,AZ,CA,TX
species_id,,,
Aeronautes saxatalis,1,1,0
Alectoris chukar,0,2,1
Anas acuta,1,1,0
Anas carolinensis,2,2,1
Anas platyrhynchos,2,1,1
Anser albifrons,2,2,0
Anser caerulescens,1,1,0
Antrostomus arizonae,2,1,1
Antrostomus vociferus,0,2,1



WITHIN-STATE SPECIES REPLICATION


,state,n_species,n_species_with_2plus,n_recordings,max_recordings_per_species
0,AZ,44,31,75,2
1,CA,66,34,100,2
2,TX,100,0,100,1



PRELIMINARY PERMUTATION IMPLICATION
Species and state are not independent grouping variables in this sampling design. Therefore, an unrestricted permutation of species labels could partially convert geographic structure into apparent species association.
The next analysis should therefore compare an unrestricted permutation scheme with a state-restricted permutation scheme, provided that the within-state replication structure supports the latter.


In [48]:
# 06F.4a — Locate Existing PCA Score/Result Artifacts

print("=" * 80)
print("PCA ANALYSIS DIRECTORY")
print("=" * 80)

print("PCA_ANALYSIS_DIR:")
print(PCA_ANALYSIS_DIR)
print()

if not PCA_ANALYSIS_DIR.exists():
    raise FileNotFoundError(
        f"PCA analysis directory does not exist:\n{PCA_ANALYSIS_DIR}"
    )

for path in sorted(PCA_ANALYSIS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PCA_ANALYSIS_DIR))

PCA ANALYSIS DIRECTORY
PCA_ANALYSIS_DIR:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis

baseline_components.npy
baseline_explained_variance.npy
baseline_explained_variance_ratio.npy
baseline_feature_names.json
baseline_pa_null_95pct.npy
baseline_pa_null_mean.npy
baseline_pa_observed_eigenvalues.npy
baseline_pa_retained.npy
baseline_scores.npy
baseline_singular_values.npy
block_balanced_components.npy
block_balanced_explained_variance.npy
block_balanced_explained_variance_ratio.npy
block_balanced_feature_names.json
block_balanced_pa_null_95pct.npy
block_balanced_pa_null_mean.npy
block_balanced_pa_observed_eigenvalues.npy
block_balanced_pa_retained.npy
block_balanced_scores.npy
block_balanced_singular_values.npy
bootstrap_stability\baseline_bootstrap_explained_variance_ratio.npy
bootstrap_stability\baseline_principal_angles_deg.npy
bootstrap_stability\block_balanced_bootstrap_explained_variance_ratio.npy
bootstrap_stability\block_balanced_princ

In [49]:
# 06F.4 — Species vs. State Structure in PCA Space
#
# Purpose:
# Quantify whether the retained PCA structure is associated with state
# as well as species, before selecting the permutation design.
#
# PCA was constructed without species or state labels. These variables are
# introduced only here for post-hoc ecological interpretation.

# -------------------------------------------------------------------
# 1. Load baseline PCA scores from 06C
# -------------------------------------------------------------------

baseline_scores_array = np.load(
    PCA_ANALYSIS_DIR / "baseline_scores.npy"
)

baseline_feature_names = json.loads(
    (PCA_ANALYSIS_DIR / "baseline_feature_names.json").read_text()
)

# 06C stores parallel-analysis retention as a Boolean/indicator vector,
# one value for each PCA component.
baseline_pa_retained = np.asarray(
    np.load(PCA_ANALYSIS_DIR / "baseline_pa_retained.npy")
)

if baseline_pa_retained.ndim != 1:
    raise ValueError(
        "Expected baseline_pa_retained.npy to be a 1-D retention mask, "
        f"but found shape {baseline_pa_retained.shape}."
    )

if baseline_pa_retained.size != baseline_scores_array.shape[1]:
    raise ValueError(
        "Retention mask length does not match the number of PCA scores: "
        f"mask={baseline_pa_retained.size}, "
        f"scores={baseline_scores_array.shape[1]}."
    )

baseline_retained = int(
    baseline_pa_retained.astype(bool).sum()
)

print("=" * 80)
print("BASELINE PCA SCORE AUDIT")
print("=" * 80)

print("Score matrix shape:", baseline_scores_array.shape)
print("Feature count:", len(baseline_feature_names))

print(
    "Parallel-analysis retention mask:",
    baseline_pa_retained.astype(int)
)

print(
    "Parallel-analysis retained PCs:",
    baseline_retained
)

assert baseline_scores_array.shape[0] == N_RECORDINGS_FULL
assert baseline_scores_array.shape[1] == len(baseline_feature_names)
assert np.isfinite(baseline_scores_array).all()

# -------------------------------------------------------------------
# 2. Load the exact recording-level feature table used for PCA
# -------------------------------------------------------------------

recording_feature_table = pd.read_csv(
    DATA_PROCESSED_DIR / "acoustic_features_recording.csv"
)

recording_feature_table["recording_id"] = (
    recording_feature_table["recording_id"].astype(str)
)

recording_metadata = recording_feature_table[
    ["recording_id", "species_id", "state"]
].copy()

assert len(recording_metadata) == baseline_scores_array.shape[0]
assert recording_metadata["recording_id"].is_unique

# -------------------------------------------------------------------
# 3. Reconstruct PCA score table
# -------------------------------------------------------------------

pc_columns = [
    f"PC{i}"
    for i in range(1, baseline_scores_array.shape[1] + 1)
]

baseline_scores = pd.DataFrame(
    baseline_scores_array,
    columns=pc_columns
)

baseline_scores.insert(
    0,
    "recording_id",
    recording_metadata["recording_id"].values
)

pca_species_state = baseline_scores.merge(
    recording_metadata,
    on="recording_id",
    how="inner",
    validate="one_to_one"
)

assert len(pca_species_state) == N_RECORDINGS_FULL
assert pca_species_state["recording_id"].is_unique

print("\nMerged PCA + metadata shape:", pca_species_state.shape)

# -------------------------------------------------------------------
# 4. Define retained PCA subspace
# -------------------------------------------------------------------

N_RETAINED_PCS = baseline_retained

retained_pc_columns = [
    f"PC{i}"
    for i in range(1, N_RETAINED_PCS + 1)
]

X_pca = pca_species_state[
    retained_pc_columns
].to_numpy(dtype=float)

print("\nRetained PCA dimensions:", X_pca.shape)
print("Retained PCs:", retained_pc_columns)

assert X_pca.shape == (
    N_RECORDINGS_FULL,
    N_RETAINED_PCS
)

# -------------------------------------------------------------------
# 5. State-level PCA centroids and dispersion
# -------------------------------------------------------------------

state_centroids = (
    pca_species_state
    .groupby("state")[retained_pc_columns]
    .mean()
)

state_distances = []

for _, row in pca_species_state.iterrows():

    point = row[
        retained_pc_columns
    ].to_numpy(dtype=float)

    centroid = (
        state_centroids
        .loc[row["state"]]
        .to_numpy(dtype=float)
    )

    state_distances.append(
        np.linalg.norm(point - centroid)
    )

pca_species_state[
    "distance_to_state_centroid"
] = state_distances

state_dispersion = (
    pca_species_state
    .groupby("state")[
        "distance_to_state_centroid"
    ]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std"
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("PCA DISPERSION BY STATE")
print("=" * 80)

display(state_dispersion)

# -------------------------------------------------------------------
# 6. Focal ≥2-recording species cohort
# -------------------------------------------------------------------

focal_species = (
    species_counts.loc[
        species_counts["n_recordings"] >= 2
    ]
    .index
)

focal = pca_species_state[
    pca_species_state["species_id"].isin(
        focal_species
    )
].copy()

print("\n" + "=" * 80)
print("FOCAL SPECIES COHORT")
print("=" * 80)

print(
    "Species:",
    focal["species_id"].nunique()
)

print(
    "Recordings:",
    len(focal)
)

assert focal["species_id"].nunique() == N_SPECIES_TARGET
assert len(focal) == N_RECORDINGS_TARGET

# -------------------------------------------------------------------
# 7. Species-level PCA centroids and dispersion
# -------------------------------------------------------------------

species_centroids = (
    focal
    .groupby("species_id")[retained_pc_columns]
    .mean()
)

species_distances = []

for _, row in focal.iterrows():

    point = row[
        retained_pc_columns
    ].to_numpy(dtype=float)

    centroid = (
        species_centroids
        .loc[row["species_id"]]
        .to_numpy(dtype=float)
    )

    species_distances.append(
        np.linalg.norm(point - centroid)
    )

focal[
    "distance_to_species_centroid"
] = species_distances

species_dispersion = (
    focal
    .groupby("species_id")[
        "distance_to_species_centroid"
    ]
    .agg(
        n="count",
        mean="mean",
        median="median"
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("PCA DISPERSION BY SPECIES")
print("=" * 80)

display(species_dispersion)

# -------------------------------------------------------------------
# 8. Compare descriptive within-group dispersion
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("DISPERSION COMPARISON")
print("=" * 80)

print(
    "Mean distance to state centroid:",
    f"{pca_species_state['distance_to_state_centroid'].mean():.4f}"
)

print(
    "Mean distance to species centroid "
    "(≥2-recording cohort):",
    f"{focal['distance_to_species_centroid'].mean():.4f}"
)

# -------------------------------------------------------------------
# 9. State sample sizes
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("STATE SAMPLE SIZES")
print("=" * 80)

display(
    pca_species_state["state"]
    .value_counts()
    .sort_index()
    .rename("n_recordings")
    .to_frame()
)

# -------------------------------------------------------------------
# 10. Species replication in focal cohort
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("SPECIES REPLICATION IN FOCAL COHORT")
print("=" * 80)

display(
    focal["species_id"]
    .value_counts()
    .value_counts()
    .sort_index()
    .rename_axis("recordings_per_species")
    .rename("n_species")
    .to_frame()
)

# -------------------------------------------------------------------
# 11. Design conclusion
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("DESIGN CONCLUSION")
print("=" * 80)

print(
    "The ≥2-recording cohort provides species-level replication, "
    "but species and state remain partially confounded by the acquisition design."
)

print(
    "Because TX has no repeated species within state, a permutation scheme "
    "restricted entirely within state cannot serve as the primary test "
    "for the full three-state cohort."
)

print(
    "The next step is to quantify species association in PCA space using "
    "a multivariate statistic and an explicit permutation design that "
    "preserves the relevant sampling structure."
)

BASELINE PCA SCORE AUDIT
Score matrix shape: (275, 50)
Feature count: 50
Parallel-analysis retention mask: [1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]
Parallel-analysis retained PCs: 9

Merged PCA + metadata shape: (275, 53)

Retained PCA dimensions: (275, 9)
Retained PCs: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9']

PCA DISPERSION BY STATE


,state,n,mean,median,std
0,AZ,75,4.930086,4.415891,2.093862
1,CA,100,5.369777,4.942237,2.145882
2,TX,100,5.543131,5.151649,2.296787



FOCAL SPECIES COHORT
Species: 56
Recordings: 172

PCA DISPERSION BY SPECIES


,species_id,n,mean,median
0,Aeronautes saxatalis,2,2.732808,2.732808
1,Alectoris chukar,3,5.933856,5.588189
2,Alopochen aegyptiaca,2,6.560954,6.560954
3,Anas acuta,2,1.604229,1.604229
4,Anas carolinensis,5,2.615072,2.698360
5,Anas diazi,2,2.057193,2.057193
6,Anas platyrhynchos,4,2.913316,2.480718
7,Anser albifrons,4,4.194226,4.156373
8,Anser caerulescens,2,3.493584,3.493584
9,Antrostomus arizonae,4,3.207344,3.334251



DISPERSION COMPARISON
Mean distance to state centroid: 5.3129
Mean distance to species centroid (≥2-recording cohort): 3.8141

STATE SAMPLE SIZES


,n_recordings
state,
AZ,75
CA,100
TX,100



SPECIES REPLICATION IN FOCAL COHORT


,n_species
recordings_per_species,
2,23
3,13
4,13
5,7



DESIGN CONCLUSION
The ≥2-recording cohort provides species-level replication, but species and state remain partially confounded by the acquisition design.
Because TX has no repeated species within state, a permutation scheme restricted entirely within state cannot serve as the primary test for the full three-state cohort.
The next step is to quantify species association in PCA space using a multivariate statistic and an explicit permutation design that preserves the relevant sampling structure.


In [50]:
# 06F.5 — Observed Multivariate Association in PCA Space
#
# Scientific question:
#
# Do recordings from the same species occupy more similar positions
# in the unsupervised PCA space than expected from the overall
# multivariate variation?
#
# This cell computes the observed pseudo-F statistic only.
# NO permutation testing is performed here.
#
# Species and state are evaluated separately because geographic structure
# may confound species-level acoustic association.

# -------------------------------------------------------------------
# 1. Prepare the focal PCA matrix
# -------------------------------------------------------------------

analysis_df = focal.copy()

analysis_df = analysis_df[
    [
        "recording_id",
        "species_id",
        "state",
        *retained_pc_columns
    ]
].copy()

analysis_df = analysis_df.reset_index(drop=True)

X = analysis_df[
    retained_pc_columns
].to_numpy(dtype=float)

species_labels = (
    analysis_df["species_id"]
    .astype(str)
    .to_numpy()
)

state_labels = (
    analysis_df["state"]
    .astype(str)
    .to_numpy()
)

assert X.shape == (
    N_RECORDINGS_TARGET,
    N_RETAINED_PCS
)

assert np.isfinite(X).all()

# -------------------------------------------------------------------
# 2. Helper function: pseudo-F statistic
# -------------------------------------------------------------------
#
# PERMANOVA-style decomposition:
#
# Total SS = sum squared distances from the overall centroid
#
# Between-group SS = sum over groups of
#                    n_g * squared distance(group centroid,
#                                            overall centroid)
#
# Within-group SS = Total SS - Between-group SS
#
# pseudo-F = (SS_between / df_between) /
#            (SS_within / df_within)

def permanova_pseudo_f(X, groups):
    """
    Calculate a PERMANOVA-style pseudo-F statistic.

    Parameters
    ----------
    X : ndarray, shape (n_samples, n_features)
        Multivariate observations.

    groups : array-like, shape (n_samples,)
        Group labels.

    Returns
    -------
    result : dict
        Total, between, and within sums of squares,
        degrees of freedom, pseudo-F, and group count.
    """

    X = np.asarray(X, dtype=float)
    groups = np.asarray(groups)

    if X.ndim != 2:
        raise ValueError(
            f"X must be 2-D; got shape {X.shape}."
        )

    if len(groups) != X.shape[0]:
        raise ValueError(
            "Number of group labels must equal number of observations."
        )

    if not np.isfinite(X).all():
        raise ValueError(
            "X contains non-finite values."
        )

    unique_groups = np.unique(groups)
    n = X.shape[0]
    k = len(unique_groups)

    if k < 2:
        raise ValueError(
            "At least two groups are required."
        )

    # Overall centroid
    grand_centroid = X.mean(axis=0)

    # Total sum of squares
    centered_total = X - grand_centroid
    ss_total = np.sum(centered_total ** 2)

    # Between-group sum of squares
    ss_between = 0.0

    group_sizes = {}

    for group in unique_groups:

        mask = groups == group

        X_group = X[mask]

        n_group = X_group.shape[0]

        group_sizes[group] = n_group

        group_centroid = X_group.mean(axis=0)

        centroid_difference = (
            group_centroid - grand_centroid
        )

        ss_between += (
            n_group
            * np.sum(centroid_difference ** 2)
        )

    # Within-group sum of squares
    ss_within = ss_total - ss_between

    # Numerical precision can create tiny negative values.
    if ss_within < 0 and np.isclose(
        ss_within,
        0.0,
        atol=1e-10
    ):
        ss_within = 0.0

    df_between = k - 1
    df_within = n - k

    if df_within <= 0:
        raise ValueError(
            "Insufficient residual degrees of freedom."
        )

    if ss_within <= 0:
        pseudo_f = np.inf
    else:
        pseudo_f = (
            (ss_between / df_between)
            /
            (ss_within / df_within)
        )

    return {
        "n": n,
        "n_groups": k,
        "ss_total": ss_total,
        "ss_between": ss_between,
        "ss_within": ss_within,
        "df_between": df_between,
        "df_within": df_within,
        "pseudo_F": pseudo_f,
        "group_sizes": group_sizes
    }


# -------------------------------------------------------------------
# 3. Calculate observed species effect
# -------------------------------------------------------------------

species_result = permanova_pseudo_f(
    X,
    species_labels
)

# -------------------------------------------------------------------
# 4. Calculate observed state effect
# -------------------------------------------------------------------

state_result = permanova_pseudo_f(
    X,
    state_labels
)

# -------------------------------------------------------------------
# 5. Display species result
# -------------------------------------------------------------------

print("=" * 80)
print("OBSERVED SPECIES ASSOCIATION")
print("=" * 80)

print(
    "Observations:",
    species_result["n"]
)

print(
    "Species groups:",
    species_result["n_groups"]
)

print(
    "Between-species SS:",
    f"{species_result['ss_between']:.6f}"
)

print(
    "Within-species SS:",
    f"{species_result['ss_within']:.6f}"
)

print(
    "Pseudo-F:",
    f"{species_result['pseudo_F']:.6f}"
)

# -------------------------------------------------------------------
# 6. Display state result
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVED STATE ASSOCIATION")
print("=" * 80)

print(
    "Observations:",
    state_result["n"]
)

print(
    "State groups:",
    state_result["n_groups"]
)

print(
    "Between-state SS:",
    f"{state_result['ss_between']:.6f}"
)

print(
    "Within-state SS:",
    f"{state_result['ss_within']:.6f}"
)

print(
    "Pseudo-F:",
    f"{state_result['pseudo_F']:.6f}"
)

# -------------------------------------------------------------------
# 7. Compare variance partitioning descriptively
# -------------------------------------------------------------------

species_between_pct = (
    100
    * species_result["ss_between"]
    / species_result["ss_total"]
)

state_between_pct = (
    100
    * state_result["ss_between"]
    / state_result["ss_total"]
)

comparison = pd.DataFrame({
    "grouping": [
        "species",
        "state"
    ],
    "n_groups": [
        species_result["n_groups"],
        state_result["n_groups"]
    ],
    "SS_between": [
        species_result["ss_between"],
        state_result["ss_between"]
    ],
    "SS_within": [
        species_result["ss_within"],
        state_result["ss_within"]
    ],
    "pseudo_F": [
        species_result["pseudo_F"],
        state_result["pseudo_F"]
    ],
    "pct_total_SS_between": [
        species_between_pct,
        state_between_pct
    ]
})

print("\n" + "=" * 80)
print("SPECIES VS. STATE — OBSERVED STRUCTURE")
print("=" * 80)

display(comparison)

# -------------------------------------------------------------------
# 8. Validate sums of squares
# -------------------------------------------------------------------

assert np.isclose(
    species_result["ss_total"],
    species_result["ss_between"]
    + species_result["ss_within"],
    rtol=1e-10,
    atol=1e-10
)

assert np.isclose(
    state_result["ss_total"],
    state_result["ss_between"]
    + state_result["ss_within"],
    rtol=1e-10,
    atol=1e-10
)

print("\n" + "=" * 80)
print("VALIDATION")
print("=" * 80)

print("Species SS decomposition: PASSED")
print("State SS decomposition:   PASSED")

# -------------------------------------------------------------------
# 9. Interpretation guardrail
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("INTERPRETATION GUARDRAIL")
print("=" * 80)

print(
    "These pseudo-F values are observed statistics only."
)

print(
    "They do NOT establish statistical significance until compared "
    "with an appropriate permutation null distribution."
)

print(
    "A larger species pseudo-F than state pseudo-F would be consistent "
    "with stronger species grouping in the observed PCA space, but this "
    "comparison is descriptive and should not be interpreted as evidence "
    "that species effects are independent of geographic sampling."
)

OBSERVED SPECIES ASSOCIATION
Observations: 172
Species groups: 56
Between-species SS: 2884.182682
Within-species SS: 3120.935610
Pseudo-F: 1.949096

OBSERVED STATE ASSOCIATION
Observations: 172
State groups: 3
Between-state SS: 157.015740
Within-state SS: 5848.102552
Pseudo-F: 2.268741

SPECIES VS. STATE — OBSERVED STRUCTURE


,grouping,n_groups,SS_between,SS_within,pseudo_F,pct_total_SS_between
0,species,56,2884.182682,3120.935610,1.949096,48.028741
1,state,3,157.015740,5848.102552,2.268741,2.614699



VALIDATION
Species SS decomposition: PASSED
State SS decomposition:   PASSED

INTERPRETATION GUARDRAIL
These pseudo-F values are observed statistics only.
They do NOT establish statistical significance until compared with an appropriate permutation null distribution.
A larger species pseudo-F than state pseudo-F would be consistent with stronger species grouping in the observed PCA space, but this comparison is descriptive and should not be interpreted as evidence that species effects are independent of geographic sampling.


In [51]:
# 06F.6 — State-Adjusted Species Association
#
# Scientific question:
#
# Does species identity explain multivariate acoustic structure in PCA space
# AFTER accounting for geographic/state structure?
#
# Reduced model:
#     Y ~ State
#
# Full model:
#     Y ~ State + Species
#
# The species effect is evaluated as the additional multivariate variation
# explained after state is accounted for.
#
# Significance is assessed using a Freedman–Lane residual permutation:
#   1. Fit the reduced State-only model.
#   2. Permute its residuals.
#   3. Reconstruct the response under the reduced model.
#   4. Fit the full State + Species model.
#   5. Calculate the species pseudo-F.
#
# IMPORTANT:
# This tests acoustic association with species identity.
# It does NOT test whether PCA directly measures biodiversity or richness.

# -------------------------------------------------------------------
# 1. Prepare analysis matrix
# -------------------------------------------------------------------

analysis_df = focal.copy()

analysis_df = analysis_df[
    [
        "recording_id",
        "species_id",
        "state",
        *retained_pc_columns
    ]
].copy()

analysis_df = analysis_df.reset_index(drop=True)

Y = analysis_df[
    retained_pc_columns
].to_numpy(dtype=float)

species_labels = (
    analysis_df["species_id"]
    .astype(str)
    .to_numpy()
)

state_labels = (
    analysis_df["state"]
    .astype(str)
    .to_numpy()
)

n = Y.shape[0]

assert n == N_RECORDINGS_TARGET
assert Y.shape[1] == N_RETAINED_PCS
assert np.isfinite(Y).all()

# -------------------------------------------------------------------
# 2. Construct design matrices
# -------------------------------------------------------------------

state_categories = np.sort(
    np.unique(state_labels)
)

species_categories = np.sort(
    np.unique(species_labels)
)

# Intercept + state indicators
X_state = np.column_stack([
    np.ones(n),
    *[
        (state_labels == state).astype(float)
        for state in state_categories[1:]
    ]
])

# Species indicators with first species as reference
X_species = np.column_stack([
    (species_labels == species).astype(float)
    for species in species_categories[1:]
])

# Full model: intercept + state + species
X_full = np.column_stack([
    X_state,
    X_species
])

# -------------------------------------------------------------------
# 3. Check model ranks
# -------------------------------------------------------------------

rank_state = np.linalg.matrix_rank(X_state)
rank_full = np.linalg.matrix_rank(X_full)

df_species = rank_full - rank_state
df_residual = n - rank_full

print("=" * 80)
print("MODEL DESIGN")
print("=" * 80)

print("Observations:", n)
print("PCA dimensions:", Y.shape[1])
print("Species groups:", len(species_categories))
print("State groups:", len(state_categories))

print("\nState-only design shape:", X_state.shape)
print("State-only rank:", rank_state)

print("\nFull design shape:", X_full.shape)
print("Full model rank:", rank_full)

print("\nSpecies degrees of freedom:", df_species)
print("Residual degrees of freedom:", df_residual)

assert rank_state == len(state_categories)
assert rank_full > rank_state
assert df_residual > 0

# -------------------------------------------------------------------
# 4. Helper function
# -------------------------------------------------------------------

def fitted_values(X, Y):
    """
    Multivariate least-squares fitted values.

    X: n observations × p parameters
    Y: n observations × q response dimensions

    Returns:
        n observations × q fitted response matrix
    """

    return X @ np.linalg.pinv(X) @ Y


# -------------------------------------------------------------------
# 5. Fit reduced and full observed models
# -------------------------------------------------------------------

Yhat_state = fitted_values(
    X_state,
    Y
)

Yhat_full = fitted_values(
    X_full,
    Y
)

residual_state = (
    Y - Yhat_state
)

residual_full = (
    Y - Yhat_full
)

SS_resid_state = np.sum(
    residual_state ** 2
)

SS_resid_full = np.sum(
    residual_full ** 2
)

# Additional SS explained by species after state
SS_species_adjusted = (
    SS_resid_state
    - SS_resid_full
)

# Numerical precision guard
if (
    SS_species_adjusted < 0
    and np.isclose(
        SS_species_adjusted,
        0.0,
        atol=1e-10
    )
):
    SS_species_adjusted = 0.0

# -------------------------------------------------------------------
# 6. Observed state-adjusted pseudo-F
# -------------------------------------------------------------------

MS_species_adjusted = (
    SS_species_adjusted
    / df_species
)

MS_residual = (
    SS_resid_full
    / df_residual
)

pseudo_F_adjusted = (
    MS_species_adjusted
    / MS_residual
)

# -------------------------------------------------------------------
# 7. Partial R²
# -------------------------------------------------------------------

partial_R2_species = (
    SS_species_adjusted
    /
    (
        SS_species_adjusted
        + SS_resid_full
    )
)

# -------------------------------------------------------------------
# 8. Display observed result
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVED STATE-ADJUSTED SPECIES ASSOCIATION")
print("=" * 80)

print(
    "Species SS after accounting for state:",
    f"{SS_species_adjusted:.6f}"
)

print(
    "Residual SS after state + species:",
    f"{SS_resid_full:.6f}"
)

print(
    "Species degrees of freedom:",
    df_species
)

print(
    "Residual degrees of freedom:",
    df_residual
)

print(
    "State-adjusted pseudo-F:",
    f"{pseudo_F_adjusted:.6f}"
)

print(
    "Partial R² for species:",
    f"{partial_R2_species:.6f}"
)

print(
    "Partial R² (%):",
    f"{100 * partial_R2_species:.3f}%"
)

# -------------------------------------------------------------------
# 9. Validate observed dimensions
# -------------------------------------------------------------------

assert Yhat_state.shape == Y.shape
assert Yhat_full.shape == Y.shape
assert residual_state.shape == Y.shape
assert residual_full.shape == Y.shape

print("\nObserved model dimension checks: PASSED")

# -------------------------------------------------------------------
# 10. Freedman–Lane permutation test
# -------------------------------------------------------------------

N_PERMUTATIONS = 4999
PERMUTATION_SEED = 42

rng = np.random.default_rng(
    PERMUTATION_SEED
)

# The reduced-model fitted values remain fixed.
Yhat_reduced = Yhat_state

# Precompute the projection matrix for the full model:
#
# H_full = X_full (X_full' X_full)^-1 X_full'
#
# Using pinv makes this robust to the categorical design matrix.

H_full = (
    X_full
    @ np.linalg.pinv(X_full)
)

# Check that the projection matrix has the correct dimensions.
assert H_full.shape == (n, n)

permutation_F = np.empty(
    N_PERMUTATIONS,
    dtype=float
)

for i in range(N_PERMUTATIONS):

    # ---------------------------------------------------------------
    # Permute residual rows
    # ---------------------------------------------------------------

    permutation_indices = rng.permutation(n)

    permuted_residuals = (
        residual_state[
            permutation_indices
        ]
    )

    # ---------------------------------------------------------------
    # Reconstruct response under the reduced model
    # ---------------------------------------------------------------

    Y_permuted = (
        Yhat_reduced
        + permuted_residuals
    )

    # ---------------------------------------------------------------
    # Reduced-model residual SS
    # ---------------------------------------------------------------

    residual_reduced_perm = (
        Y_permuted
        - Yhat_reduced
    )

    SS_reduced_perm = np.sum(
        residual_reduced_perm ** 2
    )

    # ---------------------------------------------------------------
    # Full-model fitted values
    #
    # IMPORTANT:
    # H_full is n × n, so this returns n × q fitted values.
    # ---------------------------------------------------------------

    Yhat_full_perm = (
        H_full
        @ Y_permuted
    )

    assert Yhat_full_perm.shape == Y_permuted.shape

    # ---------------------------------------------------------------
    # Full-model residual SS
    # ---------------------------------------------------------------

    residual_full_perm = (
        Y_permuted
        - Yhat_full_perm
    )

    SS_full_perm = np.sum(
        residual_full_perm ** 2
    )

    # ---------------------------------------------------------------
    # Additional species SS
    # ---------------------------------------------------------------

    SS_species_perm = (
        SS_reduced_perm
        - SS_full_perm
    )

    # Numerical precision guard
    if (
        SS_species_perm < 0
        and np.isclose(
            SS_species_perm,
            0.0,
            atol=1e-10
        )
    ):
        SS_species_perm = 0.0

    # ---------------------------------------------------------------
    # Permuted pseudo-F
    # ---------------------------------------------------------------

    permutation_F[i] = (
        (SS_species_perm / df_species)
        /
        (SS_full_perm / df_residual)
    )

# -------------------------------------------------------------------
# 11. Validate permutation output
# -------------------------------------------------------------------

assert permutation_F.shape == (
    N_PERMUTATIONS,
)

assert np.isfinite(
    permutation_F
).all()

assert np.all(
    permutation_F >= 0
)

# -------------------------------------------------------------------
# 12. Empirical permutation p-value
# -------------------------------------------------------------------

n_exceedances = int(
    np.sum(
        permutation_F >= pseudo_F_adjusted
    )
)

p_value = (
    1 + n_exceedances
) / (
    N_PERMUTATIONS + 1
)

# -------------------------------------------------------------------
# 13. Null distribution summary
# -------------------------------------------------------------------

null_summary = pd.DataFrame({
    "statistic": [
        "Observed pseudo-F",
        "Null mean",
        "Null median",
        "Null 95th percentile",
        "Null 99th percentile",
        "Permutation p-value"
    ],
    "value": [
        pseudo_F_adjusted,
        np.mean(permutation_F),
        np.median(permutation_F),
        np.quantile(
            permutation_F,
            0.95
        ),
        np.quantile(
            permutation_F,
            0.99
        ),
        p_value
    ]
})

print("\n" + "=" * 80)
print("FREEDMAN–LANE PERMUTATION TEST")
print("=" * 80)

print(
    "Permutations:",
    N_PERMUTATIONS
)

print(
    "Seed:",
    PERMUTATION_SEED
)

display(null_summary)

print(
    "Permuted statistics ≥ observed:",
    n_exceedances
)

# -------------------------------------------------------------------
# 14. Save results
# -------------------------------------------------------------------

species_association_result = {
    "n_recordings": int(n),
    "n_species": int(len(species_categories)),
    "n_states": int(len(state_categories)),
    "n_retained_pcs": int(N_RETAINED_PCS),
    "species_df": int(df_species),
    "residual_df": int(df_residual),
    "ss_species_adjusted": float(
        SS_species_adjusted
    ),
    "ss_residual_full": float(
        SS_resid_full
    ),
    "pseudo_F": float(
        pseudo_F_adjusted
    ),
    "partial_R2": float(
        partial_R2_species
    ),
    "n_permutations": int(
        N_PERMUTATIONS
    ),
    "permutation_seed": int(
        PERMUTATION_SEED
    ),
    "p_value": float(
        p_value
    ),
    "n_exceedances": int(
        n_exceedances
    )
}

species_association_result_path = (
    PCA_ANALYSIS_DIR
    / "species_association_state_adjusted.json"
)

species_association_result_path.write_text(
    json.dumps(
        species_association_result,
        indent=2
    )
)

np.save(
    PCA_ANALYSIS_DIR
    / "species_association_permutation_F.npy",
    permutation_F
)

print("\nSaved:")
print(
    species_association_result_path
)

print(
    PCA_ANALYSIS_DIR
    / "species_association_permutation_F.npy"
)

# -------------------------------------------------------------------
# 15. Interpretation guardrail
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("INTERPRETATION GUARDRAIL")
print("=" * 80)

print(
    "This test asks whether species explains additional multivariate "
    "PCA structure after accounting for state."
)

print(
    "A significant result would support an association between acoustic "
    "feature structure and species identity, not a direct measure of "
    "species richness or biodiversity."
)

print(
    "A non-significant result would also be scientifically informative: "
    "it would indicate that the apparent species structure is not stronger "
    "than expected under the specified state-adjusted null model."
)

MODEL DESIGN
Observations: 172
PCA dimensions: 9
Species groups: 56
State groups: 3

State-only design shape: (172, 3)
State-only rank: 3

Full design shape: (172, 58)
Full model rank: 58

Species degrees of freedom: 55
Residual degrees of freedom: 114

OBSERVED STATE-ADJUSTED SPECIES ASSOCIATION
Species SS after accounting for state: 2787.625769
Residual SS after state + species: 3060.476784
Species degrees of freedom: 55
Residual degrees of freedom: 114
State-adjusted pseudo-F: 1.887937
Partial R² for species: 0.476672
Partial R² (%): 47.667%

Observed model dimension checks: PASSED

FREEDMAN–LANE PERMUTATION TEST
Permutations: 4999
Seed: 42


,statistic,value
0,Observed pseudo-F,1.887937
1,Null mean,1.041180
2,Null median,1.034201
3,Null 95th percentile,1.218595
4,Null 99th percentile,1.315092
5,Permutation p-value,0.000200


Permuted statistics ≥ observed: 0

Saved:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\species_association_state_adjusted.json
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\species_association_permutation_F.npy

INTERPRETATION GUARDRAIL
This test asks whether species explains additional multivariate PCA structure after accounting for state.
A significant result would support an association between acoustic feature structure and species identity, not a direct measure of species richness or biodiversity.
A non-significant result would also be scientifically informative: it would indicate that the apparent species structure is not stronger than expected under the specified state-adjusted null model.


In [52]:
# 06F.7 — Freedman–Lane Null Diagnostics
#
# Purpose:
# Validate the state-adjusted permutation framework used in 06F.6.
#
# We specifically examine:
#   1. Residual dispersion across states
#   2. Residual centroids across states
#   3. Distribution and range of the permutation pseudo-F statistics
#   4. Whether the observed statistic lies beyond the simulated null
#
# This cell does NOT rerun the permutation test.

# -------------------------------------------------------------------
# 1. Validate required objects from 06F.6
# -------------------------------------------------------------------

required_objects = [
    "Y",
    "Yhat_state",
    "residual_state",
    "state_labels",
    "permutation_F",
    "pseudo_F_adjusted",
    "p_value",
    "N_PERMUTATIONS"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "06F.6 objects are missing: "
        + ", ".join(missing_objects)
        + ". Rerun 06F.6 before running 06F.7."
    )

assert residual_state.shape == Y.shape
assert np.isfinite(residual_state).all()
assert np.isfinite(permutation_F).all()

# -------------------------------------------------------------------
# 2. Residual magnitude for each recording
# -------------------------------------------------------------------
#
# Euclidean norm across the nine retained PCA dimensions.

residual_norm = np.linalg.norm(
    residual_state,
    axis=1
)

residual_diagnostics = pd.DataFrame({
    "state": state_labels,
    "residual_norm": residual_norm
})

state_residual_summary = (
    residual_diagnostics
    .groupby("state")["residual_norm"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max"
    )
    .reset_index()
)

print("=" * 80)
print("STATE-WISE REDUCED-MODEL RESIDUAL DISPERSION")
print("=" * 80)

display(state_residual_summary)

# -------------------------------------------------------------------
# 3. Compare residual scale across states
# -------------------------------------------------------------------

state_mean_residual = (
    state_residual_summary["mean"]
)

residual_scale_ratio = (
    state_mean_residual.max()
    /
    state_mean_residual.min()
)

print(
    "Maximum / minimum state mean residual norm:",
    f"{residual_scale_ratio:.4f}"
)

print(
    "\nThis ratio is descriptive only. It is used to identify whether "
    "one state has markedly different residual dispersion."
)

# -------------------------------------------------------------------
# 4. Residual centroids by state
# -------------------------------------------------------------------
#
# Under the state-only model, residuals should be centered approximately
# around zero within each state.

residual_columns = [
    f"residual_PC{i}"
    for i in range(1, residual_state.shape[1] + 1)
]

residual_df = pd.DataFrame(
    residual_state,
    columns=residual_columns
)

residual_df["state"] = state_labels

residual_state_centroids = (
    residual_df
    .groupby("state")[residual_columns]
    .mean()
)

print("\n" + "=" * 80)
print("STATE-WISE RESIDUAL CENTROIDS")
print("=" * 80)

display(residual_state_centroids)

max_abs_residual_centroid = (
    np.abs(
        residual_state_centroids.to_numpy()
    ).max()
)

print(
    "\nMaximum absolute residual centroid:",
    f"{max_abs_residual_centroid:.10f}"
)

# -------------------------------------------------------------------
# 5. Overall residual centroid
# -------------------------------------------------------------------

overall_residual_centroid = (
    residual_state.mean(axis=0)
)

print(
    "Maximum absolute overall residual centroid:",
    f"{np.abs(overall_residual_centroid).max():.10f}"
)

# -------------------------------------------------------------------
# 6. Permutation null summary
# -------------------------------------------------------------------

null_quantiles = np.quantile(
    permutation_F,
    [
        0.00,
        0.01,
        0.05,
        0.50,
        0.95,
        0.99,
        0.999,
        1.00
    ]
)

null_quantile_table = pd.DataFrame({
    "quantile": [
        "0%",
        "1%",
        "5%",
        "50%",
        "95%",
        "99%",
        "99.9%",
        "100%"
    ],
    "pseudo_F": null_quantiles
})

print("\n" + "=" * 80)
print("PERMUTATION NULL DISTRIBUTION")
print("=" * 80)

display(null_quantile_table)

# -------------------------------------------------------------------
# 7. Null mean / variance / coefficient of variation
# -------------------------------------------------------------------

null_mean = np.mean(permutation_F)
null_std = np.std(
    permutation_F,
    ddof=1
)

null_cv = (
    null_std / null_mean
)

print(
    "Null mean:",
    f"{null_mean:.6f}"
)

print(
    "Null standard deviation:",
    f"{null_std:.6f}"
)

print(
    "Null coefficient of variation:",
    f"{null_cv:.6f}"
)

# -------------------------------------------------------------------
# 8. Observed statistic relative to null
# -------------------------------------------------------------------

null_95 = np.quantile(
    permutation_F,
    0.95
)

null_99 = np.quantile(
    permutation_F,
    0.99
)

observed_to_null95 = (
    pseudo_F_adjusted / null_95
)

observed_to_null_mean = (
    pseudo_F_adjusted / null_mean
)

print("\n" + "=" * 80)
print("OBSERVED VS. NULL")
print("=" * 80)

print(
    "Observed pseudo-F:",
    f"{pseudo_F_adjusted:.6f}"
)

print(
    "Null 95th percentile:",
    f"{null_95:.6f}"
)

print(
    "Null 99th percentile:",
    f"{null_99:.6f}"
)

print(
    "Observed / null 95th percentile:",
    f"{observed_to_null95:.4f}"
)

print(
    "Observed / null mean:",
    f"{observed_to_null_mean:.4f}"
)

# -------------------------------------------------------------------
# 9. Check whether observed exceeds the simulated null
# -------------------------------------------------------------------

n_exceedances = int(
    np.sum(
        permutation_F >= pseudo_F_adjusted
    )
)

print(
    "\nPermuted statistics ≥ observed:",
    n_exceedances,
    "/",
    N_PERMUTATIONS
)

# -------------------------------------------------------------------
# 10. Create a compact diagnostic table
# -------------------------------------------------------------------

diagnostic_summary = pd.DataFrame({
    "diagnostic": [
        "Observed pseudo-F",
        "Null mean",
        "Null SD",
        "Null 95th percentile",
        "Null 99th percentile",
        "Null maximum",
        "Exceedances",
        "Permutation p-value",
        "Max/min state residual scale"
    ],
    "value": [
        pseudo_F_adjusted,
        null_mean,
        null_std,
        null_95,
        null_99,
        permutation_F.max(),
        n_exceedances,
        p_value,
        residual_scale_ratio
    ]
})

print("\n" + "=" * 80)
print("06F.7 DIAGNOSTIC SUMMARY")
print("=" * 80)

display(diagnostic_summary)

# -------------------------------------------------------------------
# 11. Interpretation guardrail
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("INTERPRETATION GUARDRAIL")
print("=" * 80)

print(
    "Residual centroids should be approximately zero within each state "
    "because state is explicitly included in the reduced model."
)

print(
    "The permutation null should be inspected as a distribution rather "
    "than interpreted solely through the p-value."
)

print(
    "The state residual-dispersion comparison is descriptive and does not "
    "by itself establish or reject exchangeability."
)

print(
    "If the null diagnostics are satisfactory, the state-adjusted species "
    "association can be reported as a permutation-based multivariate result."
)

STATE-WISE REDUCED-MODEL RESIDUAL DISPERSION


,state,n,mean,median,std,min,max
0,AZ,72,4.949452,4.336854,2.118697,1.685773,11.759519
1,CA,79,5.491275,5.068782,2.301871,2.323763,18.169337
2,TX,21,5.958735,5.602413,3.350373,2.420843,17.992011


Maximum / minimum state mean residual norm: 1.2039

This ratio is descriptive only. It is used to identify whether one state has markedly different residual dispersion.

STATE-WISE RESIDUAL CENTROIDS


,residual_PC1,residual_PC2,residual_PC3,residual_PC4,residual_PC5,residual_PC6,residual_PC7,residual_PC8,residual_PC9
state,,,,,,,,,
AZ,1.480297e-16,-4.857226e-16,1.850372e-16,9.116936e-17,1.572816e-16,1.657625e-17,-4.317534e-17,2.312965e-17,-2.158767e-17
CA,4.497106e-17,3.372829e-16,1.925323e-16,1.475613e-16,-2.793124e-16,1.025902e-16,-1.264811e-16,7.026728e-18,7.026728e-18
TX,-3.489272e-16,3.383537e-16,-2.008975e-16,1.057355e-17,3.383537e-16,2.114711e-17,4.229421e-17,-4.890268e-17,4.229421e-17



Maximum absolute residual centroid: 0.0000000000
Maximum absolute overall residual centroid: 0.0000000000

PERMUTATION NULL DISTRIBUTION


,quantile,pseudo_F
0,0%,0.740378
1,1%,0.823942
2,5%,0.882166
3,50%,1.034201
4,95%,1.218595
5,99%,1.315092
6,99.9%,1.438345
7,100%,1.498688


Null mean: 1.041180
Null standard deviation: 0.103660
Null coefficient of variation: 0.099560

OBSERVED VS. NULL
Observed pseudo-F: 1.887937
Null 95th percentile: 1.218595
Null 99th percentile: 1.315092
Observed / null 95th percentile: 1.5493
Observed / null mean: 1.8133

Permuted statistics ≥ observed: 0 / 4999

06F.7 DIAGNOSTIC SUMMARY


,diagnostic,value
0,Observed pseudo-F,1.887937
1,Null mean,1.041180
2,Null SD,0.103660
3,Null 95th percentile,1.218595
4,Null 99th percentile,1.315092
5,Null maximum,1.498688
6,Exceedances,0.000000
7,Permutation p-value,0.000200
8,Max/min state residual scale,1.203918



INTERPRETATION GUARDRAIL
Residual centroids should be approximately zero within each state because state is explicitly included in the reduced model.
The permutation null should be inspected as a distribution rather than interpreted solely through the p-value.
The state residual-dispersion comparison is descriptive and does not by itself establish or reject exchangeability.
If the null diagnostics are satisfactory, the state-adjusted species association can be reported as a permutation-based multivariate result.


06F.8 — Representation Sensitivity
Scientific question
Does the state-adjusted association between species identity and multivariate
acoustic structure persist when the PCA representation is changed?
Six representations are compared:
1. Baseline
2. Distribution-aware transformation
3. Yeo–Johnson transformation
4. Block-balanced feature weighting
5. NDSI-derived features excluded
6. NDSI-family features excluded
The same 172 recordings and 56 species are used in every comparison.
The statistical model is held fixed:
- reduced model: Y ~ State
- full model: Y ~ State + Species
- test: Freedman–Lane residual permutation
- permutations: 4,999
- random seed: 42
Critical data-lineage rule
PCA score rows are aligned to ecological metadata by recording_id.
No PCA representation is assumed to have the same row order as
master_metadata.csv.
This prevents metadata/PCA misalignment and allows the baseline result from
06F.6 to be reproduced exactly.

In [53]:

# ============================================================
# 06F.8 — Setup and ID-safe alignment
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PCA_ANALYSIS_DIR = DATA_PROCESSED_DIR / "pca_analysis"
PCA_REPRESENTATION_DIR = DATA_PROCESSED_DIR / "pca_representations"

# ------------------------------------------------------------------
# Analysis configuration
# ------------------------------------------------------------------

REPRESENTATIONS = [
    "baseline",
    "distribution_aware",
    "yeo_johnson",
    "block_balanced",
    "ndsi_derived_excluded",
    "ndsi_family_excluded",
]

N_PERMUTATIONS = 4999
PERMUTATION_SEED = 42

N_RECORDINGS_TARGET = 172
N_SPECIES_TARGET = 56

# ------------------------------------------------------------------
# The focal cohort comes directly from the validated 06F workflow.
# It is the >=2-recording species cohort: 56 species / 172 recordings.
# ------------------------------------------------------------------

analysis_df = focal[
    [
        "recording_id",
        "species_id",
        "state",
        *retained_pc_columns
    ]
].copy()

analysis_df["recording_id"] = (
    analysis_df["recording_id"].astype(str)
)

analysis_df["species_id"] = (
    analysis_df["species_id"].astype(str)
)

analysis_df["state"] = (
    analysis_df["state"].astype(str)
)

analysis_df = analysis_df.reset_index(drop=True)

assert len(analysis_df) == N_RECORDINGS_TARGET
assert analysis_df["recording_id"].is_unique
assert analysis_df["species_id"].nunique() == N_SPECIES_TARGET

# ------------------------------------------------------------------
# Load the canonical PCA recording-ID order.
#
# This is the row lineage generated with the PCA artifacts.
# ------------------------------------------------------------------

PCA_IDS_PATH = PCA_ANALYSIS_DIR / "recording_ids.json"

with open(
    PCA_IDS_PATH,
    "r",
    encoding="utf-8"
) as f:
    pca_recording_ids = [
        str(x)
        for x in json.load(f)
    ]

assert len(pca_recording_ids) == 275
assert len(set(pca_recording_ids)) == 275

# ------------------------------------------------------------------
# ID -> PCA row position
# ------------------------------------------------------------------

pca_position_by_id = {
    recording_id: i
    for i, recording_id
    in enumerate(pca_recording_ids)
}

analysis_positions = np.array([
    pca_position_by_id[recording_id]
    for recording_id
    in analysis_df["recording_id"]
])

assert len(analysis_positions) == N_RECORDINGS_TARGET
assert len(np.unique(analysis_positions)) == N_RECORDINGS_TARGET

print("=" * 80)
print("06F.8 — ID-SAFE SETUP")
print("=" * 80)

print(
    "Analysis cohort:",
    len(analysis_df),
    "recordings /",
    analysis_df["species_id"].nunique(),
    "species"
)

print(
    "Unique PCA recording IDs:",
    len(pca_recording_ids)
)

print(
    "Cohort IDs found in PCA:",
    len(analysis_positions)
)

print(
    "Unique PCA positions:",
    len(np.unique(analysis_positions))
)

print("=" * 80)
print("ID alignment validation passed.")


06F.8 — ID-SAFE SETUP
Analysis cohort: 172 recordings / 56 species
Unique PCA recording IDs: 275
Cohort IDs found in PCA: 172
Unique PCA positions: 172
ID alignment validation passed.


In [54]:

# ============================================================
# 06F.8 — Load PCA representations
# ============================================================

pca_results = {}

for name in REPRESENTATIONS:

    scores_path = (
        PCA_ANALYSIS_DIR
        / f"{name}_scores.npy"
    )

    feature_names_path = (
        PCA_ANALYSIS_DIR
        / f"{name}_feature_names.json"
    )

    retained_path = (
        PCA_ANALYSIS_DIR
        / f"{name}_pa_retained.npy"
    )

    scores = np.load(scores_path)

    with open(
        feature_names_path,
        "r",
        encoding="utf-8"
    ) as f:
        feature_names = json.load(f)

    retained_mask = np.asarray(
        np.load(retained_path)
    ).astype(bool)

    retained_indices = np.where(
        retained_mask
    )[0]

    assert scores.shape[0] == len(pca_recording_ids)
    assert scores.shape[1] == len(feature_names)
    assert len(retained_indices) > 0
    assert np.isfinite(scores).all()

    pca_results[name] = {
        "scores": scores,
        "feature_names": feature_names,
        "retained_mask": retained_mask,
        "retained_indices": retained_indices,
    }

print("=" * 80)
print("06F.8 — PCA REPRESENTATION VALIDATION")
print("=" * 80)

for name in REPRESENTATIONS:

    result = pca_results[name]

    print(
        f"{name:28s} | "
        f"scores={result['scores'].shape} | "
        f"retained PCs={len(result['retained_indices'])}"
    )

print("=" * 80)
print("All PCA representations validated.")


06F.8 — PCA REPRESENTATION VALIDATION
baseline                     | scores=(275, 50) | retained PCs=9
distribution_aware           | scores=(275, 50) | retained PCs=9
yeo_johnson                  | scores=(275, 50) | retained PCs=9
block_balanced               | scores=(275, 50) | retained PCs=7
ndsi_derived_excluded        | scores=(275, 48) | retained PCs=9
ndsi_family_excluded         | scores=(275, 46) | retained PCs=9
All PCA representations validated.


In [55]:

# ============================================================
# 06F.8 — State-adjusted species association
# ============================================================

def run_state_adjusted_species_test(
    Y,
    species_labels,
    state_labels,
    n_permutations=4999,
    seed=42
):
    """
    Freedman–Lane permutation test for the additional contribution
    of species after accounting for state.

    Y:
        n_recordings × n_retained_PCs

    The same design and permutation procedure is applied to every
    PCA representation.
    """

    Y = np.asarray(Y, dtype=float)
    species_labels = np.asarray(species_labels)
    state_labels = np.asarray(state_labels)

    n = Y.shape[0]

    assert Y.ndim == 2
    assert len(species_labels) == n
    assert len(state_labels) == n
    assert np.isfinite(Y).all()

    # --------------------------------------------------------------
    # Design matrices
    # --------------------------------------------------------------

    state_categories = np.sort(
        np.unique(state_labels)
    )

    species_categories = np.sort(
        np.unique(species_labels)
    )

    X_state = np.column_stack([
        np.ones(n),
        *[
            (state_labels == state).astype(float)
            for state in state_categories[1:]
        ]
    ])

    X_species = np.column_stack([
        (species_labels == species).astype(float)
        for species in species_categories[1:]
    ])

    X_full = np.column_stack([
        X_state,
        X_species
    ])

    rank_state = np.linalg.matrix_rank(
        X_state
    )

    rank_full = np.linalg.matrix_rank(
        X_full
    )

    df_species = rank_full - rank_state
    df_residual = n - rank_full

    assert df_species > 0
    assert df_residual > 0

    # --------------------------------------------------------------
    # Projection matrices
    # --------------------------------------------------------------

    H_state = (
        X_state
        @ np.linalg.pinv(X_state)
    )

    H_full = (
        X_full
        @ np.linalg.pinv(X_full)
    )

    # Numerical checks on projection matrices
    assert np.allclose(
        H_state,
        H_state.T
    )

    assert np.allclose(
        H_state @ H_state,
        H_state
    )

    assert np.allclose(
        H_full,
        H_full.T
    )

    assert np.allclose(
        H_full @ H_full,
        H_full
    )

    # --------------------------------------------------------------
    # Observed statistic
    # --------------------------------------------------------------

    Yhat_state = H_state @ Y
    Yhat_full = H_full @ Y

    residual_state = (
        Y - Yhat_state
    )

    residual_full = (
        Y - Yhat_full
    )

    SS_resid_state = np.sum(
        residual_state ** 2
    )

    SS_resid_full = np.sum(
        residual_full ** 2
    )

    SS_species_adjusted = (
        SS_resid_state
        - SS_resid_full
    )

    if (
        SS_species_adjusted < 0
        and np.isclose(
            SS_species_adjusted,
            0.0,
            atol=1e-10
        )
    ):
        SS_species_adjusted = 0.0

    pseudo_F = (
        (SS_species_adjusted / df_species)
        /
        (SS_resid_full / df_residual)
    )

    # --------------------------------------------------------------
    # Freedman–Lane permutations
    # --------------------------------------------------------------

    rng = np.random.default_rng(seed)

    permutation_F = np.empty(
        n_permutations,
        dtype=float
    )

    for i in range(n_permutations):

        permuted_residuals = residual_state[
            rng.permutation(n)
        ]

        Y_permuted = (
            Yhat_state
            + permuted_residuals
        )

        residual_full_perm = (
            Y_permuted
            - H_full @ Y_permuted
        )

        SS_full_perm = np.sum(
            residual_full_perm ** 2
        )

        SS_reduced_perm = np.sum(
            (
                Y_permuted
                - Yhat_state
            ) ** 2
        )

        SS_species_perm = (
            SS_reduced_perm
            - SS_full_perm
        )

        if (
            SS_species_perm < 0
            and np.isclose(
                SS_species_perm,
                0.0,
                atol=1e-10
            )
        ):
            SS_species_perm = 0.0

        permutation_F[i] = (
            (SS_species_perm / df_species)
            /
            (SS_full_perm / df_residual)
        )

    assert np.isfinite(
        permutation_F
    ).all()

    # --------------------------------------------------------------
    # Empirical p-value
    # --------------------------------------------------------------

    exceedances = int(
        np.sum(
            permutation_F >= pseudo_F
        )
    )

    p_value = (
        exceedances + 1
    ) / (
        n_permutations + 1
    )

    partial_R2 = (
        SS_species_adjusted
        /
        (
            SS_species_adjusted
            + SS_resid_full
        )
    )

    return {
        "n_recordings": n,
        "n_species": len(species_categories),
        "n_states": len(state_categories),
        "n_pcs": Y.shape[1],
        "df_species": df_species,
        "df_residual": df_residual,
        "ss_species_adjusted": SS_species_adjusted,
        "ss_residual_full": SS_resid_full,
        "pseudo_F": pseudo_F,
        "partial_R2": partial_R2,
        "null_mean": np.mean(permutation_F),
        "null_median": np.median(permutation_F),
        "null_sd": np.std(
            permutation_F,
            ddof=1
        ),
        "null_p95": np.quantile(
            permutation_F,
            0.95
        ),
        "null_p99": np.quantile(
            permutation_F,
            0.99
        ),
        "null_max": np.max(
            permutation_F
        ),
        "exceedances": exceedances,
        "p_value": p_value,
        "permutation_F": permutation_F,
    }


# ------------------------------------------------------------------
# Run all six representations
# ------------------------------------------------------------------

species_labels = (
    analysis_df["species_id"]
    .to_numpy()
)

state_labels = (
    analysis_df["state"]
    .to_numpy()
)

representation_results = {}
permutation_distributions = {}

print("=" * 80)
print("06F.8 — STATE-ADJUSTED SPECIES ASSOCIATION")
print("=" * 80)

for name in REPRESENTATIONS:

    scores = pca_results[name]["scores"]
    retained_indices = (
        pca_results[name]["retained_indices"]
    )

    # CRITICAL:
    # PCA rows are selected using recording_id alignment,
    # not assumed metadata row order.
    Y = scores[
        analysis_positions,
        :
    ][:, retained_indices]

    result = run_state_adjusted_species_test(
        Y=Y,
        species_labels=species_labels,
        state_labels=state_labels,
        n_permutations=N_PERMUTATIONS,
        seed=PERMUTATION_SEED
    )

    permutation_distributions[name] = (
        result.pop("permutation_F")
    )

    representation_results[name] = result

    print(
        f"{name:28s} | "
        f"PCs={result['n_pcs']:2d} | "
        f"F={result['pseudo_F']:.6f} | "
        f"p={result['p_value']:.6f} | "
        f"null95={result['null_p95']:.6f} | "
        f"nullMax={result['null_max']:.6f}"
    )

print("=" * 80)


06F.8 — STATE-ADJUSTED SPECIES ASSOCIATION
baseline                     | PCs= 9 | F=1.887937 | p=0.000200 | null95=1.218595 | nullMax=1.498688
distribution_aware           | PCs= 9 | F=1.973929 | p=0.000200 | null95=1.217979 | nullMax=1.488279
yeo_johnson                  | PCs= 9 | F=1.935942 | p=0.000200 | null95=1.210688 | nullMax=1.570057
block_balanced               | PCs= 7 | F=2.007771 | p=0.000200 | null95=1.220654 | nullMax=1.523173
ndsi_derived_excluded        | PCs= 9 | F=1.860947 | p=0.000200 | null95=1.222497 | nullMax=1.521835
ndsi_family_excluded         | PCs= 9 | F=1.860968 | p=0.000200 | null95=1.223860 | nullMax=1.492772


In [56]:

# ============================================================
# 06F.8 — Robustness summary and baseline reconciliation
# ============================================================

representation_summary = pd.DataFrame(
    [
        {
            "representation": name,
            **{
                key: value
                for key, value in result.items()
                if key != "permutation_F"
            }
        }
        for name, result
        in representation_results.items()
    ]
)

display(
    representation_summary[
        [
            "representation",
            "n_pcs",
            "pseudo_F",
            "partial_R2",
            "null_mean",
            "null_p95",
            "null_p99",
            "null_max",
            "exceedances",
            "p_value"
        ]
    ]
)

# ------------------------------------------------------------------
# Critical reconciliation:
#
# The baseline result from 06F.8 must reproduce the already-validated
# 06F.6 state-adjusted result.
# ------------------------------------------------------------------

baseline_F_06F6 = float(
    pseudo_F_adjusted
)

baseline_F_06F8 = float(
    representation_results[
        "baseline"
    ]["pseudo_F"]
)

print("=" * 80)
print("BASELINE RECONCILIATION")
print("=" * 80)

print(
    "06F.6 baseline pseudo-F:",
    f"{baseline_F_06F6:.12f}"
)

print(
    "06F.8 baseline pseudo-F:",
    f"{baseline_F_06F8:.12f}"
)

print(
    "Absolute difference:",
    f"{abs(baseline_F_06F6 - baseline_F_06F8):.12e}"
)

assert np.isclose(
    baseline_F_06F6,
    baseline_F_06F8,
    rtol=1e-10,
    atol=1e-10
)

print(
    "\nBASELINE RECONCILIATION: PASSED"
)

# ------------------------------------------------------------------
# Save a compact comparison table
# ------------------------------------------------------------------

summary_path = (
    PCA_ANALYSIS_DIR
    / "species_association_representation_sensitivity.csv"
)

representation_summary.to_csv(
    summary_path,
    index=False
)

# Save the permutation distributions separately.
permutation_path = (
    PCA_ANALYSIS_DIR
    / "species_association_representation_permutations.npz"
)

np.savez(
    permutation_path,
    **permutation_distributions
)

# Save JSON-compatible results.
json_results = {}

for name, result in representation_results.items():

    json_results[name] = {
        key: (
            float(value)
            if isinstance(
                value,
                (np.floating, float)
            )
            else int(value)
            if isinstance(
                value,
                (np.integer, int)
            )
            else value
        )
        for key, value
        in result.items()
    }

json_results_path = (
    PCA_ANALYSIS_DIR
    / "species_association_representation_sensitivity.json"
)

json_results_path.write_text(
    json.dumps(
        json_results,
        indent=2
    ),
    encoding="utf-8"
)

print("\nSaved:")
print(summary_path)
print(permutation_path)
print(json_results_path)


,representation,n_pcs,pseudo_F,partial_R2,null_mean,null_p95,null_p99,null_max,exceedances,p_value
0,baseline,9,1.887937,0.476672,1.041180,1.218595,1.315092,1.498688,0,0.0002
1,distribution_aware,9,1.973929,0.487793,1.041008,1.217979,1.308086,1.488279,0,0.0002
2,yeo_johnson,9,1.935942,0.482939,1.040693,1.210688,1.305026,1.570057,0,0.0002
3,block_balanced,7,2.007771,0.492041,1.040777,1.220654,1.314154,1.523173,0,0.0002
4,ndsi_derived_excluded,9,1.860947,0.473081,1.041239,1.222497,1.317429,1.521835,0,0.0002
5,ndsi_family_excluded,9,1.860968,0.473084,1.041463,1.223860,1.326119,1.492772,0,0.0002


BASELINE RECONCILIATION
06F.6 baseline pseudo-F: 1.887937195850
06F.8 baseline pseudo-F: 1.887937195850
Absolute difference: 0.000000000000e+00

BASELINE RECONCILIATION: PASSED

Saved:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\species_association_representation_sensitivity.csv
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\species_association_representation_permutations.npz
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\pca_analysis\species_association_representation_sensitivity.json


Interpretation guardrail
The representation-sensitivity analysis asks whether the species association
depends strongly on preprocessing or feature-family choices.
A robust result is one in which the direction and magnitude of the
state-adjusted pseudo-F statistic remain similar across representations.
This analysis does not establish that PCA estimates biodiversity or
species richness. A persistent association would instead support the narrower
claim that multivariate acoustic structure contains reproducible information
associated with species identity after accounting for state.
The focal cohort is also not regionally representative: the ≥2-recording
criterion disproportionately reduces Texas representation. This limitation
must remain explicit in interpretation.